# Phase A - overnight extraction: stage locally, then all backbones in one pass

Run cells 1 -> 15 in order and leave it. **Nothing here needs editing.**

* **Cell 7b** copies the archive onto the runtime's local disk (~171 GB, 40-90 min). The
  25-series trial showed 95% of wall clock was Drive read, at 0.5 MB/s.
* **Cell 13** builds every backbone and runs a real forward pass at the real grid, recording
  peak GPU memory. A backbone that will not load, or will not fit, is dropped with its reason
  printed and the rest carry on. If memory is tight it holds one backbone on the GPU at a
  time rather than failing. Whatever gets dropped, you run tomorrow.
* **Cell 14** reads each series ONCE and feeds every surviving backbone from it. Each
  backbone writes to its own folder and resumes independently.

Watch cell 14 for two minutes before you go to bed: the progress bar should show **single
digit seconds per series**, not the 35 s it was reading from Drive. If it does not, stop it
and say so - do not edit anything.


In [ ]:
# 0. Repository and paths -- every location is read from config.yaml (see README), none is typed here.
import os
import sys


def _find_repo():
    for c in (os.environ.get("QCT3D_REPO", ""), os.getcwd(), os.path.dirname(os.getcwd()),
              "/content/qct3d-saturation"):
        if c and os.path.isfile(os.path.join(c, "src", "qct3d", "config.py")):
            return os.path.abspath(c)
    raise RuntimeError("Cannot find the repository. Clone it first (README, 'Running on Colab') "
                       "or set the environment variable QCT3D_REPO to its folder.")


REPO_ROOT = _find_repo()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from qct3d.config import load_config  # noqa: E402

P = load_config()

In [ ]:
# 1. Google Drive -- the only cell that needs you at the keyboard
#
# Everything interactive is front-loaded here so the rest of the notebook can run unattended.
# No Hugging Face token is required: the MedicalNet ResNet-18 weights live in the public repo
# TencentMedicalNet/MedicalNet-Resnet18 and are not gated. One is read anyway IF it happens to be
# in Colab Secrets, because it raises the anonymous rate limit on huggingface.co and costs nothing.
import os

try:
    from google.colab import drive
    drive.mount(P["drive_mount"])
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("not running in Colab -- skipping the Drive mount. Point the paths in the config cell "
          "at wherever the data actually is.")

if IN_COLAB:
    _mydrive = P["drive_root"]
    if not os.path.isdir(_mydrive):
        raise RuntimeError(
            f"{_mydrive} does not exist after mounting. The mount did not complete -- re-run this "
            f"cell and finish the authorisation dialog before starting a long run.")
    print(f"Drive mounted: {_mydrive}")

HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("HF_TOKEN found in Colab Secrets (optional here -- MedicalNet is not gated).")
    except Exception:
        print("No HF_TOKEN in Colab Secrets. That is fine: the MedicalNet checkpoint is public.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

In [ ]:
# 2. Installs -- every version the next cell CHECKS is pinned here
#
# The rule: anything cell 3 asserts, this cell pins. Checking a condition that nothing establishes
# is not a safety net, it is a trap that fires on somebody else's schedule.
#
#   monai>=1.4          `resnet18(pretrained=True, ...)` pulls MedicalNet weights from the
#                       TencentMedicalNet Hugging Face repos. That code path, and the
#                       shortcut_type / bias_downsample contract cell 13 relies on, landed in 1.3.
#   pennylane>=0.35     `qml.qnn.TorchLayer`, `StronglyEntanglingLayers.shape`, and backprop
#                       through `default.qubit` with the torch interface.
#   huggingface_hub     MONAI imports it lazily for the checkpoint download; without it the
#                       pretrained=True path raises OptionalImportError, not a network error --
#                       a confusing failure that reads as "the weights are gone".
#   pydicom>=3.0        `pydicom.pixels.pixel_array(ds, decoding_plugin=...)` is a 3.x API. On 2.x
#                       it can fall back to a decoder that routes through Pillow, which pydicom's
#                       own docs warn "performs a number of undesirable operations on the decoded
#                       images which are not always reversible" -- not acceptable when HU values
#                       are the entire signal.
#   pylibjpeg/gdcm/...  pixel-data decompression plugins. RSNA-ICH ships uncompressed, but a
#                       missing plugin turns any compressed member into a hard failure rather than
#                       a skipped slice, and the cost of having them is one install.
#   SimpleITK>=2.3      physical-spacing resampling.
#   scikit-learn>=1.3   StratifiedGroupKFold, GroupShuffleSplit.
#
# numpy, torch and pillow are deliberately NOT touched: Colab already has them, and upgrading a
# package that is already imported into a running kernel leaves a stale module behind -- exactly
# the failure that only shows up three hours into an overnight run.
import subprocess
import sys

_already = [m for m in ("monai", "pennylane", "pydicom", "SimpleITK") if m in sys.modules]
if _already:
    print(f"WARNING: {_already} are already imported in this kernel. pip can change them on disk "
          f"but cannot replace what is already loaded, so a restart may be required -- cell 3 "
          f"will say so explicitly.")

PACKAGES = [
    "monai>=1.4,<2", "pennylane>=0.35,<0.50", "huggingface_hub",
    "pydicom>=3.0", "einops", "huggingface_hub", "SimpleITK>=2.3", "scikit-learn>=1.3", "pandas>=2.1", "pyarrow",
    "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "python-gdcm", "pyjpegls", "tqdm",
]
print("installing:", ", ".join(PACKAGES))
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
                    capture_output=True, text=True)
if _r.returncode != 0:
    print(_r.stdout[-2000:]); print(_r.stderr[-4000:])
    raise RuntimeError(
        f"pip exited {_r.returncode}. The exit status is CHECKED here rather than left as a "
        f"printed message under a shell magic, because a silent install failure at the top of an "
        f"unattended run is discovered at 3am as an ImportError.")
print("install ok. If cell 3 says RESTART REQUIRED: Runtime > Restart session, then Run all.")

In [ ]:
# 3. Version contract -- what is on disk, what is loaded, and whether they agree
#
# Two distinct failure modes, which need distinguishing because the fixes differ:
#
#   PIN NOT SATISFIED   pip did not install what cell 2 asked for (a resolution conflict, or the
#                       cell was skipped). Re-run cell 2.
#   RESTART REQUIRED    the right version is on disk, but this kernel already has an older/newer
#                       one loaded from before the install. pip cannot replace a module that is
#                       already in memory. Runtime > Restart session, then Run all again.
import importlib
import importlib.metadata as _md

REQUIRED = {
    # dist name        (module,            min,       max_exclusive)
    "monai":           ("monai",           (1, 4),    None),
    "pennylane":       ("pennylane",       (0, 35),   None),
    "pydicom":         ("pydicom",         (3, 0),    None),
    "pandas":          ("pandas",          (2, 1),    None),
    "scikit-learn":    ("sklearn",         (1, 3),    None),
    "SimpleITK":       ("SimpleITK",       (2, 3),    None),
    "torch":           ("torch",           (2, 0),    None),
}


def _parse(v):
    out = []
    for part in str(v).split(".")[:3]:
        num = "".join(ch for ch in part if ch.isdigit())
        out.append(int(num) if num else 0)
    while len(out) < 3:
        out.append(0)
    return tuple(out)


problems, restarts = [], []
print(f"{'package':<18}{'on disk':<16}{'imported':<16}status")
print("-" * 70)
for dist, (mod_name, lo, hi) in REQUIRED.items():
    try:
        disk = _md.version(dist)
    except Exception:
        disk = None
    try:
        mod = importlib.import_module(mod_name)
        loaded = getattr(mod, "__version__", None)
    except Exception as e:
        mod, loaded = None, f"IMPORT FAILED ({type(e).__name__})"

    status = "ok"
    if disk is None:
        status = "NOT INSTALLED"
        problems.append(f"{dist} is not installed -- re-run cell 2")
    else:
        dv = _parse(disk)
        if dv < lo + (0,) * (3 - len(lo)):
            status = f"TOO OLD (need >= {'.'.join(map(str, lo))})"
            problems.append(f"{dist} {disk} on disk, need >= {'.'.join(map(str, lo))}")
        elif hi is not None and dv >= hi + (0,) * (3 - len(hi)):
            status = f"TOO NEW (need < {'.'.join(map(str, hi))})"
            problems.append(f"{dist} {disk} on disk, need < {'.'.join(map(str, hi))}")
        elif isinstance(loaded, str) and not loaded.startswith("IMPORT FAILED"):
            if _parse(loaded)[:2] != dv[:2]:
                status = "RESTART REQUIRED"
                restarts.append(f"{dist}: {loaded} loaded, {disk} on disk")
    print(f"{dist:<18}{str(disk):<16}{str(loaded):<16}{status}")

if restarts:
    raise RuntimeError(
        "A different version is loaded in this kernel than is installed on disk:\n  "
        + "\n  ".join(restarts)
        + "\n\nRuntime > Restart session, then Run all. pip cannot replace a module that is "
          "already in memory, so continuing would run against the OLD version while the checks "
          "above pass against the new one.")
if problems:
    raise RuntimeError("version pins not satisfied:\n  " + "\n  ".join(problems))
print("\nall version pins satisfied, nothing stale in this kernel.")

# einops is imported lazily by MONAI inside SwinUNETR.forward, so a missing install
# surfaces as an OptionalImportError several hours into an extraction rather than here.
try:
    import einops  # noqa: F401
    print("einops             OK")
except Exception as _e:
    raise SystemExit("einops is missing and SwinUNETR needs it at forward time. "
                     "Re-run the install cell, then Runtime > Restart session.")


In [ ]:
# 4. CONFIGURATION -- the only cell you should normally need to edit
import errno
import os
import random
import time

import torch

# ==============================================================================================
# INPUTS -- verified against your Drive on 2026-08-26
# ==============================================================================================
# The folder holding the split archive and the labels. Its layout is expected to be
#   RSNA_ROOT/parts/part_00000 ... part_00090                        (the 193.3 GB split zip)
#   RSNA_ROOT/rsna-intracranial-hemorrhage-detection/stage_2_train.csv
# Both are checked in the preflight rather than assumed.
#
# DO NOT point this at <drive_root>/RSNA_ICH -- that folder has a doubled subfolder
# name, only stage_2_test, and no parts/. It is the obvious wrong turn here.
RSNA_ROOT = P["rsna_archive_dir"]

PARTS_DIR = os.path.join(RSNA_ROOT, "parts")
PARTS_PATTERN = r"^part_\d+$"        # change if your split used a different scheme (e.g. 'xa*')
LABELS_CSV = os.path.join(RSNA_ROOT, "rsna-intracranial-hemorrhage-detection", "stage_2_train.csv")

# Which directory inside the archive holds the labelled training images.
#
# "auto" (recommended) inspects the zip's own listing and picks it. The layout is genuinely not
# fixed: depending on how the archive was assembled the images sit either at "stage_2_train/" or
# one level down inside the competition folder, e.g.
#     rsna-intracranial-hemorrhage-detection/stage_2_train/ID_xxxxxxxxx.dcm
# and the archive on this Drive is the second kind -- all 874,037 entries live under
# "rsna-intracranial-hemorrhage-detection/". Cell 7 prints every directory it finds and says which
# one it chose and why, so the decision is visible rather than silent.
#
# stage_2_test/ is deliberately excluded: it carries no labels, and indexing it would add ~121k
# header reads for nothing.
ARCHIVE_PREFIX = "auto"          # "auto", or an explicit prefix ending in "/"

OUTPUT_DIR = P["rsna_output_dir"]

# ---- STAGE THE ARCHIVE ON LOCAL DISK BEFORE EXTRACTING -----------------------------------------
# This is the single biggest lever in the notebook. The 25-series trial measured 35.2 s per
# series, of which 33.5 s -- 95% of wall clock -- was READ: 18 MB of DICOM pulled through the
# Drive FUSE mount in 33 s, which is 0.5 MB/s. Drive is not slow at bandwidth, it is slow at
# ROUND TRIPS, and one series is ~35 separate ranged reads scattered across ~35 different 2 GB
# parts. Copied onto the runtime's own disk those same reads are sub-millisecond, the read
# bucket collapses, and what is left is preprocess + model + write.
#
#   from Drive  : 35.2 s/series  ->  39 h for 4,000, PER BACKBONE
#   from local  : ~1 h to stage  +  ~3-5 h for 4,000, ALL THREE BACKBONES
#
# Cell 7b does the copy. It stages only the parts the train split occupies (~171 GB of 195 GB;
# the rest is stage_2_test, recreated as sparse files that cost no disk), verifies every part's
# size, and is resumable -- re-running skips what is already there. None of it is in the
# preprocessing signature, so features already extracted stay valid and are skipped on resume.
#
# Set False to read straight from Drive, which works but takes 39 h per backbone.
STAGE_LOCALLY = True
LOCAL_PARTS_DIR = "/content/parts_local"   # ephemeral: gone when the Colab session ends
STAGE_COPY_WORKERS = 4                     # concurrent part copies; Drive throttles one stream
STAGE_FREE_MARGIN_GB = 8                   # refuse to start unless this much disk is left over

# ---- WHAT TO DO IF A BACKBONE WILL NOT FIT, OR WILL NOT LOAD ----------------------------------
# An overnight run must not die because one of three checkpoints is unavailable or one model is
# too large for the card. Cell 13 therefore MEASURES rather than assumes:
#
#   * a backbone that fails to build (missing checkpoint, bad keys) is DROPPED, with the reason
#     printed, and the run continues on the ones that did load;
#   * every backbone runs a real forward pass at the real grid before extraction starts, with
#     peak GPU memory recorded. If that pass runs out of memory with all backbones resident, the
#     notebook falls back to keeping ONE backbone on the GPU at a time and retries;
#   * if it still runs out of memory alone, THAT backbone is dropped and the others proceed.
#
# So the worst case tonight is "two of three extracted, the third named in the morning", never a
# dead cell at 2am. Whatever is dropped is written to Drive and picked up by simply re-running
# this notebook later with only that backbone listed -- the features already on disk are skipped.
DROP_BACKBONE_ON_FAILURE = True

# "auto"  keep every backbone on the GPU (fastest) and fall back automatically on the first OOM
# True    keep exactly one backbone on the GPU at a time from the start (safest, ~0.1 s/series)
# False   never fall back; an OOM is then a failed (series, backbone) pair
GPU_ONE_AT_A_TIME = "auto"


# ==============================================================================================
# UNATTENDED RUN
# ==============================================================================================
UNATTENDED_RUN = False            # every preflight finding is fatal; the right setting overnight
PREFLIGHT_DOWNLOAD_MODEL = True  # pull the MedicalNet weights in minute 2, not hour 4

# WHERE A GPU ACTUALLY MATTERS. Only cell 14 (the backbone forward pass) uses one at all. Parts 1
# and 2 -- including the header index, which is the multi-hour step -- are pure I/O and run exactly
# as fast on a CPU runtime, and Part 4 trains on cached vectors in seconds either way. So the
# preflight REPORTS a missing GPU rather than refusing: blocking the long I/O work because the
# short GPU work would be slow gets the priority exactly backwards.
#
# Cell 14 does the real check, and will not start a large extraction on CPU unless you say so here.
ALLOW_CPU_EXTRACTION = False
CPU_EXTRACTION_WARN_ABOVE = 50   # series; below this a CPU run is a smoke test, not a mistake

# ==============================================================================================
# SMOKE TEST
# ==============================================================================================
# True configures a coherent end-to-end plumbing check that finishes in minutes.
#
# It is NOT simply "index fewer members". That was the obvious idea and it does not work: the
# slice -> series mapping lives in the headers, so indexing 3% of members gives roughly 1 slice of
# every series, every series then fails the minimum-slice test, and the manifest comes out EMPTY
# with nothing to explain why. To actually exercise the pipeline on a partial index the slice
# minimum and the z-uniformity check have to come down with it -- which is fine for testing the
# machinery and produces volumes that are anatomically meaningless. The notebook says so loudly
# wherever it matters rather than letting you mistake the result for a finding.
SMOKE_TEST = False

# ==============================================================================================
# THE HEADER INDEX  (Part 2)  -- the one genuinely expensive, one-time step
# ==============================================================================================
# RSNA-ICH is a flat directory of ID_<hex>.dcm with no series structure in the filenames, so the
# slice -> (patient, study, series, z) mapping exists only inside the headers. Building it means
# touching every member once. It is done in ascending archive-offset order (forward-only access,
# which is what a FUSE-mounted Drive is good at), in resumable chunks, cached to Drive as parquet.
#
# COST, honestly: ~753k members at 4,638 B and one Drive round trip each. INDEX_WORKERS="auto"
# measures the best thread count first, then the cell measures your actual rate after the first
# chunk and prints an ETA. It never has to happen again, and it resumes chunk by chunk.
HEADER_BYTES = 4096              # read this much of each member; enough for the tags we need
HEADER_BYTES_ESCALATE = 65536    # retried once at this size before a member is called non-DICOM
INDEX_CHUNK = 5000               # members per resumable checkpoint
# HOW the header pass reads, which on Drive matters more than anything else here.
#   "random"  one ~4.6 KB request per member. Fewest BYTES (3.5 GB), most REQUESTS (752,803).
#   "bulk"    large contiguous blocks, members cut out of them in memory. Fewest REQUESTS
#             (~10k), most bytes (the 167 GB the members span).
#   "auto"    measures both on your mount, picks the winner, and re-measures if the rate
#             collapses mid-run -- which is what Drive throttling looks like.
# A real run measured 311 members/s, then 15/s for 56 minutes, then 153/s again. Nothing was
# wrong with it except that a request-throttled mount cannot be argued with, only avoided.
INDEX_STRATEGY = "auto"          # "auto" | "random" | "bulk"
INDEX_WORKERS = "auto"           # "auto" measures thread counts too; an integer forces one
INDEX_WORKERS_MAX = 24           # ceiling for "auto"; more threads on a FUSE mount stops helping
INDEX_CALIBRATE_N = 200          # members per candidate in that measurement
INDEX_BULK_BLOCK_BYTES = 16 << 20  # one bulk request; ~72 members of this archive
INDEX_RECALIBRATE_AT = 0.35      # re-measure when the rate falls below this fraction of the best
INDEX_RECALIBRATE_MAX = 4        # ... at most this many times in one run
INDEX_IO_RETRY_ROUNDS = 3        # extra serial passes over members that hit transient Drive I/O
INDEX_MAX_CONSEC_SKIPS = 3       # consecutive unwritable chunks before calling the mount wedged
MAX_MEMBERS_TO_INDEX = None      # None = all of them. An integer stops early (resumable).

# ==============================================================================================
# WHICH VOLUMES
# ==============================================================================================
# Slice reads for one volume are fetched CONCURRENTLY and in ARCHIVE-OFFSET order, then decoded
# serially in slice order exactly as before. RSNA is a flat directory, so a series' slices are
# scattered across the archive and reading them in z order means seeking backwards through 195 GB;
# and at ~35 slices per volume the round-trip latency, not the bandwidth, is what dominates.
# 1 restores the old strictly-serial behaviour.
# Reading the archive is ~88% of Phase A, so this is the one number worth measuring
# rather than guessing. "auto" times a few real series at each candidate before the long
# loop starts; an integer fixes it. NOT part of the preprocessing signature -- concurrency
# changes when bytes arrive, never what they are -- so it is safe to change at any point,
# including between shards or on a resume.
EXTRACT_READ_WORKERS = "auto"
EXTRACT_READ_WORKER_CANDIDATES = [4, 8, 16, 24]
EXTRACT_PREFETCH_MAX_MB = 512    # a series projected to exceed this is read serially instead, so
                                 # one pathological series cannot balloon memory

SERIES_MIN_SLICES = 20           # the dataset paper says 20-60 axial images per series
MIN_KEPT_SLICE_FRACTION = 0.5    # refuse to build a volume that lost half its slices to decoding
APPLY_RSNA_PIXEL_FIX = True      # the 12-bit modulo-4096 correction; see Part 3

# Volume-level stratified subsample. The full set is not necessary and
# a representative stratified subsample keeps this phase tractable. Sampling is proportional
# WITHIN each distinct multi-label pattern, so every subtype's prevalence survives the sample.
# None = every eligible series.
MAX_SERIES = 4000
SAMPLE_SEED = 7

# ==============================================================================================
# PREPROCESSING
# ==============================================================================================
# Whole-head fixed grid, NOT a lesion-centred crop: negatives and deployment-time cases, where no
# lesion location is known, must be framed identically to training positives, or the framing
# itself leaks the label.
#
# RSNA-ICH is ~512x512 at ~0.45 mm in plane with ~5 mm slices and ~30-40 slices, i.e. a z extent
# around 150-190 mm. The grid below is 256 x 256 mm in plane and 192 mm through plane, which
# contains that with margin and never crops anatomy away. Part 2's fingerprint report measures
# your archive's real distribution and says whether these numbers are right for it.
# In-plane resolution is set by what the winning solutions actually resolved at. Every top-5
# RSNA-ICH solution worked at 480-512 px over a ~250 mm field of view, i.e. ~0.5 mm/px. At 2 mm a
# 5 mm subdural film spans 2.5 voxels and epidural -- the rarest subtype here, 1.6% of series --
# is exactly the thin-collection case that may not survive the resampling at all. A null result
# on those subtypes at 2 mm would be uninterpretable: an adapter failure and a resolution failure
# look identical. 1 mm costs 4.7x the frozen forward pass (measured) against a step that is ~6%
# model and ~62% Drive reads, and 0.10 GB of activations on a 15.6 GB T4.
TARGET_SPACING_XYZ = (1.0, 1.0, 3.0)     # (x, y, z) mm
TARGET_SHAPE_DHW = (64, 256, 256)        # (D, H, W) voxels  -> 192 x 256 x 256 mm

# Dual HU windowing. A single wide window collapses exactly the soft-tissue contrast that
# separates acute blood from normal parenchyma.
#   brain     centre 40, width 80    -- parenchyma: grey/white contrast
#   subdural  centre 75, width 215   -- acute blood against dura and skull
# The three windows the top solutions converged on, read verbatim out of their source rather
# than from a blog post about them:
#
#   brain        (40,  80)   darraghdog 2nd, okotaku 3rd, appian42 11th, ngxbac 5th -- unanimous
#   subdural     (80, 200)   darraghdog 2nd, okotaku 3rd, appian42 11th, Anjum48 5th
#   soft tissue  (40, 380)   darraghdog 2nd, okotaku 3rd, appian42 11th
#
# Two things worth knowing before changing them. The third window is NOT the bone window: okotaku
# had (600, 2800) wired up behind a flag and shipped with it off, defaulting to (40, 380).
# And (75, 215) -- what this notebook used until now -- is ngxbac's 5th-place variant, not the
# consensus; it is defensible but it is the odd one out.
#
#   github.com/darraghdog/rsna/blob/master/scripts/prepare_meta_dicom.py  (apply_window_policy)
#   github.com/okotaku/kaggle_rsna2019_3rd_solution/blob/master/src/datasets.py
#   github.com/appian42/kaggle-rsna-intracranial-hemorrhage/blob/master/src/cnn/dataset/custom_dataset.py
#
# `hu_window` below is algebraically their `apply_window` followed by their own per-channel
# normalisation: they clamp to [C - W//2, C + W//2] and then divide by W after subtracting the
# lower bound -- literally (x-0)/80, (x+20)/200, (x+150)/380 in the 2nd-place repo, which are
# exactly the lower bounds of these three windows.
HU_WINDOWS = [("brain", 40.0, 80.0), ("subdural", 80.0, 200.0), ("soft", 40.0, 380.0)]

RESAMPLE_INTERPOLATOR = "linear"         # "linear" | "bspline"

# CT reconstructs into a CIRCLE and stores it in a SQUARE, so 1 - pi/4 = 21.5% of every slice is
# outside the reconstructed field of view. Scanners fill that region with a padding value, and on
# this dataset it is the classic one: raw -2000, which after RescaleIntercept -1024 comes out as
# exactly -3024 HU. Two things follow.
#
# It is not anatomy, so it is clamped to air before anything measures the volume. And it is more
# than a fifth of every slice, so the 1st percentile lands squarely in it -- which is why the air
# check below rejected real series with "1st HU percentile -3024, the rescale intercept is wrong
# for this series" when the intercept was perfectly correct. The synthetic fixtures this was
# tested against reconstructed into the full square and had no padding at all, so the whole class
# of scan was invisible until it met the real archive.
# One file in this dataset is known to be corrupt, and it is the same one in every top solution's
# code. appian42 has a function called `remove_corrupted_images` whose entire body is a list with
# this single entry; okotaku defines it as a module constant `skip_id` and additionally repairs
# its neighbours' 2.5D references so the triplets do not break.
#   github.com/appian42/kaggle-rsna-intracranial-hemorrhage/blob/master/src/preprocess/dicom_to_dataframe.py
#   github.com/okotaku/kaggle_rsna2019_3rd_solution/blob/master/exp/exp18_seres_concat_three.py
#
# It matters MORE here than it did for them. They classified slices independently, so a bad file
# cost one slice. This pipeline stacks a series into a volume, so a slice that fails to decode
# leaves a HOLE, the z-uniformity check sees a doubled gap, and the whole series is rejected --
# one known-bad file taking out ~33 good ones. Naming it means the failure says what it is.
KNOWN_BAD_MEMBERS = {"ID_6431af929"}

HU_AIR_FLOOR = -1024.0                   # nothing in a head CT is legitimately below this
CLAMP_HU_FLOOR = True                    # out-of-FOV padding -> air, before QC and resampling
MAX_PADDING_FRACTION = 0.45              # more than this below the floor is not a FOV circle

# Automated QC. A volume whose air is not near -1000, or whose top percentile sits up where
# wrapped values live, is not one this pipeline should embed.
QC_HU_P01_RANGE = (-1200.0, -700.0)
QC_HU_P99_MAX = 3000.0                   # uncorrected 12-bit wraparound parks air at ~+3096
# Clamping the floor would hide an intercept that is wrong by a large NEGATIVE amount: every
# voxel would land on the floor and the air check would pass. A head CT always contains skull,
# so requiring the top percentile to reach bone closes that door.
QC_HU_P99_MIN = 300.0
QC_REJECT_ON_FAIL = True

# A series with ANY member that fails its CRC-32 is rejected by default. The damaged bytes belong
# to specific slices, but you cannot tell whether the damage stopped there -- and a volume built
# from the survivors is not distinguishable downstream from an intact one. Raise this only if a
# scan of your archive shows scattered damage you have decided to tolerate.
MAX_CRC_FAILURES_PER_SERIES = 0

# Z-UNIFORMITY. This one matters more than it looks. When slices are dropped -- damaged, duplicate
# SOP, odd shape, whatever -- the survivors are stacked as if contiguous, and the through-plane
# spacing is estimated as the MEDIAN gap, which one or two missing slices do not move. So the
# volume is resampled as though the hole were never there and the anatomy is silently compressed
# at that point. Nothing downstream can see it. Comparing the LARGEST gap to the median is what
# catches it: an intact stack has a ratio of ~1.0, a stack with one missing slice has ~2.0.
# The fixed grid is a centre crop, so a series longer than the grid loses anatomy at both
# ends. What matters is not how many series are cropped but whether the POSITIVE slices
# survive: a haemorrhage cropped out of a series that stays labelled positive is label noise
# the preprocessing invented, and it is invisible downstream. Cell 10 measures it and refuses
# to continue past this limit -- because both grid settings are in the preprocessing
# signature, so changing them after extraction discards every cached vector.
MAX_POSITIVE_SLICE_CROP = 0.005   # 0.5% of positive slices
ACKNOWLEDGE_GRID_CROP = False     # True = proceed anyway, on the record

REJECT_NONUNIFORM_Z = True
MAX_Z_GAP_RATIO = 1.75

# Skull-stripping (HD-BET / SynthStrip) is NOT implemented here; see the closing notes for why
# that matters. Left as a flag so the preprocessing signature changes when it lands.
SKULL_STRIP = False

# ==============================================================================================
# THE FROZEN BACKBONE  (Part 3)
# ==============================================================================================
# MONAI refuses pretrained=True for any n_input_channels != 1, so a 2-channel input would mean
# inflating conv1's weights -- which changes the pretrained function. Instead the frozen backbone
# runs once per window and the two 512-d vectors are concatenated, keeping the published weights
# bit-exact. Feature dimension = 512 * len(HU_WINDOWS).
# ---------------------------------------------------------------------------------------
# WHICH BACKBONE THIS RUN EXTRACTS.  <<< THE ONE LINE YOU CHANGE BETWEEN RUNS >>>
#
# Run this notebook ONCE, with all three backbones listed. The read is 95% of wall clock
# and is paid once for the whole list; running it once per backbone pays it three times
# to save nothing. If a checkpoint fails to load, comment that one line out and re-run.
#
#   "B1_swinunetr_suprem"    hybrid CNN-Transformer, SuPreM weights (2,100 CT volumes)
#   "B2_segresnet_suprem"    pure CNN,               SuPreM weights (2,100 CT volumes)
#   "B3_resnet18_medicalnet" pure CNN (classic),     MedicalNet weights (23 datasets)
#   "B4_swinunetr_ssl5050"   optional: B1's architecture, self-supervised weights instead
#
# B1 and B2 share a pretraining corpus AND a supervision signal, which is what makes
# B1-vs-B2 a controlled comparison of ARCHITECTURE rather than a confounded comparison of
# "which one saw more data". B3 changes the corpus while staying in the CNN family. B4
# changes the pretraining regime while holding the architecture fixed.
# A LIST. Every backbone named here is fed from the SAME pass over the archive, because
# Phase A is I/O bound: pulling one series out of the split zip costs ~10-30 s against
# ~0.2-2 s of GPU for even the heaviest backbone. Running the notebook once per backbone
# would pay the expensive part three times to save nothing.
#
# Leave all three in and run the notebook ONCE. Resumability is still per-backbone, so a
# backbone you already extracted in an earlier single-backbone run is skipped, and adding
# a fourth later re-reads only what that one needs.
BACKBONE_IDS = [
    "B1_swinunetr_suprem",      # hybrid CNN-Transformer, SuPreM (2,100 CT volumes)
    "B2_segresnet_suprem",      # pure CNN,             SuPreM (2,100 CT volumes)
    "B3_resnet18_medicalnet",   # pure CNN (classic),   MedicalNet (23 datasets)
]
BACKBONE_ID = BACKBONE_IDS[0]   # compatibility alias for cells that report a single name

# Where the downloaded SuPreM/SSL checkpoints are cached. On Drive, so the ~250 MB download
# happens once across all sessions rather than once per session.
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

# MONAI pulls the MedicalNet weights through huggingface_hub, whose cache defaults to the
# EPHEMERAL container filesystem -- so B3 re-downloads ~90 MB on every fresh Colab session,
# even though the SuPreM checkpoints sitting beside it are cached on Drive. This has to be
# set HERE, in the config, and not in the backbone cell: the preflight downloads the same
# weights several cells earlier, and a later HF_HOME would simply mean downloading twice.
os.makedirs(CKPT_DIR, exist_ok=True)
os.environ.setdefault("HF_HOME", os.path.join(CKPT_DIR, "hf"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", os.path.join(CKPT_DIR, "hf", "hub"))

# ---------------------------------------------------------------------------------------
# SPATIAL FEATURE MAPS -- what the Q3 (quanvolutional) arm needs
#
# Q3's whole claim is that the quantum circuit sits DEEPER in the pipeline: one circuit
# evaluation per spatial position of an intermediate feature map, not one evaluation on a
# pooled vector. That is impossible from a pooled vector alone, so Phase A caches a small
# adaptively-pooled map alongside it.
#
# Cost, per series, at SPATIAL_MAP_SIZE=2 and 3 windows:
#     B1  768ch x 8 pos x 3 windows x 2 bytes (fp16) =  36 KB   -> ~145 MB for 4,000 series
#     B2  128ch x 8 pos x 3 windows x 2 bytes         =   6 KB   -> ~ 25 MB
#     B3  512ch x 8 pos x 3 windows x 2 bytes         =  24 KB   -> ~ 98 MB
# Raising it to 3 gives 27 positions and ~3.4x the storage. 2 is the honest default: it is
# 8 genuine spatial positions, which is enough for the placement question to mean
# something, and it keeps Drive usage sane.
SAVE_SPATIAL_MAP = True
SPATIAL_MAP_SIZE = 2          # -> SPATIAL_MAP_SIZE**3 positions per window
SPATIAL_MAP_DTYPE = "float16" # storage only; everything is float32 in the graph
USE_AMP = True
EXTRACT_BATCH_SIZE = 1           # volumes per forward pass

# ---- how hard the progress log hits Drive ------------------------------------------------
# Each series appends one JSON record per backbone to a log on a FUSE-mounted Drive. On a
# measured run the writes were a quarter of wall clock, and an fsync on that mount is the
# most expensive part of them -- while, by Drive's own semantics, guaranteeing very little.
# Nothing here risks CORRECTNESS: a record that never reached disk simply means that series
# is re-extracted on the next run, which is idempotent. It costs time, not data.
PROGRESS_FSYNC = False           # True forces a per-record fsync (slow; rarely meaningful)
PROGRESS_FLUSH_EVERY = 10        # flush the log every N records; 1 = every record

# ---- running several sessions against ONE Drive folder ----------------------------------
# There are two ways to split the work, and they are not equally good.
#
#   BY BACKBONE (one notebook per backbone, SHARD_COUNT = 1)
#     Each session reads the whole archive for its own backbone. On the measured run,
#     reading was 88% of wall clock -- so this pays the expensive part three times and
#     each session takes about as long as all three would have taken together.
#
#   BY SERIES (every session runs ALL the backbones on 1/N of the cohort)
#     Set SHARD_COUNT = 3 and SHARD_INDEX = 0, 1, 2 in three sessions, with
#     BACKBONE_IDS listing all three backbones in each. Every volume is still read
#     exactly once, and the reading is divided N ways. This is roughly N times faster
#     end to end and produces byte-identical features.
#
# The shard is a position in the manifest, and the manifest is a deterministic function of
# MAX_SERIES and SAMPLE_SEED -- so every session agrees on which series belongs to which
# shard without any coordination, as long as those two values match across sessions.
SHARD_INDEX = 0                  # 0 .. SHARD_COUNT-1
SHARD_COUNT = 1                  # 1 = this session handles the whole manifest

MAX_SERIES_PER_RUN = None        # None = no cap
MAX_SECONDS_PER_RUN = None       # e.g. 10 * 3600
TRIAL_RUN_MAX_SERIES = None    # None = the full cohort. The 25-series trial already passed
                               # (25/25, 0 failures), so this runs unattended end to end.
CONSOLIDATE_EVERY_N_SERIES = 200

# ==============================================================================================
# THE QUANTUM ADAPTER AND ITS CONTROLS  (Part 4)
# ==============================================================================================
# Q1: Strongly Entangling Layers on the pooled bottleneck vector. The literature this follows uses
# 8 qubits and 4 SEL layers; this runs 4 and 3 on purpose -- 41 trainable parameters and about 30 s
# per fold, so a broken joint shows up in minutes rather than after an hour of watching a flat loss
# curve. The closing notes say exactly what to change to scale up.
N_QUBITS = 4
N_SEL_LAYERS = 3

# How the FEATURE_DIM backbone vector becomes N_QUBITS rotation angles. This matters more than the
# choice of ansatz, so the notebook runs all three rather than picking one and presenting it as
# neutral:
#   "trainable"       a trainable nn.Linear(1024 -> N_QUBITS) trained jointly with the circuit.
#                     Best performance. The projection holds most of the parameters, which is
#                     exactly why the classical control shares THE SAME projection and only the
#                     adapter tail is parameter-matched.
#   "pca_supervised"  PCA fitted on the training fold, then the N_QUBITS components with the
#                     strongest point-biserial correlation with the label are selected. Fit and
#                     selection are both training-fold-only, so nothing leaks. Nothing outside the
#                     adapter is trainable: 41 parameters total.
#   "pca"             plain top-variance PCA. Label-blind. The honest floor -- the gap between
#                     this and pca_supervised is the information bottleneck, not the adapter.
REDUCERS_TO_RUN = ["trainable", "pca_supervised", "pca"]

# `default.qubit` is a torch-native statevector simulator, so "backprop" is exact, fully batched,
# and measured far faster than "adjoint" (which falls back to a per-sample loop). Cell 19
# re-measures all of them on your runtime so this is evidence rather than assumption.
DIFF_METHOD = "backprop"          # "backprop" | "adjoint" | "parameter-shift"

QUANTUM_INIT_STD = 0.01           # near-identity init, to avoid barren plateaus
ADAPTER_EPOCHS = 80
ADAPTER_LR = 0.05
ADAPTER_BATCH = 128
ANGLE_L2_BETA = 1e-3              # L2 on the rotation angles, so they cannot drift past pi
PROJECTION_WEIGHT_DECAY = 1e-3    # on the trainable reducer only
FOCAL_ALPHA, FOCAL_GAMMA = 0.25, 2.0

# ==============================================================================================
# EVALUATION
# ==============================================================================================
# "any" is hemorrhage-present. Subtypes are included when they clear MIN_POSITIVES on the sample
# actually extracted; anything below it is dropped with a printed reason rather than evaluated on
# a handful of cases.
PILOT_TARGETS = ["any", "intraparenchymal", "subdural", "subarachnoid",
                 "intraventricular", "epidural"]
MIN_POSITIVES = 30
CV_FOLDS = 5
HOLDOUT_FRAC = 0.2
TARGET_SENSITIVITY = 0.85
N_BOOTSTRAP = 2000
RANDOM_SEED = 12345
NISQ_DEPOLARIZING_P = 0.01
FORCE_RERUN_PHASEB = False

# ==============================================================================================
# SMOKE-TEST OVERRIDES -- applied coherently, so the partial index actually yields volumes
# ==============================================================================================
if SMOKE_TEST:
    # 15k of ~753k members is 2%. A 35-slice series then keeps 0.7 slices on average, and the
    # binomial tail still leaves several hundred series holding 3 or more -- comfortably more than
    # the 40 wanted, while keeping the pass short. Cell 10 prints how many were eligible; if it is
    # under 40, raise this and re-run cell 9 (already-indexed chunks are kept).
    #
    # The time budget exists because these reads go over Drive, where throughput is latency-bound
    # and unpredictable. The index is chunked and resumable, so hitting the budget stops cleanly
    # rather than leaving you watching a progress bar.
    MAX_MEMBERS_TO_INDEX = 15000
    MAX_SECONDS_PER_RUN = 15 * 60
    SERIES_MIN_SLICES = 3          # a partial index gives ~1 slice per series; 3 is what survives
    REJECT_NONUNIFORM_Z = False    # a partial stack is nothing BUT holes
    # Enough volumes that Part 4 can actually hold patients out and run. Fewer than this and the
    # split has nothing to stratify, Part 4 skips every target, and the smoke test silently fails
    # to exercise the half of the notebook you most want exercised.
    MAX_SERIES = 40
    MIN_POSITIVES = 3
    CV_FOLDS = 2
    ADAPTER_EPOCHS = 20
    N_BOOTSTRAP = 200
    REDUCERS_TO_RUN = ["trainable"]
    ALLOW_CPU_EXTRACTION = True
    UNATTENDED_RUN = False

# ==============================================================================================
# DERIVED PATHS -- nothing below here normally needs editing
# ==============================================================================================
FEATURE_DIR          = os.path.join(OUTPUT_DIR, "features")
CD_CACHE_PATH        = os.path.join(OUTPUT_DIR, "member_table.parquet")
CD_CACHE_META        = os.path.join(OUTPUT_DIR, "member_table.meta.json")
INDEX_DIR            = os.path.join(OUTPUT_DIR, "header_index")
INDEX_MERGED_PATH    = os.path.join(OUTPUT_DIR, "header_index.parquet")
LABELS_CACHE_PATH    = os.path.join(OUTPUT_DIR, "slice_labels.parquet")
SERIES_TABLE_PATH    = os.path.join(OUTPUT_DIR, "series_table.parquet")
MANIFEST_PATH        = os.path.join(OUTPUT_DIR, "manifest.parquet")
# PER-BACKBONE. In the single-backbone notebook one shared progress log was right; with
# three backbones writing to it, backbone 2 would read backbone 1's "ok" records, skip
# every series, and produce an empty feature table that looks like a successful resume.
def progress_log_path(b):
    return os.path.join(OUTPUT_DIR, f"extraction_progress_{b}.jsonl")


def features_parquet_path(b):
    return os.path.join(OUTPUT_DIR, f"features_{b}.parquet")


PROGRESS_LOG_PATH    = progress_log_path(BACKBONE_ID)
SPATIAL_DIR          = os.path.join(OUTPUT_DIR, "spatial_maps")
# Per BACKBONE and per SHARD: a single shared path means three concurrent sessions
# truncate and rewrite the same file underneath each other, and the survivor is whichever
# one wrote last. These names never collide.
def failures_csv_path(backbone_id):
    _sfx = "" if SHARD_COUNT <= 1 else f"_shard{SHARD_INDEX}of{SHARD_COUNT}"
    return os.path.join(OUTPUT_DIR, f"failed_series_{backbone_id}{_sfx}.csv")


FAILURES_CSV_PATH    = failures_csv_path(BACKBONE_ID)
FEATURES_PARQUET     = features_parquet_path(BACKBONE_ID)
RESULTS_JSONL_PATH   = os.path.join(OUTPUT_DIR, "phaseb_results.jsonl")
RESULTS_CSV_PATH     = os.path.join(OUTPUT_DIR, "phaseb_leaderboard.csv")

for _d in (OUTPUT_DIR, FEATURE_DIR, INDEX_DIR, os.path.join(FEATURE_DIR, BACKBONE_ID)):
    os.makedirs(_d, exist_ok=True)

# ==============================================================================================
# TRANSIENT DRIVE ERRORS -- a weather report, not a verdict
# ==============================================================================================
# A FUSE-mounted Google Drive returns `OSError: [Errno 5] Input/output error` from time to time:
# under rate limiting, during a token refresh, on a network hiccup. It is almost always transient
# -- the identical read succeeds a second later. Nothing about the data is wrong.
#
# Left unhandled it does two very different kinds of damage here, and the second is the dangerous
# one:
#
#   1. It aborts a run. A real session died in the PREFLIGHT, on the first substantial Drive read,
#      before doing any work at all -- because one EIO on the labels CSV was treated as a verdict.
#   2. It gets CACHED AS A PERMANENT FAILURE. A member that hits EIO during the header index is
#      written into its chunk with an error string; the chunk is then marked done; and that slice
#      is dropped from its series for good. The series quietly comes out short, or with a hole in
#      it, and nothing ever says why. A blip becomes missing anatomy.
#
# So every Drive touch in this notebook goes through `drive_retry`, and the header index refuses
# to write a chunk that still contains transient errors after retrying.
DRIVE_RETRIES = 5                # attempts, not retries: 5 means 4 retries
DRIVE_RETRY_BASE_S = 0.8         # first backoff; doubles each time, with jitter

_TRANSIENT_ERRNOS = {getattr(errno, n) for n in
                     ("EIO", "EAGAIN", "EBUSY", "ENODEV", "ESTALE", "ETIMEDOUT", "EREMOTEIO",
                      "ENOTCONN", "ECONNRESET", "EINTR", "ENOLINK", "EHOSTDOWN")
                     if hasattr(errno, n)}
_TRANSIENT_TEXT = ("input/output error", "transport endpoint", "stale file handle",
                   "resource temporarily unavailable", "connection reset", "timed out")


def is_transient_io(e):
    """True for the failures a network filesystem produces and then stops producing.

    Deliberately NOT true for ValueError (a CRC mismatch is real damage), FileNotFoundError (the
    file is not there and will not be there in a second), or PermissionError. Retrying those just
    wastes time and hides the answer.
    """
    if isinstance(e, (TimeoutError, ConnectionError)):
        return True
    if isinstance(e, OSError):
        if e.errno in _TRANSIENT_ERRNOS:
            return True
        return any(t in str(e).lower() for t in _TRANSIENT_TEXT)
    return False


def drive_retry(fn, what="a Drive read", tries=None, base=None, on_retry=None, quiet=True):
    """Run `fn`, retrying only the failures that are worth retrying, with exponential backoff.

    `on_retry` is called before each retry -- the multi-part reader uses it to throw away a file
    handle that EIO may have poisoned, because reusing one gives EIO again forever.
    """
    tries = int(tries if tries is not None else DRIVE_RETRIES)
    base = float(base if base is not None else DRIVE_RETRY_BASE_S)
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if not is_transient_io(e) or attempt == tries - 1:
                raise
            if on_retry is not None:
                try:
                    on_retry()
                except Exception:
                    pass
            delay = base * (2 ** attempt) * (0.7 + 0.6 * random.random())
            if not quiet:
                print(f"    {what}: {type(e).__name__}: {e} -- retrying in {delay:.1f}s "
                      f"({attempt+1}/{tries-1})")
            time.sleep(delay)
    raise last


DRIVE_WEDGED_HINT = (
    "If this keeps happening the Drive mount is wedged rather than busy. In a Colab cell:\n"
    "    from google.colab import drive; drive.flush_and_unmount()\n"
    "    drive.mount(<drive_mount in config.yaml>)\n"
    "then re-run from cell 6. Everything already cached on Drive is kept.")

# ==============================================================================================
# SILENCE PYDICOM'S UID VALIDATION -- not cosmetic, this one kills runs
# ==============================================================================================
# Every RSNA-ICH file stores identifiers like 'ID_daf5bab46' in tags whose VR is UI. That is not a
# legal DICOM UID (UIDs are dotted digits), so pydicom validates it and complains -- for
# StudyInstanceUID and SeriesInstanceUID, on every member, through BOTH the `warnings` module and
# the `pydicom` logger. That is about 3 million messages over 752,803 members, every one of them
# streamed from the kernel to the browser.
#
# The consequences are not cosmetic. It bloats the .ipynb by megabytes, it makes the tab crawl,
# and Colab's output limiter is a documented cause of runtime disconnection -- which is exactly
# how a header index that was 87% done gets thrown away. The files are fine and the values are
# exactly what the labels join on; only the validator objects.
#
# Turning validation off at the source stops both channels. The two lines after it are belt and
# braces for any other library that decides to warn per member.
import logging as _logging
import warnings as _warnings

import pydicom as _pydicom

_PYDICOM_VALIDATION_SILENCED = False
try:
    _pydicom.config.settings.reading_validation_mode = _pydicom.config.IGNORE
    _PYDICOM_VALIDATION_SILENCED = True
except Exception as _e:                      # very old pydicom: fall back to filtering
    print(f"note: could not set pydicom reading_validation_mode ({_e}); filtering instead.")
_warnings.filterwarnings("ignore", message="Invalid value for VR")
_logging.getLogger("pydicom").setLevel(_logging.ERROR)


def _pydicom_silence_self_test():
    """Prove it, on a header carrying exactly the malformed UIDs this dataset uses. A regression
    here costs hours of runtime, so it is checked rather than assumed."""
    import io as _io

    from pydicom.dataset import Dataset as _DS, FileMetaDataset as _FM
    from pydicom.uid import ExplicitVRLittleEndian as _EVR
    _ds = _DS()
    _ds.PatientID = "ID_abc"
    _ds.SOPInstanceUID = "ID_daf5bab46"
    _ds.StudyInstanceUID = "ID_542e211224"
    _ds.SeriesInstanceUID = "ID_1e80f458ad"
    _fm = _FM()
    _fm.MediaStorageSOPClassUID = "1.2.840.10008.5.1.4.1.1.2"
    _fm.MediaStorageSOPInstanceUID = "ID_daf5bab46"
    _fm.TransferSyntaxUID = _EVR
    _ds.file_meta = _fm
    _buf = _io.BytesIO()
    try:
        _ds.save_as(_buf, enforce_file_format=True)       # pydicom >= 3
    except TypeError:
        _ds.is_little_endian, _ds.is_implicit_VR = True, False
        _ds.save_as(_buf, write_like_original=False)      # pydicom 2.x
    _raw = _buf.getvalue()

    class _Counter(_logging.Handler):
        def __init__(self):
            super().__init__()
            self.n = 0

        def emit(self, record):
            self.n += 1

    _h = _Counter()
    _lg = _logging.getLogger("pydicom")
    _lg.addHandler(_h)
    try:
        with _warnings.catch_warnings(record=True) as _w:
            _warnings.simplefilter("always")
            for _ in range(4):
                _d = _pydicom.dcmread(_io.BytesIO(_raw), stop_before_pixels=True, force=True)
                _ = str(_d.get("StudyInstanceUID", "")), str(_d.get("SeriesInstanceUID", ""))
        return len(_w), _h.n
    finally:
        _lg.removeHandler(_h)


_nw, _nl = _pydicom_silence_self_test()
if _nw or _nl:
    print(f"*** WARNING: pydicom still emitted {_nw} warning(s) and {_nl} log record(s) on four "
          f"reads of an RSNA-style header. Over 752,803 members that is ~{(_nw+_nl)*188_000:,.0f} "
          f"messages to the browser, which is enough to disconnect a Colab runtime. Check the "
          f"pydicom version before starting the header index.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# FEATURE_DIM is derived from the registry rather than hard-coded, because the three
# backbones do NOT share a width (768 / 128 / 512 per window). Hard-coding 512 here was
# correct for a one-backbone notebook and is a silent corruption in a three-backbone one:
# the feature table would load vectors of the wrong length and only the assertion in the
# feature-table cell would catch it, several hours later.
_BACKBONE_DIMS = {
    "B1_swinunetr_suprem": 768,
    "B2_segresnet_suprem": 128,
    "B3_resnet18_medicalnet": 512,
    "B4_swinunetr_ssl5050": 768,
}
for _b in BACKBONE_IDS:
    if _b not in _BACKBONE_DIMS:
        raise SystemExit(f"unknown backbone {_b!r}; choose from {list(_BACKBONE_DIMS)}")
if not BACKBONE_IDS:
    raise SystemExit("BACKBONE_IDS is empty -- nothing to extract.")
# Per-backbone, because the three do NOT share a width (768 / 128 / 512 per window).
FEATURE_DIMS = {b: _BACKBONE_DIMS[b] * len(HU_WINDOWS) for b in BACKBONE_IDS}
FEATURE_DIM = FEATURE_DIMS[BACKBONE_ID]
RSNA_SUBTYPES = ["epidural", "intraparenchymal", "intraventricular",
                 "subarachnoid", "subdural", "any"]

_fov = tuple(round(TARGET_SHAPE_DHW[i] * TARGET_SPACING_XYZ[::-1][i]) for i in range(3))
print(f"RSNA_ROOT       : {RSNA_ROOT}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")
print(f"archive prefix  : {ARCHIVE_PREFIX!r}"
      + ("   (cell 7 resolves this against the archive's own listing)"
         if ARCHIVE_PREFIX == "auto" else ""))
print(f"backbone        : {BACKBONE_ID}  ->  {FEATURE_DIM}-d "
      f"({len(HU_WINDOWS)} windows x {_BACKBONE_DIMS[BACKBONE_ID]}, frozen)")
print(f"volume grid     : {TARGET_SHAPE_DHW} (D,H,W) at {TARGET_SPACING_XYZ} mm "
      f"= {_fov[0]} x {_fov[1]} x {_fov[2]} mm FOV")
print(f"windows         : {[(n, f'C{c:.0f}/W{w:.0f}') for n, c, w in HU_WINDOWS]}")
print(f"quantum adapter : Q1 SEL, {N_QUBITS} qubits x {N_SEL_LAYERS} layers, "
      f"diff_method={DIFF_METHOD!r}")
print(f"reducers        : {REDUCERS_TO_RUN}")
print(f"max series      : {MAX_SERIES if MAX_SERIES is not None else 'all eligible'}")
print(f"DEVICE          : {DEVICE}  (AMP={'on' if USE_AMP and DEVICE == 'cuda' else 'off'})")
print(f"header index    : strategy={INDEX_STRATEGY}, workers={INDEX_WORKERS}"
      + ("   (cell 9 measures random vs bulk reads on your\n"
         "                    mount before committing, and re-measures if Drive throttles)"
         if "auto" in (str(INDEX_STRATEGY), str(INDEX_WORKERS)) else "   (fixed)"))
print(f"Drive I/O       : up to {DRIVE_RETRIES} attempts per read, backing off from "
      f"{DRIVE_RETRY_BASE_S:.1f}s   (Errno 5 is usually transient)")
print(f"pydicom UID validation : {'off' if _PYDICOM_VALIDATION_SILENCED else 'FILTERED ONLY'}"
      f"   ({_nw} warning(s) + {_nl} log record(s) on a 4-read self-test -- "
      f"RSNA's UIDs are not valid DICOM UIDs and would otherwise warn ~3M times)")
if SMOKE_TEST:
    print()
    print("*" * 94)
    print("SMOKE_TEST is ON. This is a plumbing check, not a result.")
    print(f"  index capped at {MAX_MEMBERS_TO_INDEX:,} members, so each series keeps only a few of")
    print(f"  its slices; SERIES_MIN_SLICES is lowered to {SERIES_MIN_SLICES} and the z-uniformity")
    print("  check is switched OFF to let those partial stacks through. The volumes that come out")
    print("  are anatomically meaningless -- every AUROC below is noise. What it proves is that the")
    print("  archive reads, the labels join, the index parses, a volume builds, the backbone runs")
    print("  and the circuit trains. Set SMOKE_TEST = False for anything you intend to believe.")
    print("*" * 94)
if TRIAL_RUN_MAX_SERIES is not None:
    print(f"\n*** TRIAL RUN: extraction stops after {TRIAL_RUN_MAX_SERIES} series. ***")
    print("    Set TRIAL_RUN_MAX_SERIES = None for the real run -- nothing from the trial is")
    print("    wasted, those series are logged 'ok' and skipped next time.")
if DEVICE == "cpu":
    import torch as _t
    _cpu_only_wheel = (_t.version.cuda is None)
    print()
    print("*** No GPU visible to torch. ***")
    if _cpu_only_wheel:
        print("  torch itself is a CPU-only build (torch.version.cuda is None), so even attaching a")
        print("  GPU runtime will not help until torch is reinstalled with CUDA. On Colab this")
        print("  normally means the runtime type was CPU when the session started: change it under")
        print("  Runtime > Change runtime type > GPU and then Runtime > Restart session.")
    else:
        print("  torch has CUDA support but no device is attached:")
        print("  Runtime > Change runtime type > GPU, then Runtime > Restart session.")
    print("  This does NOT block Parts 1-2. The header index is the long step and it is pure I/O --")
    print("  it runs at the same speed here. Only cell 14 wants a GPU.")

In [ ]:
# 5. PREFLIGHT -- fail in the next two minutes, or not at all tonight
#
# Every check below covers something that CAN fail and, if it does, fails better now than at 3am
# with nine hours of GPU time already spent: the GPU, the checkpoint download, the quantum
# simulator, the parts, the labels CSV, the output directory, disk, and the run budget.
import os
import shutil
import time

import torch

_fatal, _warn = [], []


def _probe_readable(path, budget_s=60.0, block=4 << 20):
    """When a Drive read keeps failing, say WHICH of the two things is wrong.

    A busy mount and a damaged file produce the identical `[Errno 5]`, and the remedies are
    opposites: wait / remount, versus re-upload the file. So on a surviving failure this reads the
    file sequentially and reports how far it got. Stopping at the same byte offset every time is a
    damaged or truncated upload; failing immediately, or getting a different distance each time,
    is the mount.
    """
    bits = []
    try:
        size = os.path.getsize(path)
        bits.append(f"stat: {size/1e6:.1f} MB")
    except Exception as e:
        return f"stat() also failed ({type(e).__name__}), so Drive cannot see the file at all -- " \
               f"that is the mount, not the file."
    t0, read = time.time(), 0
    try:
        with open(path, "rb") as f:
            while read < size and time.time() - t0 < budget_s:
                b = f.read(block)
                if not b:
                    break
                read += len(b)
    except Exception as e:
        return (f"{'; '.join(bits)}; a sequential read failed after {read/1e6:.1f} MB with "
                f"{type(e).__name__}. If it stops at the same place on a retry, the file on Drive "
                f"is damaged or the upload never finished -- re-upload it. If the distance varies, "
                f"it is the mount.")
    if read >= size:
        bits.append(f"but a full sequential read of all {read/1e6:.1f} MB succeeded just now")
        bits.append("so the file is intact and the mount was momentarily busy")
    else:
        bits.append(f"read {read/1e6:.1f} MB in {budget_s:.0f}s without error but did not finish "
                    f"-- the mount is very slow rather than broken")
    return "; ".join(bits)


_transient = []


def _check(name, fn, fatal=True):
    """Runs a check, retrying the failures that a network filesystem invents and then forgets.

    A transient I/O error that survives every retry is reported as a WARNING, not a failure, even
    when the check is fatal. That is a deliberate reversal: a real run died right here, on the
    first substantial Drive read, because one `[Errno 5] Input/output error` on the labels CSV
    aborted the whole unattended session before any work had been done. Drive being briefly busy
    is the condition this notebook is built to survive -- every read it does later retries -- so
    refusing to start because of it is exactly backwards. A missing file or a permission problem
    is a different thing entirely and stays fatal.
    """
    t0 = time.time()
    try:
        msg = drive_retry(fn, what=name, quiet=False)
        print(f"  [ok]   {name:<34s} {msg}   ({time.time()-t0:.1f}s)")
        return True
    except Exception as e:
        detail = f"{type(e).__name__}: {e}"
        if is_transient_io(e):
            print(f"  [warn] {name:<34s} {detail}")
            print(f"         Drive returned a transient I/O error {DRIVE_RETRIES} times. Every "
                  f"read this notebook makes later retries the same way, so this is a warning")
            print(f"         rather than a stop -- but if the next cells fail too, the mount is "
                  f"wedged rather than busy.")
            _warn.append(f"{name}: {detail} (transient I/O, retried {DRIVE_RETRIES}x)")
            _transient.append(name)
            return False
        print(f"  [{'FAIL' if fatal else 'warn'}] {name:<34s} {detail}")
        (_fatal if fatal else _warn).append(f"{name}: {detail}")
        return False


print("=" * 94)
print(f"PREFLIGHT   backbones={','.join(BACKBONE_IDS)}   unattended={UNATTENDED_RUN}"
      + (f"   shard {SHARD_INDEX+1}/{SHARD_COUNT}" if SHARD_COUNT > 1 else "")
      + ("   SMOKE_TEST" if SMOKE_TEST else ""))
print("=" * 94)

# ---- running more than one session against this Drive folder ---------------------------
# Feature files, spatial maps and progress logs are per backbone (and the failure CSVs are
# per backbone and per shard), so those never collide. The INDEX CACHES are different: the
# member table, the header index and the manifest are built once and shared, and building
# them concurrently in three sessions is three times the Drive reads for one result, with
# three sessions writing the same paths.
#
# They are all written temp-then-renamed, so a race corrupts nothing -- it just wastes
# hours. Build them once, then start the others.
_shared_caches = {
    "member table":  CD_CACHE_PATH,
    "header index":  INDEX_MERGED_PATH,
    "series table":  SERIES_TABLE_PATH,
    "manifest":      MANIFEST_PATH,
}
_missing_caches = [k for k, v in _shared_caches.items() if not os.path.exists(v)]
if _missing_caches:
    print()
    print(f"FIRST RUN AGAINST THIS OUTPUT_DIR -- these shared caches do not exist yet:")
    print(f"  {', '.join(_missing_caches)}")
    print(f"  If you intend to run several notebooks against this folder at once, RUN THIS")
    print(f"  ONE ALONE until the extraction cell starts, then launch the others. They")
    print(f"  will find the caches already built and skip straight to extracting.")
    print(f"  (Nothing breaks if you start them together -- every cache is written")
    print(f"   temp-then-renamed. You would simply pay for the index three times.)")
else:
    print(f"shared caches present -- safe to run alongside other sessions on this folder")
print()


def _gpu():
    if not torch.cuda.is_available():
        why = ("torch is a CPU-only build (torch.version.cuda is None)"
               if torch.version.cuda is None else "no CUDA device is attached")
        raise RuntimeError(
            f"{why}. Parts 1-2 are unaffected -- the header index is the long step and it is pure "
            f"I/O. Only cell 14 wants a GPU, and it will refuse a large CPU extraction unless "
            f"ALLOW_CPU_EXTRACTION = True.")
    p = torch.cuda.get_device_properties(0)
    return f"{p.name}, {p.total_memory/1e9:.1f} GB"


# NOT fatal. Refusing to start because the SHORT step would be slow, when the LONG step needs no
# GPU at all, gets the priority backwards -- and it is exactly what stopped the first real run.
_check("GPU", _gpu, fatal=False)


_PREFLIGHT_CKPTS = {
    "B1_swinunetr_suprem": ("suprem_swinunetr",
        "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/supervised_suprem_swinunetr_2100.pth"),
    "B2_segresnet_suprem": ("suprem_segresnet",
        "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/supervised_suprem_segresnet_2100.pth"),
    "B4_swinunetr_ssl5050": ("ssl_swinunetr_5050",
        "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/self_supervised_nv_swin_unetr_5050.pt"),
}


def _weights_for(_bid):
    """Fetch the weights THIS run needs, not whichever the notebook used to need."""
    if _bid == "B3_resnet18_medicalnet":
        from monai.networks.nets import resnet18
        m = resnet18(pretrained=True, spatial_dims=3, n_input_channels=1,
                     feed_forward=False, shortcut_type="A", bias_downsample=True)
        n = sum(p.numel() for p in m.parameters())
        del m
        return f"MedicalNet resnet18, {n:,} parameters"
    key, url = _PREFLIGHT_CKPTS[_bid]
    os.makedirs(CKPT_DIR, exist_ok=True)
    dest = os.path.join(CKPT_DIR, os.path.basename(url))
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        return f"{os.path.basename(dest)} already cached ({os.path.getsize(dest)/1e6:.0f} MB)"
    try:
        from huggingface_hub import hf_hub_download
        got = hf_hub_download(repo_id="MrGiovanni/SuPreM",
                              filename=os.path.basename(url), local_dir=CKPT_DIR)
        return f"{os.path.basename(got)} downloaded ({os.path.getsize(got)/1e6:.0f} MB)"
    except Exception:
        import urllib.request
        tmp = dest + ".part"
        urllib.request.urlretrieve(url, tmp)
        os.replace(tmp, dest)
        return f"{os.path.basename(dest)} downloaded ({os.path.getsize(dest)/1e6:.0f} MB)"


if PREFLIGHT_DOWNLOAD_MODEL:
    for _bid in BACKBONE_IDS:
        _check(f"checkpoint for {_bid}", (lambda b=_bid: _weights_for(b)))
else:
    print(f"  [skip] checkpoints for {len(BACKBONE_IDS)} backbone(s)   "
          f"PREFLIGHT_DOWNLOAD_MODEL = False")


def _pennylane():
    import pennylane as qml
    dev = qml.device("default.qubit", wires=N_QUBITS)

    @qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
    def c(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]

    shp = qml.StronglyEntanglingLayers.shape(n_layers=N_SEL_LAYERS, n_wires=N_QUBITS)
    w = torch.zeros(shp, requires_grad=True)
    torch.stack(c(torch.zeros(N_QUBITS), w)).sum().backward()
    if w.grad is None:
        raise RuntimeError(f"no gradient flowed with diff_method={DIFF_METHOD!r}")
    return (f"pennylane {qml.__version__}, {N_QUBITS} qubits, SEL shape {tuple(shp)}, "
            f"gradients flow")


_check("PennyLane / default.qubit", _pennylane)


# ---- when the data is not where the config says, go and find it ---------------------------
def locate_dataset(max_dirs=4000, max_depth=5):
    """Bounded search of the Drive for the split parts and the labels CSV.

    Bounded on purpose: a full walk of somebody's Drive can take minutes and this runs inside
    a preflight whose entire promise is that it finishes in two. It looks in the likely places
    first, stops at the first hit of each kind, and gives up quietly rather than hanging.
    """
    import re
    rx = re.compile(PARTS_PATTERN)
    roots = [p for p in [P["drive_root"], os.path.join(P["drive_mount"], "Shareddrives"),
                         os.path.dirname(RSNA_ROOT.rstrip("/")), "/content"]
             if p and os.path.isdir(p)]
    seen, found_parts, found_csv, n = set(), [], [], 0
    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath in seen:
                continue
            seen.add(dirpath)
            n += 1
            if n > max_dirs or (found_parts and found_csv):
                break
            depth = dirpath[len(root):].count(os.sep)
            if depth >= max_depth:
                dirnames[:] = []
                continue
            # never descend into our own output tree: thousands of feature files, no data
            dirnames[:] = [d for d in dirnames
                           if not d.startswith(".")
                           and os.path.join(dirpath, d) != OUTPUT_DIR
                           and d not in ("features", "spatial_maps", "header_index",
                                         "checkpoints", "figures", "qig")]
            if not found_parts and sum(1 for f in filenames if rx.match(f)) >= 2:
                found_parts.append(dirpath)
            if not found_csv:
                for f in filenames:
                    if f.lower() == os.path.basename(LABELS_CSV).lower():
                        found_csv.append(os.path.join(dirpath, f))
                        break
        if found_parts and found_csv:
            break
    return found_parts, found_csv, n


def _suggest_paths():
    print()
    print("  LOOKING for the archive elsewhere on this Drive (bounded search) ...")
    try:
        parts, csvs, n = locate_dataset()
    except Exception as e:
        print(f"    search failed ({type(e).__name__}: {e})")
        return
    print(f"    {n:,} directories examined")
    if not parts and not csvs:
        print("    nothing found. Check that Drive is mounted and that the folder is in")
        print("    MyDrive rather than a shortcut to someone else's drive (a shortcut can")
        print("    look like a folder in the web UI and read as a FILE through the mount --")
        print("    which is exactly what NotADirectoryError means here).")
        return
    if parts:
        print(f"    parts directory : {parts[0]}")
    if csvs:
        print(f"    labels CSV      : {csvs[0]}")
    print()
    print("    Paste into the configuration cell (cell 4) and re-run from there:")
    if parts:
        print(f'        PARTS_DIR  = "{parts[0]}"')
    if csvs:
        print(f'        LABELS_CSV = "{csvs[0]}"')
    if parts and csvs:
        _common = os.path.commonpath([parts[0], os.path.dirname(csvs[0])])
        print(f"    (or set RSNA_ROOT = \"{_common}\" if the layout underneath matches)")
    print()
    print("    Nothing already cached is affected: the member table, header index and")
    print("    manifest live in OUTPUT_DIR and are keyed by content, not by this path.")


def _parts():
    import re
    if not os.path.isdir(PARTS_DIR):
        raise NotADirectoryError(PARTS_DIR)
    rx = re.compile(PARTS_PATTERN)
    names = sorted(n for n in os.listdir(PARTS_DIR) if rx.match(n))
    if not names:
        raise FileNotFoundError(
            f"no files matching {PARTS_PATTERN!r} in {PARTS_DIR}. Set PARTS_PATTERN if your split "
            f"used a different naming scheme (e.g. 'xa*' from a bare `split`).")
    sizes = [os.path.getsize(os.path.join(PARTS_DIR, n)) for n in names]
    # `split` produces equal parts except the last. A short part in the MIDDLE means an upload is
    # still in flight or a part is truncated, and reading past it silently corrupts every member
    # after it -- so this refuses rather than guessing.
    if len(sizes) > 1:
        bad = [(names[i], sizes[i]) for i in range(len(sizes) - 1) if sizes[i] != sizes[0]]
        if bad:
            raise RuntimeError(
                f"part(s) {bad[:4]} differ in size from part 0 ({sizes[0]:,} B) but are not last. "
                f"Either an upload is still running or a part is truncated.")
    return (f"{len(names)} part(s), {sum(sizes)/1e9:.1f} GB total, "
            f"{sizes[0]/1e9:.2f} GB each + {sizes[-1]/1e9:.2f} GB last")


_check("split archive parts", _parts)


def _labels():
    if not os.path.exists(LABELS_CSV):
        raise FileNotFoundError(LABELS_CSV)
    import pandas as pd
    try:
        head = pd.read_csv(LABELS_CSV, nrows=5)
    except OSError as e:
        if is_transient_io(e):
            # Add the one piece of information that decides what to do about it.
            raise OSError(e.errno, f"{e.strerror or e}  [{_probe_readable(LABELS_CSV)}]") from e
        raise
    cols = {c.lower() for c in head.columns}
    if not {"id", "label"} <= cols:
        raise ValueError(f"expected ID and Label columns, found {list(head.columns)}")
    ex = str(head.iloc[0][[c for c in head.columns if c.lower() == "id"][0]])
    if "_" not in ex:
        raise ValueError(f"first ID {ex!r} has no underscore -- expected ID_<hex>_<subtype>")
    return f"{os.path.getsize(LABELS_CSV)/1e6:.0f} MB, e.g. {ex}"


_check("labels CSV", _labels)


def _out():
    os.makedirs(FEATURE_DIR, exist_ok=True)
    probe = os.path.join(OUTPUT_DIR, ".write_probe")
    with open(probe, "w") as f:
        f.write("ok")
    os.remove(probe)
    return f"writable: {OUTPUT_DIR}"


def _disk():
    free = shutil.disk_usage("/content" if os.path.isdir("/content") else ".").free / 1e9
    return f"{free:.0f} GB free locally (the archive is never reassembled, so this is scratch only)"


_check("output directory", _out)
_check("local disk", _disk, fatal=False)


def _budget():
    if UNATTENDED_RUN and TRIAL_RUN_MAX_SERIES is not None:
        raise RuntimeError(
            f"UNATTENDED_RUN is True but TRIAL_RUN_MAX_SERIES = {TRIAL_RUN_MAX_SERIES}. The run "
            f"would stop early and report success. Set one of them.")
    return "no trial cap in an unattended run"


_check("run budget sanity", _budget)

print()
if _warn:
    print(f"{len(_warn)} warning(s):")
    for w in _warn:
        print(f"  - {w}")
if _fatal:
    print(f"{len(_fatal)} FAILED check(s):")
    if any("NotADirectoryError" in f for f in _fatal):
        print("  NOTE: NotADirectoryError means a component of that path EXISTS but is not a")
        print("        folder -- typically a Drive shortcut, or the dataset still zipped.")
    if any(("split archive parts" in f) or ("labels CSV" in f) for f in _fatal):
        _suggest_paths()
    for f in _fatal:
        print(f"  - {f}")
    if UNATTENDED_RUN:
        raise RuntimeError(
            "Preflight failed and UNATTENDED_RUN is True, so this stops rather than starting a run "
            "that cannot finish. Fix the items above, or set UNATTENDED_RUN = False to proceed "
            "with warnings while you work interactively.")
    print("\nUNATTENDED_RUN is False -- continuing anyway.")
else:
    print("preflight clean." if not _transient else
          f"preflight passed, but {len(_transient)} check(s) hit transient Drive I/O errors: "
          f"{', '.join(_transient)}.")
if _transient:
    print()
    print(DRIVE_WEDGED_HINT)

---

## Part 1 — reading a 193 GB split archive without reassembling it

`split -b 2G` left 91 parts. Concatenating them back would need 193 GB of scratch, which Colab
does not have and which is not needed: the parts are presented as **one seekable byte stream**
and `zipfile` does random access over it, so only the bytes actually wanted ever move.

Two things are verified before anything is trusted, because both failures are silent:

* **Part sizes.** `split` produces equal parts except the last. A short part *in the middle* means
  an upload is still in flight or a part is truncated — and reading past it does not raise, it
  corrupts every member after it. The preflight already refused on that; the reader re-checks.
* **Archive format.** The first 512 bytes are read and the magic bytes identified. Your part_00000
  is a zip; if that ever changes, this stops rather than handing garbage to `zipfile`.

### The 4 GiB offset trap

A zip's central directory stores each member's position in a **32-bit** field. Past 4 GiB that
field is supposed to be superseded by a 64-bit ZIP64 extra field — but a great many writers, and
almost every tool that *rewrites* a central directory it did not create, simply store
`true_offset % 2**32` and move on. The archive still opens. Every member past the 4 GiB mark then
points 4 GiB (or 8, or 12…) too early, and `zipfile` reports it as `Bad magic number for file
header`, `Truncated file header`, or — the interesting one — `Overlapped entries (possible zip
bomb)`.

This is not hypothetical for this project: an earlier extraction run over 102 day-zip archives
needed this repair on **102 of 102** of them. Your RSNA archive is 194.7 GB, which is 45 wraps past
the boundary, so if its directory has this defect every single image read fails.

Cell 7 checks for it in a few reads and repairs it if present. The repair does **not** probe
members one at a time — on Drive that is one network round trip per member, 874,037 of them, with
no bound on the time. Instead it uses the one invariant a zip does guarantee: members appear in the
central directory in the order they were written, so their true offsets increase monotonically.
Every backwards jump in the stored low word is a wrap. One linear pass reconstructs the high word
for all of them; the absolute base is pinned by probing the first member, and the result is checked
on a spread sample of twelve before being applied wholesale. Anything it still gets wrong fails its
own CRC-checked read and drops one series with a stated reason, which is a bounded cost.

The detector stays silent on a correctly written ZIP64 archive — a false positive there would
"repair" offsets that were already right, which is worse than not looking at all — and that case is
covered by a test on a real 4.87 GB archive built both ways.

### Reading 4 KB out of a 500 KB member, for real this time

The member reader below deliberately does **not** go through `ZipFile.open()`. It parses each
member's local file header and decompresses only as far as asked, which is what makes Part 2's
header pass affordable. It is also what makes threading possible — a `ZipFile` cannot be shared
across threads, and re-opening one per thread would re-read the whole central directory each time.

Two details of *how* it does that were wrong in an earlier version of this notebook and cost real
hours on Drive. Both are worth stating, because the corrected numbers are what the ETA in Part 2
now rests on.

**Your archive is 100% DEFLATED, not stored** — 752,803 members, 395.2 GB uncompressed inside
194.7 GB of parts. The old claim that "asking for 4 KB transfers exactly 4 KB" was only ever true
for a *stored* member. A deflated one has to be fed to zlib, and the reader fed it in fixed 64 KB
chunks, so producing 4 KB of header pulled 64 KB across. Measured on a real 513 KB RSNA-sized
slice: **16× amplification**, which over 752,803 members is 49 GB of Drive traffic to collect 3 GB
of headers. The chunk now starts at about what the caller asked for and doubles only if that was
not enough, so the ordinary case is one ~4 KB chunk and a pathological fat header still gets its
64 KB retry.

**Every member cost two round trips.** The central directory says where a member's *local header*
is, not where its data starts, and the local header carries its own filename and extra fields whose
lengths vary — so the data offset genuinely has to be read. It does not need a request of its own:
one read covering the header, a 512-byte reservation for the variable part, and the first data
chunk gets everything at once. On a FUSE-mounted Drive, where a 4 KB read and a 64 KB read cost
almost the same and it is the *round trip* that hurts, this is the larger of the two wins.

Measured together on 200 real-sized deflated members: **65,566 → 4,638 bytes and 2.00 → 1.00 reads
per member**, with every full read still byte-identical to `ZipFile.read()` and every CRC still
verified.

In [ ]:
# 6. One seekable stream over the parts, and what kind of archive it is
import io
import os
import re
import struct
import threading
import zipfile
import zlib

import numpy as np


class MultiPartFile(io.RawIOBase):
    """Read-only seekable stream over `split`-produced parts, presented as one file.

    Deliberately RAW (unbuffered). Wrapping this in a large BufferedReader is exactly wrong for
    the header pass: we read ~4 KB out of every ~500 KB member, so a 32 MB read-ahead would pull
    the entire archive across the network to collect a few hundred bytes each time.
    """

    def __init__(self, paths, sizes=None):
        """`sizes`, when given, skips one os.path.getsize() per part.

        That sounds like a micro-optimisation and is not. A reader used to be constructed per
        worker per work unit, and each construction stat'd all 91 parts: 220,000 metadata calls
        across the header index and 2.9 MILLION across a 4,000-series extraction -- 21 stat calls
        for every slice actually read. On a FUSE-mounted Drive those are not free, and they are
        pure waste: the sizes were already measured once in `discover_parts`.
        """
        super().__init__()
        if not paths:
            raise ValueError("no parts given")
        self._paths = list(paths)
        self._sizes = ([int(x) for x in sizes] if sizes is not None
                       else [os.path.getsize(p) for p in self._paths])
        if len(self._sizes) != len(self._paths):
            raise ValueError("sizes does not match paths")
        self._starts, acc = [], 0
        for s in self._sizes:
            self._starts.append(acc)
            acc += s
        self._total = acc
        self._pos, self._fh, self._fh_idx = 0, None, -1

    @property
    def size(self):
        return self._total

    def readable(self):
        return True

    def seekable(self):
        return True

    def writable(self):
        return False

    def tell(self):
        return self._pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            new = offset
        elif whence == io.SEEK_CUR:
            new = self._pos + offset
        elif whence == io.SEEK_END:
            new = self._total + offset
        else:
            raise ValueError(f"bad whence {whence}")
        if new < 0:
            raise OSError("negative seek")
        self._pos = new          # seeking past the end is legal; a read there returns b""
        return self._pos

    def _part_for(self, pos):
        lo, hi = 0, len(self._paths) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if self._starts[mid] <= pos:
                lo = mid
            else:
                hi = mid - 1
        return lo

    def _ensure(self, idx):
        if self._fh_idx != idx:
            if self._fh is not None:
                try:
                    self._fh.close()
                except Exception:
                    pass
            self._fh = open(self._paths[idx], "rb")
            self._fh_idx = idx
        return self._fh

    def _drop_handle(self):
        """Throw away the open file object. After an EIO on a FUSE mount the handle can stay
        poisoned -- every subsequent read on it returns EIO again -- so a retry that reuses it
        retries nothing. Reopening is what actually clears it."""
        if self._fh is not None:
            try:
                self._fh.close()
            except Exception:
                pass
        self._fh, self._fh_idx = None, -1

    def _read_at(self, idx, local, n):
        """One physical read, retried through the transient errors Drive produces."""
        def _once():
            fh = self._ensure(idx)
            if fh.tell() != local:
                fh.seek(local)
            return fh.read(n)
        return drive_retry(_once, what=f"read of {os.path.basename(self._paths[idx])}",
                           on_retry=self._drop_handle)

    def readinto(self, b):
        want = len(b)
        if want == 0 or self._pos >= self._total:
            return 0
        got, mv = 0, memoryview(b)
        while got < want and self._pos < self._total:
            idx = self._part_for(self._pos)
            local = self._pos - self._starts[idx]
            chunk = self._read_at(idx, local, min(want - got, self._sizes[idx] - local))
            if not chunk:
                break            # short/truncated part: stop rather than spin
            mv[got:got + len(chunk)] = chunk
            got += len(chunk)
            self._pos += len(chunk)
        return got

    def read(self, n=-1):
        if n is None or n < 0:
            n = max(0, self._total - self._pos)
        buf = bytearray(n)
        got = self.readinto(buf)
        return bytes(buf[:got])

    def close(self):
        if self._fh is not None:
            try:
                self._fh.close()
            finally:
                self._fh, self._fh_idx = None, -1
        super().close()


# ---- one reader per thread, for the life of the thread ------------------------------------------
# A MultiPartFile carries a file position, so it cannot be shared between threads. The obvious
# response -- build one per work unit -- is what produced the stat storm described above, plus 91
# file opens and closes each time. A thread-local reader is shared safely (each thread has its
# own) and constructed once.
_READERS = threading.local()


def thread_reader():
    r = getattr(_READERS, "mpf", None)
    if r is None:
        r = MultiPartFile(PART_PATHS, sizes=PART_SIZES)
        _READERS.mpf = r
    return r


def discover_parts(parts_dir=None, pattern=None):
    parts_dir = parts_dir or PARTS_DIR
    rx = re.compile(pattern or PARTS_PATTERN)
    names = sorted(n for n in drive_retry(lambda: os.listdir(parts_dir),
                                          what=f"listing {parts_dir}") if rx.match(n))
    if not names:
        raise FileNotFoundError(f"no files matching {rx.pattern!r} in {parts_dir}")
    paths = [os.path.join(parts_dir, n) for n in names]
    sizes = [drive_retry(lambda q=p: os.path.getsize(q), what=f"size of {os.path.basename(p)}")
             for p in paths]
    if len(sizes) > 1:
        bad = [(names[i], sizes[i]) for i in range(len(sizes) - 1) if sizes[i] != sizes[0]]
        if bad:
            raise RuntimeError(
                f"part(s) {bad[:4]} are not the same size as part 0 ({sizes[0]:,} bytes) but are "
                f"not last. `split` produces equal parts except the final one, so either an upload "
                f"is still running or a part is truncated. Reading past a short part silently "
                f"corrupts every member after it, so this refuses rather than guessing.")
    return paths, sizes


def detect_format(mpf):
    mpf.seek(0)
    head = mpf.read(512)
    if head[:4] in (b"PK\x03\x04", b"PK\x05\x06", b"PK\x06\x06"):
        return "zip"
    if len(head) >= 262 and head[257:262] == b"ustar":
        return "tar"
    if head[:2] == b"\x1f\x8b":
        return "gzip"
    if head[:6] == b"\xfd7zXZ\x00":
        return "xz"
    if head[:3] == b"BZh":
        return "bzip2"
    if head[:6] == b"7z\xbc\xaf\x27\x1c":
        return "7z"
    if head[:4] == b"Rar!":
        return "rar"
    return "unknown"


# ---- direct member access, without ZipFile ------------------------------------------------------
# The central directory records where each member's LOCAL HEADER is, not where its DATA starts --
# and the local header carries its own filename and extra fields, whose lengths vary per member.
# So the data offset has to be read; it cannot be derived from the central directory alone.
_LFH = struct.Struct("<4sHHHHHIIIHH")      # 30-byte local file header


def _verify_crc(blob, expected, header_offset):
    got = zlib.crc32(blob) & 0xFFFFFFFF
    if got != (int(expected) & 0xFFFFFFFF):
        raise ValueError(
            f"CRC mismatch on the member at offset {header_offset}: the zip recorded "
            f"{int(expected) & 0xFFFFFFFF:08x}, the bytes read hash to {got:08x}. Those bytes are "
            f"not what was archived -- a part is damaged, or the parts are out of order.")


# ---- the 32-bit central-directory offset overflow, and how to survive it -----------------------
# A zip stores each member's local-header offset as a 32-bit field in the central directory. Past
# 4 GiB that field is supposed to be replaced by a 64-bit ZIP64 extra field -- but plenty of
# writers (and plenty of re-packagers that rewrite a central directory they did not create) simply
# store `true_offset % 2**32` instead. The archive is then internally inconsistent: every member
# past the 4 GiB mark points at a byte position that is 4 GiB (or 8, or 12, ...) too small.
#
# Python's zipfile does not detect this. What it does do is compute
#     concat = <EOCD position> - size_cd - offset_cd
# and add `concat` to every header offset. When offset_cd itself has wrapped, `concat` comes out as
# floor(true_cd_start / 2**32) * 2**32 -- an exact multiple of 2**32 -- so `zi.header_offset %
# 2**32` recovers the raw stored low word, which is what the repair below starts from.
#
# The repair: members appear in the central directory in the order they were written, so their TRUE
# offsets are monotonically increasing. Every place the raw low word jumps BACKWARDS is a wrap.
# Counting those reconstructs the high word for every member from one linear pass, with the
# absolute base pinned by probing the first member. That is a handful of reads in total.
#
# What is deliberately NOT done: probing every member individually. On Drive that is one network
# round trip per member -- hundreds of thousands of them, unbounded, for an archive this size. The
# reconstruction is applied wholesale after being verified on a spread sample; any member it still
# gets wrong fails its own read with a stated reason, which is a bounded, per-series cost.
ZIP32_WRAP = 1 << 32


def _read_local_header(mpf, offset):
    """(name_bytes, compress_size, data_offset) for a local file header at `offset`, else None."""
    if offset < 0 or offset + 30 > mpf.size:
        return None
    mpf.seek(offset)
    raw = mpf.read(30)
    if len(raw) < 30:
        return None
    sig, _v, _f, _m, _t, _d, _c, cs, _us, fn_len, ex_len = _LFH.unpack(raw)
    if sig != b"PK\x03\x04" or fn_len == 0 or fn_len > 4096:
        return None
    name = mpf.read(fn_len)
    if len(name) < fn_len:
        return None
    return name, cs, offset + 30 + fn_len + ex_len


def _name_byte_candidates(name):
    out = []
    for enc in ("utf-8", "cp437"):
        try:
            out.append(str(name).encode(enc, "surrogateescape" if enc == "utf-8" else "strict"))
        except Exception:
            pass
    return out


def local_header_matches(mpf, offset, name, compress_size=None):
    """True only if the bytes at `offset` are the local header OF THIS MEMBER.

    Checking the PK\x03\x04 signature alone is not enough: at a 4 GiB stride through an archive
    made of ~500 KB members, a wrong candidate offset lands on *some* member's header often enough
    to matter. The filename has to match too.
    """
    got = _read_local_header(mpf, offset)
    if got is None:
        return False
    got_name, got_cs, _ = got
    # zipfile decodes a member name as UTF-8 or as cp437 depending on a flag bit, so re-encoding
    # has to allow both to get back the bytes that are actually in the local header. RSNA's names
    # are pure ASCII and identical either way; this is here so the check cannot false-negative on
    # an archive whose names are not.
    if got_name not in _name_byte_candidates(name):
        return False
    if compress_size is not None:
        # 0 means "sizes are in the data descriptor"; 0xFFFFFFFF means "see the ZIP64 extra field".
        if got_cs not in (0, 0xFFFFFFFF, int(compress_size) & 0xFFFFFFFF):
            return False
    return True


def offsets_look_wrapped(members_cd_order, mpf):
    """Cheap detection: is this archive's central directory storing wrapped 32-bit offsets?

    Two signals, both nearly free. (1) Header offsets in central-directory order are monotonically
    increasing in any sequentially written zip; a backwards jump means the field wrapped. (2) The
    member with the largest true offset is the one most likely to be wrong, so its local header is
    probed directly. Under 4 GiB none of this can happen, so the check exits immediately.
    """
    if mpf.size <= ZIP32_WRAP or len(members_cd_order) == 0:
        return False, "archive is under 4 GiB, so a 32-bit offset cannot overflow"
    off = members_cd_order["header_offset"].to_numpy(dtype=np.int64)
    n_back = int((np.diff(off) < 0).sum()) if len(off) > 1 else 0
    if n_back:
        return True, (f"header offsets jump backwards {n_back:,} time(s) in central-directory "
                      f"order, which only happens when the 32-bit offset field wrapped")
    last = members_cd_order.iloc[-1]
    if int(last["header_offset"]) + 30 > mpf.size:
        return True, (f"the last member's recorded offset ({int(last['header_offset']):,}) is past "
                      f"the end of the archive ({mpf.size:,} bytes)")
    if not local_header_matches(mpf, int(last["header_offset"]), last["name"],
                                last.get("compress_size")):
        return True, (f"there is no local header for {last['name']!r} at its recorded offset "
                      f"{int(last['header_offset']):,}")
    return False, "the last member's local header is exactly where the central directory says"


def repair_zip32_offsets(members_cd_order, mpf, verify_samples=12, max_base_wraps=64):
    """Reconstruct true 64-bit header offsets. Returns (repaired int64 array, report dict).

    `members_cd_order` must be in CENTRAL-DIRECTORY order -- that is what makes the offsets
    monotonic and the wrap count recoverable. Sorting by offset first would destroy the signal.
    """
    raw = members_cd_order["header_offset"].to_numpy(dtype=np.int64) % ZIP32_WRAP
    n = len(raw)
    wraps = np.zeros(n, dtype=np.int64)
    if n > 1:
        wraps[1:] = np.cumsum((np.diff(raw) < 0).astype(np.int64))
    n_wraps = int(wraps[-1]) if n else 0

    # Pin the absolute base. Normally the first member sits near the start of the archive and the
    # base is 0, but an archive whose FIRST member is already past 4 GiB (a re-packaged tail) is
    # possible, so probe rather than assume. This is a handful of reads, not one per member.
    first = members_cd_order.iloc[0]
    base = None
    limit = min(max_base_wraps, int(mpf.size // ZIP32_WRAP)) + 1
    for j in range(limit):
        cand = int(raw[0]) + j * ZIP32_WRAP
        if cand + 30 > mpf.size:
            break
        if local_header_matches(mpf, cand, first["name"], first.get("compress_size")):
            base = j
            break
    if base is None:
        raise RuntimeError(
            f"the first member ({first['name']!r}) has no local file header at any offset of the "
            f"form {int(raw[0]):,} + j*4GiB, so the 32-bit offset repair cannot be anchored. The "
            f"parts are most likely out of order or one is truncated -- check that part_00000 is "
            f"the first chunk and that every part except the last is the same size.")

    repaired = (base + wraps) * ZIP32_WRAP + raw

    # Verify on a spread sample rather than everywhere. A wrong reconstruction shows up on the
    # first sample past a wrap boundary; a right one costs `verify_samples` reads instead of n.
    idx = sorted(set(int(round(x)) for x in np.linspace(0, n - 1, min(verify_samples, n))))
    ok = [i for i in idx
          if local_header_matches(mpf, int(repaired[i]), members_cd_order.iloc[i]["name"],
                                  members_cd_order.iloc[i].get("compress_size"))]
    report = {"n_members": n, "n_wraps": n_wraps, "base_wraps": base,
              "n_checked": len(idx), "n_verified": len(ok),
              "first_true": int(repaired[0]), "last_true": int(repaired[-1])}
    if not ok:
        raise RuntimeError(
            f"the 32-bit offset repair was anchored but none of {len(idx)} spread-sampled members "
            f"landed on its own local header afterwards. Applying it would make every read wrong "
            f"in a new way, so it is refused. This archive's central directory does not match its "
            f"contents: re-download it, or re-split the parts.")
    return repaired, report


# One read per member, sized to what is actually wanted.
#
# Two things about this were wrong in an earlier version and cost real hours on Drive, so they are
# spelled out.
#
# (1) THE ARCHIVE IS 100% DEFLATED, not stored. The comment "asking for 4 KB transfers exactly
#     4 KB" was only ever true for a STORED member. For a deflated one the compressed stream has
#     to be fed to zlib, and the old code fed it in fixed 64 KB chunks -- so producing the 4 KB of
#     header the index needs transferred 64 KB. Measured on a real 513 KB RSNA-sized slice: 16x
#     amplification, which over 752,803 members is 48 GB of Drive traffic instead of 3 GB.
#     The chunk now starts at roughly what the caller asked for and only grows if that was not
#     enough, so the common case is one chunk of ~4 KB and the pathological header still gets its
#     64 KB.
#
# (2) EVERY MEMBER COST TWO ROUND TRIPS. The central directory records where a member's LOCAL
#     HEADER is, not where its data starts, and the local header carries its own filename and
#     extra fields whose lengths vary -- so the data offset genuinely has to be read. But it does
#     not need a read of its own: one request covering the header, a reservation for the variable
#     part, and the first data chunk gets everything at once. On a FUSE-mounted Drive, where a
#     4 KB read and a 64 KB read cost nearly the same and it is the round trip that hurts, halving
#     the round trips is the larger of the two wins.
_LFH_RESERVE = 512          # bytes reserved for filename + extra fields in the combined read


def _open_member(mpf, header_offset, want):
    """(data_offset, first `want` bytes of member data) in ONE read where possible.

    Falls back to a second read only when the filename plus extra fields overflow the reservation,
    which for this dataset's ~70-character names does not happen.
    """
    want = max(0, int(want))
    mpf.seek(header_offset)
    buf = mpf.read(30 + _LFH_RESERVE + want)
    if len(buf) < 30:
        raise ValueError(f"short read at local header offset {header_offset}")
    sig, _v, _f, _m, _t, _d, _c, _cs, _us, fn_len, ex_len = _LFH.unpack(buf[:30])
    if sig != b"PK\x03\x04":
        raise ValueError(
            f"no local file header at offset {header_offset} (found {sig!r}). Either the parts are "
            f"out of order / truncated, or this member's offset overflowed the zip's 32-bit field "
            f"and the repair in cell 7 did not reconstruct it correctly.")
    start = 30 + fn_len + ex_len
    data_off = header_offset + start
    have = buf[start:start + want] if start < len(buf) else b""
    if len(have) < want:
        mpf.seek(data_off + len(have))
        have += mpf.read(want - len(have))
    return data_off, have


def read_member(mpf, header_offset, compress_size, compress_type, max_bytes=None, crc=None):
    """Up to `max_bytes` decompressed bytes of one member (all of it when max_bytes is None).

    `crc`, when given AND the whole member is being read, is checked against the CRC-32 the zip
    central directory recorded at creation time. This is the notebook's real integrity guarantee
    and it is worth understanding why: a spot check on a handful of members cannot detect
    corruption confined to one region of a 195 GB file, and corruption that is not detected does
    not raise -- it produces a volume of noise that the backbone happily embeds. Verifying the CRC
    of every member that is actually used turns that into a specific, logged, per-series failure.
    A partial read cannot be checked (there is nothing to compare a prefix against), which is
    another reason the header pass and the pixel pass are kept separate.
    """
    compress_size = int(compress_size)
    if compress_type == zipfile.ZIP_STORED:
        want = compress_size if max_bytes is None else min(int(max_bytes), compress_size)
        _, raw = _open_member(mpf, header_offset, want)
        if crc is not None and max_bytes is None:
            _verify_crc(raw, crc, header_offset)
        return raw
    if compress_type != zipfile.ZIP_DEFLATED:
        raise ValueError(f"unsupported zip compression method {compress_type} for this member")

    CH_MAX = 1 << 16
    ch = CH_MAX if max_bytes is None else max(4096, min(int(max_bytes), CH_MAX))
    data_off, chunk = _open_member(mpf, header_offset, min(ch, compress_size))
    pos = len(chunk)
    d = zlib.decompressobj(-15)
    out, got = [], 0
    while chunk:
        limit = 0 if max_bytes is None else max(0, int(max_bytes) - got)
        piece = d.decompress(chunk, limit)
        if piece:
            out.append(piece)
            got += len(piece)
        if max_bytes is not None and got >= max_bytes:
            break
        if d.eof:
            break
        # zlib hands back whatever it could not use under `limit`; feed that before reading more,
        # or those bytes are silently dropped and the stream decodes to garbage.
        chunk = d.unconsumed_tail
        if not chunk:
            if pos >= compress_size:
                break
            ch = min(ch * 2, CH_MAX)
            mpf.seek(data_off + pos)
            chunk = mpf.read(int(min(ch, compress_size - pos)))
            pos += len(chunk)
    blob = b"".join(out)
    if crc is not None and max_bytes is None:
        _verify_crc(blob, crc, header_offset)
    return blob


# ---- decoding a member out of a block that has already been read --------------------------------
# The random strategy above issues one request per member: ~4.6 KB each, 752,803 of them. That is
# the minimum possible BYTES and the maximum possible REQUESTS, and a Drive mount throttles on
# requests -- a real run went 311 members/s, then 15/s for the better part of an hour, then back
# to 153/s. Nothing was wrong with it except that it was asking too often.
#
# The opposite strategy: read the archive in large contiguous blocks and cut each member's header
# out of the block in memory. The members of one chunk span ~222 KB each, so a 16 MB block covers
# ~72 of them in ONE request. Bytes go up by the ratio of stride to header (about 48x); requests
# go down by the same factor. Which of the two wins is a property of the mount on the day, not
# something to decide in advance -- so cell 9 measures both and can switch mid-run.
def member_from_buffer(buf, base_offset, header_offset, compress_size, compress_type,
                       max_bytes=None):
    """Same bytes `read_member` would return, taken from an already-read block instead."""
    local = int(header_offset) - int(base_offset)
    if local < 0 or local + 30 > len(buf):
        raise ValueError(f"member at {header_offset} is not inside this block")
    sig, _v, _f, _m, _t, _d, _c, _cs, _us, fn_len, ex_len = _LFH.unpack(buf[local:local + 30])
    if sig != b"PK\x03\x04":
        raise ValueError(f"no local file header at offset {header_offset} (found {sig!r})")
    ds = local + 30 + fn_len + ex_len
    if ds > len(buf):
        raise ValueError(f"member data at {header_offset} starts past the end of this block")
    if compress_type == zipfile.ZIP_STORED:
        end = ds + (int(compress_size) if max_bytes is None
                    else min(int(max_bytes), int(compress_size)))
        return bytes(buf[ds:end])
    if compress_type != zipfile.ZIP_DEFLATED:
        raise ValueError(f"unsupported zip compression method {compress_type} for this member")
    raw = bytes(buf[ds:ds + int(compress_size)])
    if not raw:
        raise ValueError(f"no compressed bytes for the member at {header_offset} in this block")
    d = zlib.decompressobj(-15)
    out = d.decompress(raw, 0 if max_bytes is None else int(max_bytes))
    if max_bytes is None and not d.eof:
        raise ValueError(f"the member at {header_offset} is not fully inside this block")
    return out


PART_PATHS, PART_SIZES = discover_parts()
MPF = MultiPartFile(PART_PATHS)
ARCHIVE_FORMAT = detect_format(MPF)

print(f"{len(PART_PATHS)} part(s), {MPF.size/1e9:.2f} GB total")
print(f"  uniform part size : {PART_SIZES[0]/1e9:.3f} GB    last part: {PART_SIZES[-1]/1e9:.3f} GB")
print(f"  archive format    : {ARCHIVE_FORMAT}")
if ARCHIVE_FORMAT != "zip":
    raise RuntimeError(
        f"the concatenated parts start with a {ARCHIVE_FORMAT!r} signature, not a zip one. This "
        f"notebook reads a zip. If your archive really is a {ARCHIVE_FORMAT}, the parts may also "
        f"simply be in the wrong order -- check that part_00000 is the first chunk of the file.")
if MPF.size > (1 << 32):
    print(f"  ZIP64             : required ({MPF.size/1e9:.0f} GB > 4 GB); Python's zipfile "
          f"handles it transparently")

In [ ]:
# 7. The zip central directory -> a compact member table  [CACHED]
#
# The central directory lives at the END of the archive and lists every member with its offset and
# sizes. Reading it is one pass; at ~875k members it is also the single biggest memory event in
# the notebook, so the ZipInfo objects are converted to plain arrays and thrown away immediately.
# A ZipInfo is roughly 500 bytes and Python keeps one per member -- holding them for the session
# would cost several hundred MB for four numbers each.
#
# The result is cached to Drive keyed on the total archive size, so later sessions skip it.
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


def _cd_key():
    # Deliberately does NOT include ARCHIVE_PREFIX. Reading the central directory of a 195 GB
    # archive is a ~30 s pass over the end of the file; the prefix is just a filter applied after.
    # Keying the cache on the prefix would make every experiment with it pay that cost again.
    return {"total_size": int(MPF.size), "n_parts": len(PART_PATHS), "schema": 3}


def _atomic_write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def read_central_directory(mpf):
    """Every member, unfiltered. ZipInfo objects are converted to plain arrays and dropped: one is
    roughly 500 bytes and Python keeps one per member, so at ~874k members holding them for the
    session costs several hundred MB to carry four numbers each."""
    zf = zipfile.ZipFile(mpf)
    try:
        names, offs, csz, usz, ctype, crcs = [], [], [], [], [], []
        n_all = n_dir = 0
        for zi in zf.infolist():
            n_all += 1
            if zi.is_dir():
                n_dir += 1
                continue
            # infolist() preserves central-directory order, and that order is load-bearing: it is
            # the only thing that makes a wrapped 32-bit offset field recoverable (see below), and
            # the table gets sorted by offset a few lines later, which destroys it. So keep it.
            names.append(zi.filename)
            offs.append(zi.header_offset)
            csz.append(zi.compress_size)
            usz.append(zi.file_size)
            ctype.append(zi.compress_type)
            crcs.append(zi.CRC)
    finally:
        zf.close()
    return (pd.DataFrame({"name": names,
                          "cd_order": np.arange(len(names), dtype=np.int64),
                          "header_offset": np.asarray(offs, dtype=np.int64),
                          "compress_size": np.asarray(csz, dtype=np.int64),
                          "file_size": np.asarray(usz, dtype=np.int64),
                          "compress_type": np.asarray(ctype, dtype=np.int16),
                          "crc": np.asarray(crcs, dtype=np.int64)}),
            {"n_all": n_all, "n_dirs": n_dir})


_key = _cd_key()
MEMBERS_ALL = None
OFFSET_REPAIR = None          # None = not needed; a dict = what the repair did
if os.path.exists(CD_CACHE_PATH) and os.path.exists(CD_CACHE_META):
    try:
        with open(CD_CACHE_META) as f:
            _meta = json.load(f)
        if {k: _meta.get(k) for k in _key} == _key:
            MEMBERS_ALL = drive_retry(lambda: pd.read_parquet(CD_CACHE_PATH),
                                      what="reading the member-table cache")
            OFFSET_REPAIR = _meta.get("offset_repair")
            print(f"member table: cache HIT -- {len(MEMBERS_ALL):,} members from {CD_CACHE_PATH}")
            if OFFSET_REPAIR:
                print(f"  (these offsets were already repaired for a 32-bit overflow: "
                      f"{OFFSET_REPAIR['n_wraps']} wrap(s) across "
                      f"{OFFSET_REPAIR['n_members']:,} members)")
    except Exception as e:
        print(f"member table cache unreadable ({e}) -- rebuilding.")

if MEMBERS_ALL is None:
    print("reading the zip central directory (one pass over the end of the archive) ...")
    t0 = time.time()
    MEMBERS_ALL, _info = read_central_directory(MPF)
    print(f"  {_info['n_all']:,} entries ({_info['n_dirs']:,} directories) in {time.time()-t0:.1f}s")

    # ---- 32-bit offset overflow: detect, and repair before anything is cached ------------------
    # This has to happen here, once, on the FULL table in central-directory order -- not lazily at
    # read time, and not after the sort by offset. Left unrepaired it is not a subtle degradation:
    # every member past the 4 GiB mark reads from the wrong byte position, and what comes back is
    # either a "no local file header" error or, worse, a different member's bytes.
    # `cd_order` is already the row order here; sorting on it is free insurance against a future
    # edit that reorders the table before this point, which would silently break the wrap count.
    MEMBERS_ALL = MEMBERS_ALL.sort_values("cd_order", kind="mergesort").reset_index(drop=True)
    _wrapped, _why_wrap = offsets_look_wrapped(MEMBERS_ALL, MPF)
    print(f"  32-bit offset check: {'OVERFLOW DETECTED' if _wrapped else 'clean'} -- {_why_wrap}")
    if _wrapped:
        t1 = time.time()
        _fixed, OFFSET_REPAIR = repair_zip32_offsets(MEMBERS_ALL, MPF)
        MEMBERS_ALL["header_offset"] = _fixed
        print(f"  repaired {OFFSET_REPAIR['n_members']:,} offsets in {time.time()-t1:.1f}s: "
              f"{OFFSET_REPAIR['n_wraps']} wrap(s) of 4 GiB, base wrap "
              f"{OFFSET_REPAIR['base_wraps']}")
        print(f"    true offsets now span {OFFSET_REPAIR['first_true']:,} .. "
              f"{OFFSET_REPAIR['last_true']:,} in a {MPF.size:,}-byte archive")
        print(f"    verified on a spread sample: {OFFSET_REPAIR['n_verified']}/"
              f"{OFFSET_REPAIR['n_checked']} sampled members sit on their own local header")
        if OFFSET_REPAIR["n_verified"] < OFFSET_REPAIR["n_checked"]:
            print("    NOTE: not every sample verified. The reconstruction is applied anyway --")
            print("    probing all {:,} members individually would be that many Drive round trips"
                  .format(OFFSET_REPAIR["n_members"]))
            print("    with no bound on the time. Any member still wrong fails its own read with a")
            print("    stated reason and drops only its own series, which is a bounded cost.")

    drive_retry(lambda: MEMBERS_ALL.to_parquet(CD_CACHE_PATH + ".tmp", index=False),
                what="writing the member table", quiet=False)
    drive_retry(lambda: os.replace(CD_CACHE_PATH + ".tmp", CD_CACHE_PATH),
                what="committing the member table")
    _atomic_write_json(CD_CACHE_META, {**_key, "offset_repair": OFFSET_REPAIR})
    print(f"  cached to {CD_CACHE_PATH} -- this does not have to happen again")

# ---- what is actually in this archive ------------------------------------------------------------
# Reported by DIRECTORY, not just by top-level component. A real Kaggle download of this dataset
# puts everything one level down inside a competition folder, so a top-level-only report says
# "one prefix, 874,037 members" and tells you nothing about where the images are.
_dirs = (MEMBERS_ALL["name"].str.rsplit("/", n=1).str[0]
         .where(MEMBERS_ALL["name"].str.contains("/"), "(root)")
         .value_counts())
_dcm = (MEMBERS_ALL[MEMBERS_ALL["name"].str.lower().str.endswith(".dcm")]["name"]
        .str.rsplit("/", n=1).str[0].value_counts())
print()
print(f"directories inside the archive ({len(_dirs)}):")
for _d, _n in _dirs.head(10).items():
    _nd = int(_dcm.get(_d, 0))
    print(f"  {_d + '/':<62}{_n:>9,} members" + (f"   ({_nd:,} .dcm)" if _nd else "   (no .dcm)"))
if len(_dirs) > 10:
    print(f"  ... and {len(_dirs)-10} more")


def resolve_archive_prefix(requested, dcm_counts):
    """Work out which directory holds the labelled training images.

    The layout is not fixed: depending on how the archive was made, the images can sit at
    'stage_2_train/' or at '<competition-folder>/stage_2_train/'. Guessing wrong costs a 30-second
    central-directory read and a confusing empty result, so this looks rather than assumes.
    """
    if requested == "":
        # An explicit empty string means "no filter at all" -- keep that meaning rather than
        # letting a falsy value fall through to auto-detection, which would silently do something
        # different from what was asked.
        return "", "ARCHIVE_PREFIX is empty, so every member in the archive is included"
    if requested and requested != "auto":
        return requested, "set explicitly in the config"
    if len(dcm_counts) == 0:
        raise RuntimeError("no member in this archive has a .dcm extension, so there is nothing to "
                           "auto-detect. Set ARCHIVE_PREFIX explicitly from the listing above.")
    exact = [d for d in dcm_counts.index if d.rsplit("/", 1)[-1].lower() == "stage_2_train"]
    if exact:
        best = max(exact, key=lambda d: dcm_counts[d])
        return best + "/", "the directory named 'stage_2_train'"
    trainish = [d for d in dcm_counts.index if "train" in d.rsplit("/", 1)[-1].lower()]
    if trainish:
        best = max(trainish, key=lambda d: dcm_counts[d])
        return best + "/", "the only .dcm directory whose name mentions 'train'"
    best = str(dcm_counts.index[0])
    return best + "/", f"the directory holding the most .dcm files ({int(dcm_counts.iloc[0]):,})"


_requested = ARCHIVE_PREFIX
ARCHIVE_PREFIX, _why = resolve_archive_prefix(_requested, _dcm)
print()
print(f"ARCHIVE_PREFIX = {ARCHIVE_PREFIX!r}" + ("   (no filter)" if ARCHIVE_PREFIX == "" else ""))
print(f"  chosen because: {_why}")
if _requested == "auto":
    _others = [d for d in _dcm.index if d + "/" != ARCHIVE_PREFIX]
    if _others:
        print(f"  not chosen: " + ", ".join(f"{d}/ ({int(_dcm[d]):,} .dcm)" for d in _others[:4]))
        print(f"  (the test split carries no labels, which is why it is excluded rather than "
              f"merged in)")

MEMBERS = (MEMBERS_ALL if ARCHIVE_PREFIX == ""
           else MEMBERS_ALL[MEMBERS_ALL["name"].str.startswith(ARCHIVE_PREFIX)]).reset_index(drop=True)
if len(MEMBERS) == 0:
    raise RuntimeError(
        f"no member starts with ARCHIVE_PREFIX={ARCHIVE_PREFIX!r}. The directories actually "
        f"present are listed above -- set ARCHIVE_PREFIX to one of them, or to \"auto\" to let "
        f"the notebook pick.")
_non_dcm = int((~MEMBERS["name"].str.lower().str.endswith(".dcm")).sum())
if _non_dcm:
    print(f"  note: {_non_dcm:,} member(s) under this prefix are not .dcm files; they are kept in "
          f"the table and will be rejected by the header index as non-DICOM.")

# Offset order = forward-only access across the archive. A FUSE-mounted Drive is far happier with
# a monotonically advancing read pattern than with 750k backward seeks, and the earlier attempt at
# this dataset lost 28.9% of its series to exactly that failure mode.
MEMBERS = MEMBERS.sort_values("header_offset", kind="mergesort").reset_index(drop=True)

_stored = int((MEMBERS["compress_type"] == 0).sum())
_dup_names = int(MEMBERS["name"].duplicated().sum())
print()
print(f"members under {ARCHIVE_PREFIX!r}: {len(MEMBERS):,}")
if _dup_names:
    print(f"  DUPLICATE NAMES    : {_dup_names:,} member(s) share a filename with another. A zip "
          f"can hold these;")
    print(f"                       the header index deduplicates on the member name and keeps the "
          f"last, so no")
    print(f"                       slice is counted twice -- but it is worth knowing the archive "
          f"has them.")
print(f"  compression        : {_stored:,} stored, {len(MEMBERS)-_stored:,} deflated")
print(f"  uncompressed size  : median {MEMBERS['file_size'].median()/1e3:.0f} KB, "
      f"total {MEMBERS['file_size'].sum()/1e9:.1f} GB")
print(f"  offsets            : {MEMBERS['header_offset'].iloc[0]:,} .. "
      f"{MEMBERS['header_offset'].iloc[-1]:,}")

# ---- prove the direct reader agrees with zipfile, on real members of THIS archive --------------
# The whole notebook depends on read_member() returning exactly what ZipFile.read() would. That is
# a claim about your archive, not a general one, so it is tested here rather than assumed.
#
# This is a SPOT CHECK and it is not the integrity guarantee: a dozen samples cannot detect damage
# confined to one region of a 193 GB file. The guarantee is the per-member CRC-32 recorded in the
# central directory, which read_member() verifies on every full read -- so any member that is
# actually used to build a volume is checked, and a damaged one becomes a specific logged failure
# for that series rather than a volume of noise.
def _reader_self_test(n=12, seed=0, cross_check_zipfile=True):
    """A CRC failure here is not a bug in the reader -- it means those bytes on Drive are not the
    bytes that were archived. It is caught and counted rather than allowed to escape, because the
    useful output is a diagnosis of the archive, not a traceback about one member.

    `cross_check_zipfile` is switched OFF when the offsets had to be repaired. That is not ducking
    the comparison: on such an archive ZipFile reads from the un-repaired offsets and fails on
    every member, so it is not a reference for anything. The CRC-32 from the central directory is
    the stronger check in that case anyway -- it says the bytes are the archived bytes, which is
    exactly what agreement with ZipFile was standing in for."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(MEMBERS), size=min(n, len(MEMBERS)), replace=False)
    zf = zipfile.ZipFile(MultiPartFile(PART_PATHS)) if cross_check_zipfile else None
    try:
        mism, partial_ok, crc_bad, other = 0, 0, 0, []
        for i in idx:
            r = MEMBERS.iloc[int(i)]
            try:
                full = read_member(MPF, r.header_offset, r.compress_size, r.compress_type,
                                   crc=r.crc)
            except ValueError as e:
                if "CRC mismatch" in str(e):
                    crc_bad += 1
                else:
                    other.append(str(e)[:120])
                continue
            if zf is not None:
                try:
                    ref = zf.read(r["name"])
                except Exception as e:
                    other.append(f"ZipFile could not read {r['name']}: {type(e).__name__}")
                    continue
                if full != ref:
                    mism += 1
            else:
                ref = full            # CRC-verified above, so it is a sound reference for a prefix
            head = read_member(MPF, r.header_offset, r.compress_size, r.compress_type,
                               max_bytes=HEADER_BYTES)
            if head == ref[:len(head)] and len(head) <= HEADER_BYTES:
                partial_ok += 1
        return len(idx), mism, partial_ok, crc_bad, other
    finally:
        if zf is not None:
            zf.close()


_n, _mism, _pok, _crc_bad, _other = _reader_self_test(cross_check_zipfile=not OFFSET_REPAIR)
_good = max(_n - _crc_bad, 1)
print()
print(f"reader self-test on {_n} real members of this archive:")
print(f"  CRC-32 verified                                : {_n - _crc_bad}/{_n} passed")
if OFFSET_REPAIR:
    print(f"  ZipFile cross-check                            : skipped (offsets were repaired; "
          f"ZipFile would read from the broken ones)")
else:
    print(f"  full-member reads identical to ZipFile.read()  : {_good - _mism}/{_good}")
print(f"  partial reads are a true prefix, <= {HEADER_BYTES} B   : {_pok}/{_good}")
for _o in _other[:3]:
    print(f"  other: {_o}")
if _mism:
    raise RuntimeError(
        f"{_mism} of {_n} members read differently through the direct reader than through "
        f"ZipFile, with matching CRCs. That is a reader bug, not damaged data -- do not continue, "
        f"because every volume built from this archive would be suspect.")
if _crc_bad:
    print()
    print(f"*** {_crc_bad} of {_n} sampled members FAILED their CRC-32 check. ***")
    print("    Those bytes on Drive are not the bytes that were archived. At this sampling rate")
    print(f"    that suggests roughly {_crc_bad/_n:.0%} of the archive is affected.")
    print("    What happens next: every member is CRC-checked when it is actually read, so a")
    print("    damaged slice is dropped and its series fails individually with a stated reason --")
    print("    nothing corrupt reaches the feature store. But if the affected fraction is large,")
    print("    re-upload the parts rather than extracting around the damage.")
    print("    Likeliest causes, in order: an upload that did not finish; a part that was")
    print("    re-downloaded and truncated; parts concatenated in the wrong order.")

In [ ]:
# ================================================================================================
# 7b. STAGE THE ARCHIVE ON LOCAL DISK
# ================================================================================================
# Set STAGE_LOCALLY in cell 4 to turn this off. Cells 8-14 then read from local disk.
#
# WHY. The trial measured 35.2 s per series, of which 33.5 s (95%) was READ: 18 MB of DICOM pulled
# through a FUSE mount in 33 s = 0.5 MB/s. Drive is not slow at bandwidth, it is slow at ROUND
# TRIPS, and one series is ~35 separate ranged reads scattered across ~35 different 2 GB parts.
# From local disk those reads are sub-millisecond, the read bucket collapses, and what is left is
# preprocess + model + write.
#
# WHY THE CACHES SURVIVE. The member table is keyed on {total_size, n_parts, schema} and the header
# index on {n_members, archive_size, ...}. Neither key contains a PATH, so the staged directory
# hits both caches and none of the 752,803-member indexing is repeated. That is also why the
# unused parts must still be PRESENT at their exact size instead of omitted -- omitting them
# changes n_parts and total_size, discards both caches, and shifts every offset in the archive.
#
# WHY THE HOLES ARE SAFE. The train split starts ~27 GB in; the parts before it hold stage_2_test,
# which this notebook never reads. They are recreated as sparse files of the exact right size, so
# the offset arithmetic is identical and they cost no disk. A read that landed in one would not
# silently return zeros: _open_member requires a PK\x03\x04 signature and read_member verifies the
# CRC-32 of every full member, so a hole raises immediately. Part 0 is always staged for real,
# because detect_format reads byte 0; so is the last part, which holds the central directory.
#
# NOTHING HERE IS IN THE PREPROCESSING SIGNATURE. The 25 series already extracted stay valid and
# are skipped on resume. Staging changes where bytes come from, not what is computed from them.
import concurrent.futures as _cf
import os
import shutil
import time

STAGE_SLACK_BYTES  = 1 << 20     # safety margin past the last needed byte
STAGE_FREE_MARGIN  = int(STAGE_FREE_MARGIN_GB) << 30


def _fmt(n):
    return f"{n / 1e9:.1f} GB"


def parts_actually_needed(members, part_sizes, slack=STAGE_SLACK_BYTES):
    """Indices of the parts the members' bytes land in, plus part 0 and the last part."""
    lo = int(members["header_offset"].min())
    hi = int((members["header_offset"] + members["compress_size"]).max()) + slack
    starts, acc = [], 0
    for s in part_sizes:
        starts.append(acc)
        acc += int(s)
    hi = min(hi, acc)
    need = {0, len(part_sizes) - 1}
    for i, s in enumerate(part_sizes):
        a, b = starts[i], starts[i] + int(s)
        if a < hi and b > lo:
            need.add(i)
    return sorted(need), lo, hi


def _is_sparse(path):
    st = os.stat(path)
    return (st.st_blocks * 512) < (st.st_size // 8)


def _stage_one(src, dst, size):
    if os.path.exists(dst) and os.path.getsize(dst) == size:
        return ("skip", dst)
    tmp = dst + ".partial"
    with open(src, "rb", buffering=0) as fi, open(tmp, "wb", buffering=0) as fo:
        shutil.copyfileobj(fi, fo, length=8 << 20)
    got = os.path.getsize(tmp)
    if got != size:
        os.remove(tmp)
        raise IOError(f"{os.path.basename(src)}: copied {got:,} bytes, expected {size:,}")
    os.replace(tmp, dst)
    return ("copy", dst)


def stage_archive_locally(part_paths, part_sizes, members, local_dir=LOCAL_PARTS_DIR,
                          workers=STAGE_COPY_WORKERS, dry_run=False):
    os.makedirs(local_dir, exist_ok=True)
    need, lo, hi = parts_actually_needed(members, part_sizes)
    need_set = set(need)
    holes = [i for i in range(len(part_sizes)) if i not in need_set]
    bytes_needed = sum(int(part_sizes[i]) for i in need)
    already = sum(os.path.getsize(os.path.join(local_dir, os.path.basename(part_paths[i])))
                  for i in need
                  if os.path.exists(os.path.join(local_dir, os.path.basename(part_paths[i]))))
    to_fetch = bytes_needed - already
    free = shutil.disk_usage(local_dir).free

    print("=" * 94)
    print("STAGE ARCHIVE ON LOCAL DISK")
    print("=" * 94)
    print(f"  train bytes span      : {lo:,} .. {hi:,}")
    print(f"  parts to stage        : {len(need)} of {len(part_sizes)}   ({_fmt(bytes_needed)})")
    print(f"  parts left as holes   : {len(holes)}   "
          f"({_fmt(sum(int(part_sizes[i]) for i in holes))} of stage_2_test, never read)")
    print(f"  already staged        : {_fmt(already)}")
    print(f"  to fetch              : {_fmt(to_fetch)}")
    print(f"  local free            : {_fmt(free)}   -> {_fmt(free - to_fetch)} left after")
    if free - to_fetch < STAGE_FREE_MARGIN:
        raise RuntimeError(
            f"only {_fmt(free)} free locally and this needs {_fmt(to_fetch)}, leaving under "
            f"{_fmt(STAGE_FREE_MARGIN)} of headroom. Colab keeps writing to this disk during a "
            f"run and filling it mid-extraction fails in confusing ways. Either take a runtime "
            f"with a bigger disk, or skip staging and raise EXTRACT_READ_WORKERS instead.")
    if dry_run:
        print("  dry run -- nothing copied.")
        return None, None, {"need": need, "holes": holes}

    for i in holes:                       # cheap, and it proves the filesystem does sparse files
        dst = os.path.join(local_dir, os.path.basename(part_paths[i]))
        if os.path.exists(dst) and os.path.getsize(dst) == int(part_sizes[i]):
            continue
        with open(dst, "wb") as f:
            f.truncate(int(part_sizes[i]))
        if not _is_sparse(dst):
            os.remove(dst)
            raise RuntimeError(
                f"{os.path.basename(dst)}: this filesystem allocated real blocks for a truncated "
                f"file, so the unused parts cannot be free. Staging everything would need "
                f"{_fmt(sum(int(s) for s in part_sizes))}.")

    t0, done, copied = time.time(), 0, 0
    jobs = [(part_paths[i], os.path.join(local_dir, os.path.basename(part_paths[i])),
             int(part_sizes[i])) for i in need]
    with _cf.ThreadPoolExecutor(max_workers=workers) as ex:
        for fut in _cf.as_completed({ex.submit(_stage_one, *j): j for j in jobs}):
            what, dst = fut.result()
            done += 1
            if what == "copy":
                copied += os.path.getsize(dst)
            el = max(time.time() - t0, 1e-9)
            eta = (len(jobs) - done) * (el / done) / 60 if done else 0
            print(f"\r  staged {done}/{len(jobs)}   {_fmt(copied)} in {el/60:.1f} min "
                  f"({copied/el/1e6:.0f} MB/s)   ETA {eta:.0f} min      ", end="", flush=True)
    print()

    new_paths, new_sizes = [], []
    for i, p in enumerate(part_paths):
        dst = os.path.join(local_dir, os.path.basename(p))
        if not os.path.exists(dst):
            raise RuntimeError(f"{dst} is missing after staging")
        got = os.path.getsize(dst)
        if got != int(part_sizes[i]):
            raise RuntimeError(f"{dst} is {got:,} bytes, the source part is {int(part_sizes[i]):,}")
        new_paths.append(dst)
        new_sizes.append(got)
    print(f"  all {len(new_paths)} parts present at their true sizes; "
          f"apparent total {_fmt(sum(new_sizes))}, actually on disk "
          f"{_fmt(sum(os.stat(p).st_blocks * 512 for p in new_paths))}")
    return new_paths, new_sizes, {"need": need, "holes": holes, "seconds": time.time() - t0}


if not STAGE_LOCALLY:
    print('STAGE_LOCALLY is False -- reading straight from Drive (~39 h per backbone).')
else:
    # ---- do it, then repoint the reader ------------------------------------------------------------
    _new_paths, _new_sizes, _info = stage_archive_locally(PART_PATHS, PART_SIZES, MEMBERS)

    PARTS_DIR              = LOCAL_PARTS_DIR
    PART_PATHS, PART_SIZES = _new_paths, _new_sizes
    MPF                    = MultiPartFile(PART_PATHS, sizes=PART_SIZES)
    _READERS               = threading.local()      # drop any thread-local reader still on Drive

    assert detect_format(MPF) == "zip", "the staged archive does not start with a zip signature"
    assert len(PART_PATHS) == len(_new_paths) and MPF.size == sum(PART_SIZES)
    print(f"\nreader repointed at {PARTS_DIR}: {len(PART_PATHS)} parts, {MPF.size/1e9:.2f} GB")
    print("the member-table and header-index caches are keyed on size and count, not path,")
    print("so cells 8 onward still hit them -- no re-indexing. Carry on to cell 8.")


---

## Part 2 — labels, the header index, and the series table

The header index is the one genuinely expensive step, and it is worth being clear about why it
cannot be avoided. RSNA-ICH is a flat directory of `ID_<hex>.dcm`. Which slices belong together
exists **only** inside the DICOM headers, so there is no way to select "N complete series" without
first reading every candidate header — a partial index gives a partial, scattered fraction of every
series and no complete volume at all.

So it is done once, properly, and cached:

* **~4 KB per member, not ~500 KB.** Only the tags up to `ImagePositionPatient` are needed. A
  member whose header does not parse within that budget is retried once at 64 KB before being
  called non-DICOM — otherwise a scanner that writes a fat header would silently lose its whole
  cohort. Every member of this archive is DEFLATE-compressed, so "4 KB" means feeding zlib about
  4 KB of compressed input and stopping — see Part 1 for why that distinction was worth 14× the
  Drive traffic.
* **Ascending archive-offset order**, so the access pattern advances forward through the parts
  rather than seeking backwards ~750k times.
* **Resumable in chunks of `INDEX_CHUNK`**, each written to Drive as it completes. A dropped Colab
  session costs one chunk.
* **Threaded**, each worker with its own reader — a `MultiPartFile` carries a file position and
  cannot be shared.

After the first chunk the notebook measures your actual throughput and prints an ETA for the rest,
so you find out what this will cost within a couple of minutes rather than after an hour.

* **Strategy and thread count measured, not guessed.** There are two ways to read 752,803
  headers off a Drive mount and they are opposites:

  | | requests | bytes |
  |---|---|---|
  | `"random"` — one ~4.6 KB read per member | **752,803** | 3.5 GB |
  | `"bulk"` — 16 MB contiguous blocks, headers cut out in memory | **~10,000** | 167 GB |

  Members of this archive sit ~222 KB apart, so nothing in between is useful — which means you
  can minimise requests or minimise bytes, never both. Which one wins is a property of your mount
  on the day. `INDEX_STRATEGY = "auto"` measures both on ~200 members each, prints the rate and
  ETA for every candidate, and picks. Either strategy produces identical rows — cell 6 checks
  `member_from_buffer` against `read_member` on real members of your archive — so the choice is
  purely about speed.

* **And re-measured if the rate collapses.** Reported per chunk as an *instantaneous* rate, not a
  running average. When it falls below 35% of the best seen for three chunks in a row, the cell
  says so, re-measures both strategies, and switches if that helps. This is not defensive
  programming for its own sake: see below.

**What this used to cost, and what actually goes wrong.** Two real runs of this notebook on the
full archive, both of which failed to finish, are the reason the cell looks like this.

*Run one* managed **113 members/s at 8 threads**. Two bugs, both in cell 6: every header read
pulled **64 KB** instead of 4 KB (the archive is entirely DEFLATE and the reader chunked the
compressed stream at a fixed 64 KB — 49 GB of traffic to collect 3 GB of headers), and every
member took **two round trips**, one for the local file header and one for the data. Fixing both
took it to **4,638 bytes and 1.00 read per member**, still byte-identical to `ZipFile.read()`.

*Run two* opened at **311 members/s** — the fix worked — and then this happened:

```
 chunks   1..10      267/s
 chunks  10..20      248/s
 chunks  20..30       61/s
 chunks  30..40       15/s     <-- 56 minutes for 50,000 members
 chunks  40..50       50/s
 chunks  50..60      153/s     <-- released again
```

That is not decay, it is **Google Drive throttling on request rate**, and it cannot be argued
with — only avoided. It is also why the rate is now reported per chunk and instantaneously: run
two printed a *cumulative average* of 311, 271, 259, 124, 44, which reads like a gentle decline
and was in fact a 20× collapse. Averages do not report weather.

The `"bulk"` strategy exists for exactly this. Under a simulated per-request throttle it recovered
**5.9×** (252 → 1,475 members/s) by asking 96× less often for the same result.

It is resumable, so a dropped session costs one chunk of `INDEX_CHUNK` members, not the run — and
the earlier run's 128 completed chunks were still there and still valid when it restarted. If you
want to confirm the pipeline works before paying for the full pass, set `SMOKE_TEST = True` in the
config cell: it caps the index, the series count and the epochs so the whole notebook finishes in
about fifteen minutes, and it says so at every step so you cannot mistake a smoke run for a real
one.

In [ ]:
# 8. Per-slice labels from stage_2_train.csv  [CACHED]
#
# The file is long-format -- one row per (slice, subtype), ~4.5M rows and 120 MB -- and is read in
# chunks and pivoted to one row per slice with six 0/1 columns.
#
# THE JOIN KEY IS THE FILENAME. IDs look like `ID_000039fa0_epidural`, and `ID_000039fa0` is the
# member's filename, which in this dataset is a DIFFERENT string from the DICOM's own
# SOPInstanceUID. Joining on SOPInstanceUID produces an all-zero label column and raises nothing,
# so the index records both and the merge below is checked for coverage rather than trusted.
import os
import time

import numpy as np
import pandas as pd


def member_to_slice_id(member):
    """'stage_2_train/ID_000039fa0.dcm' -> 'ID_000039fa0'"""
    base = member.rsplit("/", 1)[-1]
    return base[:-4] if base.lower().endswith(".dcm") else base


def load_slice_labels(csv_path, chunksize=1_000_000):
    parts = []
    # A 120 MB CSV off Drive is exactly the kind of read that returns EIO once and then works.
    for chunk in drive_retry(lambda: pd.read_csv(csv_path, chunksize=chunksize),
                             what="opening the labels CSV", quiet=False):
        cols = {c.lower(): c for c in chunk.columns}
        idc, lblc = cols.get("id"), cols.get("label")
        if idc is None or lblc is None:
            raise ValueError(f"expected 'ID' and 'Label' columns, found {list(chunk.columns)}")
        s = chunk[idc].astype(str).str.rsplit("_", n=1, expand=True)
        parts.append(pd.DataFrame({
            "slice_id": s[0],
            "subtype": s[1],
            "v": pd.to_numeric(chunk[lblc], errors="coerce").fillna(0).clip(0, 1).astype("int8"),
        }))
    long = pd.concat(parts, ignore_index=True)
    unknown = sorted(set(long["subtype"].dropna().unique()) - set(RSNA_SUBTYPES))
    if unknown:
        raise ValueError(
            f"unexpected subtype token(s) in the labels CSV: {unknown[:8]}. The ID format is "
            f"expected to be ID_<hex>_<subtype> with subtype in {RSNA_SUBTYPES}.")
    wide = (long.pivot_table(index="slice_id", columns="subtype", values="v",
                             aggfunc="max", fill_value=0)
                .reindex(columns=RSNA_SUBTYPES, fill_value=0)
                .astype("int8").reset_index())
    wide.columns.name = None
    return wide


if os.path.exists(LABELS_CACHE_PATH):
    SLICE_LABELS = drive_retry(lambda: pd.read_parquet(LABELS_CACHE_PATH),
                               what="reading the labels cache")
    print(f"slice labels: cache HIT -- {len(SLICE_LABELS):,} slices from {LABELS_CACHE_PATH}")
else:
    print(f"reading {LABELS_CSV} ({os.path.getsize(LABELS_CSV)/1e6:.0f} MB) ...")
    t0 = time.time()
    SLICE_LABELS = load_slice_labels(LABELS_CSV)
    drive_retry(lambda: SLICE_LABELS.to_parquet(LABELS_CACHE_PATH + ".tmp", index=False),
                what="writing the labels cache", quiet=False)
    drive_retry(lambda: os.replace(LABELS_CACHE_PATH + ".tmp", LABELS_CACHE_PATH),
                what="committing the labels cache")
    print(f"  pivoted to {len(SLICE_LABELS):,} slices x {len(RSNA_SUBTYPES)} subtypes "
          f"in {time.time()-t0:.0f}s -> {LABELS_CACHE_PATH}")

print()
print(f"{'subtype':<20}{'positive slices':>16}{'prevalence':>13}")
for c in RSNA_SUBTYPES:
    print(f"  {c:<18}{int(SLICE_LABELS[c].sum()):>16,}{SLICE_LABELS[c].mean():>13.4f}")

# `any` should be the union of the five compartments -- but not exactly: an annotator can mark
# hemorrhage present without committing to a compartment. Quantify it rather than assume it.
_five = [c for c in RSNA_SUBTYPES if c != "any"]
_union = (SLICE_LABELS[_five].to_numpy() > 0).any(axis=1)
_anyc = SLICE_LABELS["any"].to_numpy() > 0
print()
print(f"'any' vs OR(five compartments): differ on {int((_union != _anyc).sum()):,} slice(s) "
      f"({(_union != _anyc).mean():.3%})")
print(f"  any=1 with no compartment marked : {int((_anyc & ~_union).sum()):,}")
print(f"  a compartment marked with any=0  : {int((_union & ~_anyc).sum()):,}")
print("  'any' is used as the labelled ground truth for hemorrhage presence, not re-derived.")

_ids = set(SLICE_LABELS["slice_id"])
_sample = [member_to_slice_id(m) for m in MEMBERS["name"].head(2000)]
_hit = sum(1 for s in _sample if s in _ids)
print()
print(f"filename -> label-id join, checked on {len(_sample):,} members: {_hit/len(_sample):.1%} matched")
if _hit / max(len(_sample), 1) < 0.9:
    raise RuntimeError(
        f"only {_hit/len(_sample):.1%} of member filenames appear in the labels CSV. Either "
        f"ARCHIVE_PREFIX points at an unlabelled split (stage_2_test has no labels), or the "
        f"filename convention differs from ID_<hex>.dcm. Fix this before indexing 750k headers.")

In [ ]:
# 9. The header index: slice -> (patient, study, series, z, spacing)  [RESUMABLE, CACHED]
#
# This is the expensive pass and it happens once, ever. Each member's first HEADER_BYTES are read
# and parsed; nothing touches pixel data. Chunks are written to Drive as they finish, so a dropped
# session costs one chunk and re-running this cell continues where it stopped.
import concurrent.futures as _cf
import contextlib
import glob
import io
import json
import os
import time

import numpy as np
import pandas as pd
import pydicom
from tqdm.auto import tqdm

INDEX_FIELDS = ["member", "sop", "patient", "study", "series", "z", "sx", "sy", "sz_tag",
                "rows", "cols", "iop", "bits_stored", "px_repr", "intercept", "slope", "err"]


def safe_float_attr(ds, name, default=None):
    """getattr(ds, name, default) returns the default only when the tag is ABSENT -- not when it
    is present but blank, which is a common export quirk and would otherwise crash on float('')."""
    try:
        v = getattr(ds, name, default)
        if v is None or v == "":
            return default
        return float(v)
    except Exception:
        return default


def parse_header(blob, member):
    rec = {k: None for k in INDEX_FIELDS}
    rec["member"] = member
    try:
        ds = pydicom.dcmread(io.BytesIO(blob), stop_before_pixels=True, force=True)
    except Exception as e:
        rec["err"] = f"{type(e).__name__}: {e}"
        return rec
    series = getattr(ds, "SeriesInstanceUID", None)
    if not series:
        # force=True happily parses a text file into an empty Dataset, so "it parsed" proves
        # nothing at all. A missing SeriesInstanceUID is what actually identifies a non-DICOM
        # member -- this is the check that catches a stray README or a truncated file.
        rec["err"] = "no SeriesInstanceUID (not a DICOM member)"
        return rec
    rec["series"] = str(series)
    rec["study"] = str(getattr(ds, "StudyInstanceUID", "") or "")
    rec["patient"] = str(getattr(ds, "PatientID", "") or "")
    rec["sop"] = str(getattr(ds, "SOPInstanceUID", "") or "")
    iop = getattr(ds, "ImageOrientationPatient", None)
    if iop is not None and len(iop) == 6:
        try:
            rec["iop"] = ",".join(f"{float(t):.6f}" for t in iop)
        except Exception:
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    if ipp is not None and len(ipp) == 3:
        try:
            p = np.array([float(ipp[0]), float(ipp[1]), float(ipp[2])])
            n = None
            if rec["iop"]:
                v = [float(t) for t in rec["iop"].split(",")]
                n = np.cross(v[:3], v[3:6])
                nn = float(np.linalg.norm(n))
                n = (n / nn) if np.isfinite(nn) and nn > 1e-6 else None
            # The stacking coordinate is the position projected onto the slice NORMAL, not raw z.
            # They agree for an ordinary axial study and they do not for a tilted gantry -- which
            # is exactly the case where raw z stops being monotonic and sorting on it interleaves
            # the stack.
            rec["z"] = float(p @ n) if n is not None else float(p[2])
        except Exception:
            pass
    px = getattr(ds, "PixelSpacing", None)
    if px is not None and len(px) == 2:
        try:
            rec["sx"], rec["sy"] = float(px[1]), float(px[0])   # DICOM order is [row(y), col(x)]
        except Exception:
            pass
    rec["sz_tag"] = (safe_float_attr(ds, "SpacingBetweenSlices")
                     or safe_float_attr(ds, "SliceThickness"))
    rec["rows"] = int(getattr(ds, "Rows", 0) or 0)
    rec["cols"] = int(getattr(ds, "Columns", 0) or 0)
    rec["bits_stored"] = int(getattr(ds, "BitsStored", 0) or 0)
    rec["px_repr"] = int(getattr(ds, "PixelRepresentation", 0) or 0)
    rec["intercept"] = safe_float_attr(ds, "RescaleIntercept", 0.0)
    rec["slope"] = safe_float_attr(ds, "RescaleSlope", 1.0)
    return rec


def _err_rec(name, e):
    rec = {k: None for k in INDEX_FIELDS}
    rec["member"] = name
    rec["err"] = f"{type(e).__name__}: {e}"
    return rec


def _is_transient_err(err):
    return bool(err) and any(t in str(err).lower() for t in
                             ("input/output error", "errno 5", "timeouterror", "stale file handle",
                              "transport endpoint", "connection reset", "temporarily unavailable"))


def _retry_transient_records(rows, recs):
    """Re-read only the members whose error looks like Drive weather, one at a time, with backoff.

    Serially and unthreaded on purpose: whatever the mount is unhappy about, asking it harder is
    not the answer. A handful of members is cheap to redo even slowly.
    """
    by_name = {r[0]: r for r in rows}
    for rnd in range(int(INDEX_IO_RETRY_ROUNDS)):
        bad = [i for i, r in enumerate(recs) if _is_transient_err(r.get("err"))]
        if not bad:
            return recs
        time.sleep(DRIVE_RETRY_BASE_S * (2 ** rnd))
        mpf = thread_reader()
        for i in bad:
            name = recs[i]["member"]
            row = by_name.get(name)
            if row is None:
                continue
            _n, off, csz, ct = row
            try:
                blob = read_member(mpf, off, csz, ct, max_bytes=HEADER_BYTES)
                rec = parse_header(blob, name)
                if rec["series"] is None and int(csz) > HEADER_BYTES:
                    blob = read_member(mpf, off, csz, ct, max_bytes=HEADER_BYTES_ESCALATE)
                    rec = parse_header(blob, name)
                recs[i] = rec
            except Exception as e:
                recs[i] = _err_rec(name, e)
    return recs


def _index_block(block):
    """RANDOM strategy: one request per member, ~4.6 KB each.

    The reader is thread-local and lives as long as the thread, rather than being built per call.
    Building one per call re-stat'd all 91 parts every time -- 220,000 metadata round trips over
    the full index, for numbers that were already known.
    """
    mpf = thread_reader()
    out = []
    for name, off, csz, ct in block:
        try:
            blob = read_member(mpf, off, csz, ct, max_bytes=HEADER_BYTES)
            rec = parse_header(blob, name)
            if rec["series"] is None and csz > HEADER_BYTES:
                # A header longer than the budget looks exactly like a non-DICOM member.
                # Retry once, larger, before believing it.
                blob = read_member(mpf, off, csz, ct, max_bytes=HEADER_BYTES_ESCALATE)
                rec = parse_header(blob, name)
        except Exception as e:
            rec = _err_rec(name, e)
        out.append(rec)
    return out


def _index_block_bulk(block, block_bytes=None):
    """BULK strategy: read the archive in large contiguous blocks, cut the headers out in memory.

    Same rows as `_index_block`, and cell 6's `member_from_buffer` is checked against
    `read_member` on real members of this archive, so the two strategies are interchangeable and
    a mid-run switch cannot change the index. What changes is the shape of the load on Drive:
    ~72 members per request instead of one, at the cost of transferring the gaps between them.
    """
    bb = int(block_bytes or INDEX_BULK_BLOCK_BYTES)
    need = 30 + 512 + int(HEADER_BYTES)
    mpf = thread_reader()
    rows = sorted(block, key=lambda r: int(r[1]))
    out = []
    i, n, total = 0, len(rows), MPF.size
    while i < n:
        start = int(rows[i][1])
        j = i
        while j < n and int(rows[j][1]) + need <= start + bb:
            j += 1
        if j == i:
            j = i + 1                                   # always make progress
        end = min(int(rows[j - 1][1]) + need, total)
        mpf.seek(start)
        buf = mpf.read(end - start)
        for k in range(i, j):
            name, off, csz, ct = rows[k]
            try:
                blob = member_from_buffer(buf, start, int(off), int(csz), int(ct),
                                          max_bytes=HEADER_BYTES)
                rec = parse_header(blob, name)
                if rec["series"] is None and int(csz) > HEADER_BYTES:
                    blob = read_member(mpf, off, csz, ct, max_bytes=HEADER_BYTES_ESCALATE)
                    rec = parse_header(blob, name)
            except Exception as e:
                rec = _err_rec(name, e)
            out.append(rec)
        i = j
    return out


# One executor for the whole pass, not one per chunk. The thread-local readers above only pay off
# if the threads outlive a single chunk.
_INDEX_EXEC = {"ex": None, "w": None}


def _index_executor(w):
    if _INDEX_EXEC["ex"] is None or _INDEX_EXEC["w"] != w:
        if _INDEX_EXEC["ex"] is not None:
            _INDEX_EXEC["ex"].shutdown(wait=True)
        _INDEX_EXEC["ex"] = _cf.ThreadPoolExecutor(max_workers=max(1, int(w)))
        _INDEX_EXEC["w"] = int(w)
    return _INDEX_EXEC["ex"]


def _run_blocks(rows, strategy, workers):
    """Split `rows` across `workers`, keeping each worker's slice contiguous in offset order."""
    fn = _index_block_bulk if strategy == "bulk" else _index_block
    w = max(1, int(workers))
    per = (len(rows) + w - 1) // w
    blocks = [rows[i:i + per] for i in range(0, len(rows), per)]
    if w == 1 or len(blocks) == 1:
        recs = []
        for b in blocks:
            recs.extend(fn(b))
        return recs
    recs = []
    for r in _index_executor(w).map(fn, blocks):
        recs.extend(r)
    return recs


def _calibrate(members, start_at=None, n=None, quiet=False):
    """Measure both read strategies on THIS mount and return the fastest.

    Worth the minute it costs, because the answer is not stable and not guessable. On local disk
    four threads of random reads win. On a Drive mount the same code measured 311 members/s, then
    15/s under throttling, then 153/s -- and a bulk pass, which asks ~70x less often, is barely
    affected by the thing that caused that. So this is re-run mid-pass when the rate collapses.

    Candidates are measured on DIFFERENT members so that no candidate is scored against a region
    the previous one just pulled into the mount's cache.
    """
    cands = [("random", w) for w in (4, 8, 16, INDEX_WORKERS_MAX) if 1 <= w <= INDEX_WORKERS_MAX]
    cands += [("bulk", w) for w in (2, 4, 8) if w <= INDEX_WORKERS_MAX]
    cands = sorted(set(cands))
    n = int(n or INDEX_CALIBRATE_N)
    if len(members) < n * (len(cands) + 1):
        return ("random", min(8, INDEX_WORKERS_MAX)), []
    base = int(start_at if start_at is not None else len(members) // 2)
    base = max(0, min(base, len(members) - n * (len(cands) + 1)))
    results = []
    for slot, (strat, w) in enumerate(cands):
        sub = members.iloc[base + slot * n: base + (slot + 1) * n]
        rows = list(zip(sub["name"], sub["header_offset"], sub["compress_size"],
                        sub["compress_type"]))
        t0 = time.time()
        try:
            _run_blocks(rows, strat, w)
            results.append((strat, w, len(rows) / max(time.time() - t0, 1e-6)))
        except Exception as e:
            if not quiet:
                print(f"    {strat:>6} x{w:<3} FAILED ({type(e).__name__}: {str(e)[:50]}) -- "
                      f"not a candidate")
    if not results:
        return ("random", 8), []
    best = max(results, key=lambda t: t[2])
    return (best[0], best[1]), results


def _report_calibration(results, chosen, remaining):
    for strat, w, rate in results:
        eta = remaining / max(rate, 1e-6) / 60
        mark = "   <-- chosen" if (strat, w) == tuple(chosen) else ""
        print(f"    {strat:>6} x{w:<3} {rate:8,.0f} members/s   -> {eta:6.0f} min for what is "
              f"left{mark}")
    _r = [r for _s, _w, r in results]
    if _r and max(_r) < 2 * min(_r):
        print(f"    (the spread here is narrow -- {min(_r):,.0f} to {max(_r):,.0f} members/s -- so "
              f"do not read much into the winner; the mid-run check below is what matters.)")


def _chunk_path(i):
    return os.path.join(INDEX_DIR, f"chunk_{i:06d}.parquet")


# Chunk files are numbered by POSITION in the member list, so a chunk is only meaningful against
# the exact member list and chunk size that produced it. Change ARCHIVE_PREFIX, or INDEX_CHUNK, or
# add parts to the archive, and chunk 7 now covers a different set of members -- the old files
# would be silently merged alongside the new ones, producing an index with gaps and duplicates and
# no symptom at all. The key below turns that into a rebuild instead of a corruption.
# If the offsets were repaired for a 32-bit overflow, the member table is sorted differently and
# the chunk boundaries fall in different places, so chunks written before the repair must not be
# reused -- hence `offset_repair` in the key.
_INDEX_KEY = {"prefix": ARCHIVE_PREFIX, "chunk": int(INDEX_CHUNK),
              "n_members": int(len(MEMBERS)), "archive_size": int(MPF.size),
              "header_bytes": int(HEADER_BYTES),
              "offset_repair": bool(OFFSET_REPAIR), "schema": 1}
# Keys added to this dict AFTER a key file was written must not, by themselves, invalidate chunks
# that are still perfectly valid. A key file from before `offset_repair` existed is not evidence
# that a repair happened -- it is evidence that the notebook was older. Comparing field by field
# against a default turns a notebook upgrade into a no-op instead of silently throwing away hours
# of completed indexing. (This is not hypothetical: a real run had 128 of 151 chunks on disk.)
_INDEX_KEY_DEFAULTS = {"offset_repair": False}
_KEY_PATH = os.path.join(INDEX_DIR, "index_key.json")
if os.path.exists(_KEY_PATH):
    try:
        with open(_KEY_PATH) as f:
            _old_key = json.load(f)
    except Exception:
        _old_key = None
    _changed = [k for k, v in _INDEX_KEY.items()
                if (_old_key or {}).get(k, _INDEX_KEY_DEFAULTS.get(k, object())) != v]
    if _old_key is None or _changed:
        _stale = glob.glob(os.path.join(INDEX_DIR, "chunk_*.parquet"))
        print(f"index settings changed since the cached chunks were written -- discarding "
              f"{len(_stale)} stale chunk(s) and rebuilding.")
        print(f"  changed: {_changed}")
        for _p in _stale:
            os.remove(_p)
        with contextlib.suppress(Exception):
            os.remove(INDEX_MERGED_PATH)
with open(_KEY_PATH + ".tmp", "w") as f:
    json.dump(_INDEX_KEY, f, indent=2)
os.replace(_KEY_PATH + ".tmp", _KEY_PATH)

_total = len(MEMBERS) if MAX_MEMBERS_TO_INDEX is None else min(MAX_MEMBERS_TO_INDEX, len(MEMBERS))
_n_chunks = (_total + INDEX_CHUNK - 1) // INDEX_CHUNK
_done = {int(os.path.basename(p)[6:12]) for p in glob.glob(os.path.join(INDEX_DIR, "chunk_*.parquet"))}
_todo = [i for i in range(_n_chunks) if i not in _done]

print(f"header index: {_total:,} member(s) -> {_n_chunks:,} chunk(s) of {INDEX_CHUNK:,}")
print(f"  already done : {len(_done):,}   remaining: {len(_todo):,}")

_left_members = max(0, _total - len(_done) * INDEX_CHUNK)
_STRATEGY = INDEX_STRATEGY
if _todo and (INDEX_WORKERS == "auto" or INDEX_STRATEGY == "auto"):
    print(f"  measuring both read strategies on {INDEX_CALIBRATE_N} members each -- this decides "
          f"the next hour ...")
    (_cs, _cw), _cal = _calibrate(MEMBERS)
    if INDEX_STRATEGY == "auto":
        _STRATEGY = _cs
    if INDEX_WORKERS == "auto":
        INDEX_WORKERS = _cw
    _report_calibration(_cal, (_STRATEGY, INDEX_WORKERS), _left_members)
elif INDEX_WORKERS == "auto":
    INDEX_WORKERS = 8
if _STRATEGY == "auto":
    _STRATEGY = "random"
INDEX_WORKERS = max(1, int(INDEX_WORKERS))
print(f"  strategy: {_STRATEGY}, {INDEX_WORKERS} worker(s), {HEADER_BYTES:,} B of header per "
      f"member, in offset order")
if _STRATEGY == "bulk":
    print(f"    (bulk reads {INDEX_BULK_BLOCK_BYTES/1e6:.0f} MB per request and takes the headers "
          f"out of it, so it transfers far more and asks far less often)")

if _todo:
    t_start = time.time()
    n_members_done = 0
    _best_rate, _slow_streak, _recals = 0.0, 0, 0
    _skipped_chunks, _consec_skipped = [], 0
    _t_chunk = t_start
    for pos, ci in enumerate(tqdm(_todo, desc="index chunks")):
        lo, hi = ci * INDEX_CHUNK, min((ci + 1) * INDEX_CHUNK, _total)
        sub = MEMBERS.iloc[lo:hi]
        rows = list(zip(sub["name"], sub["header_offset"], sub["compress_size"],
                        sub["compress_type"]))
        recs = _run_blocks(rows, _STRATEGY, INDEX_WORKERS)

        # A transient Drive error must never be written into a chunk. The chunk would then be
        # marked done, the member would carry an error string forever, and its slice would be
        # dropped from its series with no symptom other than a volume that is quietly short a few
        # slices. Retry those members; if they still fail, the chunk is NOT written and this run
        # simply does it again next time.
        recs = _retry_transient_records(rows, recs)
        n_transient = sum(1 for r in recs if _is_transient_err(r.get("err")))
        if n_transient:
            _skipped_chunks.append(ci)
            _consec_skipped += 1
            print(f"    chunk {pos+1}: {n_transient:,} member(s) still failing with transient "
                  f"Drive I/O after {INDEX_IO_RETRY_ROUNDS} extra round(s). NOT writing this "
                  f"chunk --")
            print(f"    freezing a blip into the index would silently drop those slices from "
                  f"their series. It will be redone on the next run.")
            if _consec_skipped >= INDEX_MAX_CONSEC_SKIPS:
                print()
                print(f"    {_consec_skipped} chunks in a row failed this way, so Drive is wedged "
                      f"rather than busy. Stopping.")
                print(DRIVE_WEDGED_HINT)
                break
            continue
        _consec_skipped = 0

        df = pd.DataFrame(recs, columns=INDEX_FIELDS)
        tmp = _chunk_path(ci) + ".tmp"
        drive_retry(lambda: df.to_parquet(tmp, index=False),
                    what=f"writing chunk {ci}", quiet=False)
        drive_retry(lambda: os.replace(tmp, _chunk_path(ci)), what=f"committing chunk {ci}")
        n_members_done += len(rows)

        # INSTANTANEOUS, not cumulative. A cumulative average is what hid a 20x collapse in a real
        # run: it read 311, 271, 259, 124, 44 members/s, which looks like a gentle decline and was
        # actually 311 -> 15/s for the better part of an hour. Averages do not report weather.
        _now = time.time()
        inst = len(rows) / max(_now - _t_chunk, 1e-6)
        _t_chunk = _now
        avg = n_members_done / max(_now - t_start, 1e-6)
        left = (len(_todo) - pos - 1) * INDEX_CHUNK
        _best_rate = max(_best_rate, inst)
        _slow = _best_rate > 0 and inst < INDEX_RECALIBRATE_AT * _best_rate
        _slow_streak = _slow_streak + 1 if _slow else 0
        if pos == 0 or _slow or _slow_streak == 0 and (pos + 1) % 5 == 0 or pos + 1 == len(_todo):
            print(f"    chunk {pos+1:>4}/{len(_todo)}   now {inst:>6,.0f}/s   avg {avg:>6,.0f}/s"
                  f"   {left:>9,} left ~ {left/max(inst,1e-6)/60:>4.0f} min"
                  + (f"   <-- {inst/_best_rate:.0%} of the best seen ({_best_rate:,.0f}/s)"
                     if _slow else "")
                  + ("   (measured, not estimated)" if pos == 0 else ""))
        # Two chunks, not three. Detection is only as fast as a chunk, and a chunk of 5,000
        # members at a throttled 15/s is five and a half minutes -- so three of them means
        # sixteen minutes of crawling before anything happens. A false trigger costs a
        # sixty-second re-measurement that will simply re-pick the current strategy, which is a
        # far cheaper mistake to make.
        if _slow_streak >= 2 and _recals < INDEX_RECALIBRATE_MAX and left > 3 * INDEX_CHUNK:
            _recals += 1
            print(f"    the rate has been under {INDEX_RECALIBRATE_AT:.0%} of the best for two "
                  f"chunks. That is what a request-throttled Drive mount looks like, so this")
            print(f"    re-measures rather than waiting it out ({_recals}/"
                  f"{INDEX_RECALIBRATE_MAX}) ...")
            (_ns, _nw), _cal2 = _calibrate(MEMBERS, start_at=hi, quiet=True)
            _report_calibration(_cal2, (_ns, _nw), left)
            if (_ns, _nw) != (_STRATEGY, INDEX_WORKERS):
                print(f"    switching: {_STRATEGY} x{INDEX_WORKERS} -> {_ns} x{_nw}. The two "
                      f"strategies produce identical rows (cell 6 checks that), so nothing "
                      f"already written is affected.")
                _STRATEGY, INDEX_WORKERS = _ns, _nw
            _best_rate, _slow_streak, _t_chunk = 0.0, 0, time.time()
        if MAX_SECONDS_PER_RUN is not None and time.time() - t_start > MAX_SECONDS_PER_RUN:
            print(f"    time budget reached -- stopping. Re-run this cell to continue; "
                  f"{len(_todo)-pos-1} chunk(s) left.")
            break
    if _INDEX_EXEC["ex"] is not None:
        _INDEX_EXEC["ex"].shutdown(wait=True)
        _INDEX_EXEC["ex"], _INDEX_EXEC["w"] = None, None
    if _skipped_chunks:
        print()
        print(f"    {len(_skipped_chunks)} chunk(s) were not written because Drive kept returning "
              f"transient I/O errors for some of their members.")
        print(f"    Nothing is lost and nothing is wrong with the index that exists -- re-run this "
              f"cell and it will do exactly those chunks.")

_files = sorted(glob.glob(os.path.join(INDEX_DIR, "chunk_*.parquet")))
if not _files:
    raise RuntimeError("no index chunks were written -- nothing downstream can run.")
INDEX = pd.concat([pd.read_parquet(p) for p in _files], ignore_index=True)
# A previous run with a larger MAX_MEMBERS_TO_INDEX leaves chunks past the current horizon. They
# are valid (same key) and worth keeping -- but a duplicated member would double-count a slice in
# its series, so the merge is deduplicated explicitly rather than trusted to be disjoint.
_before = len(INDEX)
INDEX = INDEX.drop_duplicates(subset=["member"], keep="last").reset_index(drop=True)
if _before != len(INDEX):
    print(f"  merged chunks held {_before - len(INDEX):,} duplicate member row(s) from "
          f"overlapping runs; kept the most recent of each.")
# ---- repair rows that a transient Drive error poisoned in an EARLIER run ------------------------
# Chunks written before this notebook learned to retry can contain members whose only problem was
# a one-second Drive hiccup. They sit in the cache with an error string, are excluded from the
# series table, and quietly cost their series a slice -- which for a stack of ~33 is enough to
# open a z-gap and fail the whole series later, for no reason at all. Re-reading them is bounded
# work (one read each) and the repaired rows are written back into their chunks, so this happens
# once rather than every session.
_bad_io = INDEX[INDEX["err"].map(_is_transient_err)]
if len(_bad_io):
    print()
    print(f"repairing {len(_bad_io):,} index row(s) left behind by transient Drive I/O in an "
          f"earlier run ...")
    _pos = MEMBERS.reset_index().rename(columns={"index": "row"})
    _j = _bad_io[["member"]].merge(_pos, left_on="member", right_on="name", how="inner")
    _mpf = thread_reader()
    _fixed_rows, _still_bad = [], 0
    for _r in _j.itertuples(index=False):
        try:
            _blob = drive_retry(
                lambda: read_member(_mpf, _r.header_offset, _r.compress_size, _r.compress_type,
                                    max_bytes=HEADER_BYTES),
                what=f"re-reading {_r.member}")
            _rec = parse_header(_blob, _r.member)
            if _rec["series"] is None and int(_r.compress_size) > HEADER_BYTES:
                _blob = read_member(_mpf, _r.header_offset, _r.compress_size, _r.compress_type,
                                    max_bytes=HEADER_BYTES_ESCALATE)
                _rec = parse_header(_blob, _r.member)
        except Exception as _e:
            _rec = _err_rec(_r.member, _e)
        if _is_transient_err(_rec.get("err")):
            _still_bad += 1
        else:
            _fixed_rows.append((int(_r.row), _rec))
    print(f"  recovered {len(_fixed_rows):,} of {len(_j):,}"
          + (f"; {_still_bad:,} still failing (Drive, not the archive)" if _still_bad else ""))

    if _fixed_rows:
        _by_chunk = {}
        for _row, _rec in _fixed_rows:
            _by_chunk.setdefault(_row // INDEX_CHUNK, []).append(_rec)
        _rewritten = 0
        for _ci, _recs2 in sorted(_by_chunk.items()):
            _cp = _chunk_path(int(_ci))
            if not os.path.exists(_cp):
                continue
            _new = pd.DataFrame(_recs2, columns=INDEX_FIELDS)
            _old = pd.read_parquet(_cp)
            _old = _old[~_old["member"].isin(set(_new["member"]))]
            _out = pd.concat([_old, _new], ignore_index=True)[INDEX_FIELDS]
            drive_retry(lambda d=_out, q=_cp: d.to_parquet(q + ".tmp", index=False),
                        what=f"rewriting chunk {_ci}")
            drive_retry(lambda q=_cp: os.replace(q + ".tmp", q), what=f"committing chunk {_ci}")
            _rewritten += 1
        print(f"  rewrote {_rewritten} chunk file(s), so this repair does not have to happen again")
        _repaired = pd.DataFrame([r for _, r in _fixed_rows], columns=INDEX_FIELDS)
        INDEX = (pd.concat([INDEX[~INDEX["member"].isin(set(_repaired["member"]))], _repaired],
                           ignore_index=True)
                 .drop_duplicates(subset=["member"], keep="last").reset_index(drop=True))

drive_retry(lambda: INDEX.to_parquet(INDEX_MERGED_PATH + ".tmp", index=False),
            what="writing the merged index")
drive_retry(lambda: os.replace(INDEX_MERGED_PATH + ".tmp", INDEX_MERGED_PATH),
            what="committing the merged index")

_ok = INDEX[INDEX["err"].isna()]
_bad = INDEX[INDEX["err"].notna()]
print()
INDEX_IS_COMPLETE = bool(len(INDEX) >= len(MEMBERS))
print(f"indexed {len(INDEX):,} member(s) of {len(MEMBERS):,} in the archive "
      f"({len(INDEX)/max(len(MEMBERS),1):.1%})")
if not INDEX_IS_COMPLETE:
    _why = ("MAX_MEMBERS_TO_INDEX is set" if MAX_MEMBERS_TO_INDEX is not None
            else "this run stopped before finishing")
    print(f"  THE INDEX IS PARTIAL ({_why}). A partial index does not give you a few whole")
    print(f"  series -- it gives you a few slices of nearly every series. The next cell says what")
    print(f"  that means for the manifest.")
print(f"  parsed as DICOM : {len(_ok):,}")
print(f"  rejected        : {len(_bad):,}")
if len(_bad):
    for msg, n in _bad["err"].str[:70].value_counts().head(6).items():
        print(f"      [{n:,}x] {msg}")
print(f"  patients {_ok['patient'].nunique():,} | studies {_ok['study'].nunique():,} | "
      f"series {_ok['series'].nunique():,}")
print(f"  ImagePositionPatient present: {_ok['z'].notna().mean():.2%}   "
      f"PixelSpacing present: {_ok['sx'].notna().mean():.2%}")
_wrapped = pd.to_numeric(_ok["intercept"], errors="coerce") > -100
print(f"  slices whose RescaleIntercept is > -100 (the 12-bit wraparound signature): "
      f"{int(_wrapped.sum()):,} ({_wrapped.mean():.1%})")

In [ ]:
# 10. The series table, the dataset fingerprint, and the volume-level sample
#
# Slices are grouped into series, labels are aggregated (a series is positive for a subtype when
# ANY of its slices is -- which is how the RSNA labels are defined and what makes a volume-level
# task well posed), and the result is the manifest Part 3 works from.
#
# The fingerprint printed at the end is what says whether TARGET_SPACING_XYZ and TARGET_SHAPE_DHW
# are right for this archive, or a guess that is quietly cropping anatomy away.
import os

import contextlib
import json

import numpy as np
import pandas as pd

_ok = INDEX[INDEX["err"].isna()].copy()
_ok["slice_id"] = [member_to_slice_id(m) for m in _ok["member"]]

# Drop the known-corrupt file(s) HERE rather than discovering them at decode time. This does NOT
# rescue the series that owns one: the slice is still missing from the stack, so the z-uniformity
# check will still see a doubled gap and still reject it. What it buys is that the rejection is
# legible -- "a known-corrupt file was excluded" is printed here, so the gap message later has an
# explanation instead of looking like damaged data on Drive. The cost is one series out of 21,744,
# and rejecting a volume with a real hole in it is the correct outcome, not a workaround.
_bad = _ok["slice_id"].isin(KNOWN_BAD_MEMBERS)
if int(_bad.sum()):
    _bad_series = sorted(set(_ok.loc[_bad, "series"]))
    print(f"excluding {int(_bad.sum())} known-corrupt file(s) named in the top Kaggle solutions: "
          f"{sorted(set(_ok.loc[_bad, 'slice_id']))}")
    print(f"  they belong to {len(_bad_series)} series. Those series are now one slice short, so "
          f"the z-uniformity check will")
    print(f"  most likely reject them later -- correctly, since the stack really does have a hole. "
          f"Expect that failure and know why.")
    _ok = _ok[~_bad].copy()
_merged = _ok.merge(SLICE_LABELS, on="slice_id", how="left")
_n_unlabelled = int(_merged[RSNA_SUBTYPES].isna().all(axis=1).sum())
if _n_unlabelled:
    print(f"NOTE: {_n_unlabelled:,} indexed slice(s) had no row in the labels CSV and are treated "
          f"as all-negative. If that is a large fraction, check ARCHIVE_PREFIX.")
for c in RSNA_SUBTYPES:
    _merged[c] = pd.to_numeric(_merged[c], errors="coerce").fillna(0).astype("int8")

_g = _merged.groupby("series", sort=False)
SERIES = _g.agg(
    patient=("patient", "first"), study=("study", "first"),
    n_slices=("member", "size"),
    n_with_z=("z", lambda s: int(s.notna().sum())),
    z_min=("z", "min"), z_max=("z", "max"),
    sx=("sx", "median"), sy=("sy", "median"),
    rows_px=("rows", "median"), cols_px=("cols", "median"),
    n_shapes=("rows", "nunique"),
    n_wrapped=("intercept", lambda s: int((pd.to_numeric(s, errors="coerce") > -100).sum())),
).reset_index()
for c in RSNA_SUBTYPES:
    SERIES[c] = _g[c].max().to_numpy()
SERIES["n_pos_slices_any"] = _g["any"].sum().to_numpy()

_span = (SERIES["z_max"] - SERIES["z_min"]).abs()
SERIES["z_extent_mm"] = _span
SERIES["sz_measured"] = np.where(SERIES["n_slices"] > 1, _span / (SERIES["n_slices"] - 1), np.nan)
SERIES["eligible"] = (
    (SERIES["n_slices"] >= SERIES_MIN_SLICES)
    & (SERIES["n_with_z"] == SERIES["n_slices"])      # every slice has a real position
    & (SERIES["sx"].notna())
    & (SERIES["sz_measured"] > 0.1) & (SERIES["sz_measured"] < 20.0)
)

_e = SERIES[SERIES["eligible"]]
print("=" * 96)
print("SERIES TABLE")
print("=" * 96)
if not globals().get("INDEX_IS_COMPLETE", True):
    _cov = len(INDEX) / max(len(MEMBERS), 1)
    print()
    print("!" * 96)
    print(f"THE HEADER INDEX COVERS ONLY {_cov:.1%} OF THE ARCHIVE.")
    print("Every series below is missing most of its slices -- the index is a random sample of")
    print(f"MEMBERS, not of series, so a {len(SERIES):,}-series table here means ~{_cov:.0%} of each")
    print("series' slices, not a complete subset of series. Two consequences:")
    print("  * slice counts, z extents and measured spacings are NOT the real ones;")
    print("  * with the normal SERIES_MIN_SLICES almost nothing is eligible and the manifest comes")
    print("    out empty or tiny. That is the index being partial, not a bug.")
    if not SMOKE_TEST:
        print("Set SMOKE_TEST = True for a coherent plumbing check, or finish the index (set")
        print("MAX_MEMBERS_TO_INDEX = None and re-run cell 9 until it reports 100%).")
    print("!" * 96)
    print()
print(f"  series total     : {len(SERIES):,}")
print(f"  eligible         : {len(_e):,}"
      f"   (>= {SERIES_MIN_SLICES} slices, every slice positioned, plausible spacing)")
for _reason, _mask in (
        (f"fewer than {SERIES_MIN_SLICES} slices", SERIES["n_slices"] < SERIES_MIN_SLICES),
        ("some slice had no ImagePositionPatient", SERIES["n_with_z"] != SERIES["n_slices"]),
        ("no PixelSpacing", SERIES["sx"].isna()),
        ("implausible measured z spacing", ~((SERIES["sz_measured"] > 0.1)
                                             & (SERIES["sz_measured"] < 20.0)))):
    _n = int((_mask & ~SERIES["eligible"]).sum())
    if _n:
        print(f"      excluded, {_reason:<40s}: {_n:,}")
print(f"  patients {_e['patient'].nunique():,} | studies {_e['study'].nunique():,} | "
      f"{len(_e)/max(_e['patient'].nunique(),1):.2f} series per patient")

print()
print(f"{'':16}{'median':>10}{'p05':>10}{'p95':>10}{'min':>10}{'max':>10}")
for c, lbl in (("n_slices", "slices"), ("sx", "x spacing mm"), ("sz_measured", "z spacing mm"),
               ("z_extent_mm", "z extent mm"), ("rows_px", "rows px")):
    q = _e[c].quantile([0.05, 0.5, 0.95])
    print(f"  {lbl:<14}{q[0.5]:>10.2f}{q[0.05]:>10.2f}{q[0.95]:>10.2f}"
          f"{_e[c].min():>10.2f}{_e[c].max():>10.2f}")

_fov = tuple(TARGET_SHAPE_DHW[i] * TARGET_SPACING_XYZ[::-1][i] for i in range(3))
_inplane_mm = _e["sx"] * _e["rows_px"]
print()
print("FINGERPRINT vs the configured grid")
print(f"  cohort median spacing (x, y, z) : ({_e['sx'].median():.3f}, {_e['sy'].median():.3f}, "
      f"{_e['sz_measured'].median():.3f}) mm")
print(f"  TARGET_SPACING_XYZ              : {TARGET_SPACING_XYZ} mm")
_ratio = (TARGET_SPACING_XYZ[0] / max(_e["sx"].median(), 1e-6),
          TARGET_SPACING_XYZ[1] / max(_e["sy"].median(), 1e-6),
          TARGET_SPACING_XYZ[2] / max(_e["sz_measured"].median(), 1e-6))
print(f"  resampling factor (target/median): ({_ratio[0]:.2f}x, {_ratio[1]:.2f}x, {_ratio[2]:.2f}x)"
      f"   [>1 downsamples, <1 upsamples]")
print(f"  grid field of view (D,H,W)      : {_fov[0]:.0f} x {_fov[1]:.0f} x {_fov[2]:.0f} mm")
print(f"  native in-plane extent          : median {_inplane_mm.median():.0f} mm, "
      f"p95 {_inplane_mm.quantile(0.95):.0f} mm")
if min(_ratio) < 0.8:
    print("    ^ the grid is FINER than the data on some axis. Harmless but wasteful: it invents")
    print("      no information and costs backbone time.")

print()
print(f"{'subtype':<20}{'positive series':>17}{'prevalence':>13}{'positive slices':>18}")
for c in RSNA_SUBTYPES:
    print(f"  {c:<18}{int(_e[c].sum()):>17,}{_e[c].mean():>13.4f}"
          f"{int(_merged[c].sum()):>18,}")

# ---- does the configured grid actually fit this cohort? ----------------------------------
# THE NUMBER THAT MATTERS is not "how many series are cropped" -- it is "how many POSITIVE
# slices are cropped away". A series whose haemorrhage was cropped out is still labelled
# positive. That is label noise manufactured by the preprocessing, it is invisible to every
# metric downstream, and it would be baked into every cached vector.
def grid_fit_report(series_df, slice_df, shape_dhw, spacing_xyz, verbose=True):
    fov_d = float(shape_dhw[0]) * float(spacing_xyz[2])
    fov_h = float(shape_dhw[1]) * float(spacing_xyz[1])
    ext = series_df["z_extent_mm"].astype(float)
    inplane = (series_df["sx"].astype(float) * series_df["rows_px"].astype(float))
    lost_each_end = (ext - fov_d).clip(lower=0) / 2.0

    # per-slice: is this slice inside the centre crop its series will get?
    s = slice_df[["series", "z", "any"]].dropna(subset=["z"]).copy()
    s["z"] = s["z"].astype(float)
    keep_ids = set(series_df["series"])
    s = s[s["series"].isin(keep_ids)]
    ctr = s.groupby("series")["z"].agg(lambda v: (float(v.min()) + float(v.max())) / 2.0)
    s["kept"] = (s["z"] - s["series"].map(ctr)).abs() <= (fov_d / 2.0) + 1e-6
    pos = s[s["any"].astype(int) > 0]
    n_pos = int(len(pos))
    pos_lost = int((~pos["kept"]).sum()) if n_pos else 0
    # a series is EMPTIED when every one of its positive slices is cropped away
    emptied = 0
    if n_pos:
        byser = pos.groupby("series")["kept"].max()
        emptied = int((~byser).sum())
    rep = {
        "fov_d_mm": fov_d, "fov_h_mm": fov_h,
        "frac_series_cropped_z": float((ext > fov_d).mean()),
        "frac_series_cropped_xy": float((inplane > fov_h).mean()),
        "mm_lost_each_end_p50": float(lost_each_end[lost_each_end > 0].median())
                                if (lost_each_end > 0).any() else 0.0,
        "mm_lost_each_end_p95": float(lost_each_end.quantile(0.95)),
        "n_positive_slices": n_pos,
        "frac_positive_slices_cropped": (pos_lost / n_pos) if n_pos else 0.0,
        "n_series_emptied": emptied,
        "n_positive_series": int(pos["series"].nunique()) if n_pos else 0,
    }
    if verbose:
        print(f"  grid FOV                        : {fov_d:.0f} mm deep x {fov_h:.0f} mm "
              f"in plane")
        print(f"  series cropped in z             : {rep['frac_series_cropped_z']:.1%}"
              + (f"   (median {rep['mm_lost_each_end_p50']:.0f} mm lost at EACH end, "
                 f"p95 {rep['mm_lost_each_end_p95']:.0f} mm)"
                 if rep["frac_series_cropped_z"] else ""))
        print(f"  series cropped in plane         : {rep['frac_series_cropped_xy']:.1%}")
        print(f"  POSITIVE slices cropped away    : "
              f"{rep['frac_positive_slices_cropped']:.2%} of {n_pos:,}")
        print(f"  positive series left with NO lesion in frame : "
              f"{emptied:,} of {rep['n_positive_series']:,}")
    return rep


def suggest_grid(series_df, slice_df, shape_dhw, spacing_xyz, target_keep=0.995):
    """Candidate grids that keep >= target_keep of positive slices, cheapest first.

    Depth and z-spacing trade off at CONSTANT cost: 64 slices at 4 mm covers a third more
    anatomy than 64 at 3 mm for exactly the same number of voxels. And on this dataset 3 mm
    is finer than the acquisition, so the extra depth is not bought with real resolution --
    it is bought with interpolation that was inventing nothing to begin with.
    """
    base_vox = float(shape_dhw[0]) * shape_dhw[1] * shape_dhw[2]
    out = []
    for D in (shape_dhw[0], 80, 96, 112, 128):
        for sz in (2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0):
            r = grid_fit_report(series_df, slice_df, (D, shape_dhw[1], shape_dhw[2]),
                                (spacing_xyz[0], spacing_xyz[1], sz), verbose=False)
            keep = 1.0 - r["frac_positive_slices_cropped"]
            cost = (D * shape_dhw[1] * shape_dhw[2]) / base_vox
            out.append((cost, D, sz, D * sz, keep, r["frac_series_cropped_z"]))
    ok = sorted([o for o in out if o[4] >= target_keep])
    return out, ok


print()
print("GRID FIT -- does the configured volume actually contain the findings?")
GRID_FIT = grid_fit_report(_e, _merged, TARGET_SHAPE_DHW, TARGET_SPACING_XYZ)
_all_opts, _ok_opts = suggest_grid(_e, _merged, TARGET_SHAPE_DHW, TARGET_SPACING_XYZ)
if _ok_opts:
    print()
    print(f"  grids that keep >= 99.5% of positive slices, cheapest first:")
    print(f"    {'cost':>6}  {'D':>4} {'z mm':>6} {'FOV mm':>8}  {'pos kept':>9}  "
          f"{'series cropped':>15}")
    _seen = set()
    for cost, D, sz, fov, keep, fcs in _ok_opts[:6]:
        if (D, sz) in _seen:
            continue
        _seen.add((D, sz))
        _star = "  <- same cost as now" if abs(cost - 1.0) < 1e-9 else ""
        print(f"    {cost:>5.2f}x  {D:>4} {sz:>6.1f} {fov:>8.0f}  {keep:>8.2%}  "
              f"{fcs:>14.1%}{_star}")
    print()
    print("  Depth and z-spacing trade off at CONSTANT cost. Changing TARGET_SPACING_XYZ or")
    print("  TARGET_SHAPE_DHW changes the preprocessing signature, so decide NOW -- after")
    print("  extraction starts, changing either one discards every cached vector.")

# ---- the gate ----------------------------------------------------------------------------
# A print is not enough here. UNATTENDED_RUN is the documented overnight setting, and a
# warning in hour zero of a fifty-hour run scrolls past unread.
if not ACKNOWLEDGE_GRID_CROP and (
        GRID_FIT["frac_positive_slices_cropped"] > MAX_POSITIVE_SLICE_CROP
        or GRID_FIT["n_series_emptied"] > 0):
    raise RuntimeError(
        f"THE CONFIGURED GRID CROPS AWAY FINDINGS.\n"
        f"  {GRID_FIT['frac_positive_slices_cropped']:.2%} of positive slices fall outside "
        f"the {GRID_FIT['fov_d_mm']:.0f} mm centre crop "
        f"(limit {MAX_POSITIVE_SLICE_CROP:.2%}), and {GRID_FIT['n_series_emptied']:,} "
        f"positive series would be left with no lesion in frame at all.\n"
        f"  Those series stay labelled positive. That is label noise created by the "
        f"preprocessing, it is invisible to every metric downstream, and it would be baked "
        f"into every cached vector for the rest of the study.\n"
        f"  * Pick one of the grids listed above -- several cost exactly what the current "
        f"one costs, because depth and z-spacing trade off at constant voxel count.\n"
        f"  * Or set ACKNOWLEDGE_GRID_CROP = True if you have decided this loss is "
        f"acceptable and want it on the record.\n"
        f"  Deciding now is cheap. Deciding after extraction is not: TARGET_SHAPE_DHW and "
        f"TARGET_SPACING_XYZ are both in the preprocessing signature, so changing either "
        f"one discards every vector already cached.")
if GRID_FIT["frac_positive_slices_cropped"] > 0:
    print(f"\n  NOTE: {GRID_FIT['frac_positive_slices_cropped']:.2%} of positive slices are "
          f"cropped away"
          + (" (acknowledged)" if ACKNOWLEDGE_GRID_CROP else "") + ".")


# ---- the volume-level stratified sample ---------------------------------------------------------
def stratified_series_sample(df, n_target, label_cols, seed=0):
    """Sample proportionally WITHIN each distinct multi-label pattern.

    That is what preserves every subtype's prevalence, so average precision and PPV mean the same
    thing on the sample as on the whole set. Sampling rows uniformly would not, and rare patterns
    would vanish entirely.
    """
    # SORTED FIRST, and sorted again on the way out. The permutation below indexes rows by
    # POSITION, so without this the subset drawn depends on the order the caller happened to
    # hand them over -- and that order comes from a groupby(sort=False) over a header index
    # built by threads, which is not stable between sessions. Parallel shards are taken by
    # position in this manifest, so a manifest that is not reproducible means the shards do
    # not partition the cohort: some series extracted twice, others never, silently.
    if n_target is None or n_target >= len(df):
        return df.sort_values("series", kind="mergesort").reset_index(drop=True)
    d = df.sort_values("series", kind="mergesort").reset_index(drop=True)
    d = d.assign(_pat=d[label_cols].astype(int).astype(str).agg("".join, axis=1))
    rng = np.random.default_rng(seed)
    frac = n_target / len(d)
    picks = []
    for _p, grp in d.groupby("_pat", sort=True):
        k = max(1, min(int(round(len(grp) * frac)), len(grp)))
        picks.append(grp.iloc[np.sort(rng.permutation(len(grp))[:k])])
    out = pd.concat(picks).drop(columns=["_pat"]).sort_values("series", kind="mergesort")
    out = out.reset_index(drop=True)
    if len(out) > n_target:
        out = out.iloc[np.sort(rng.permutation(len(out))[:n_target])].reset_index(drop=True)
    return out.sort_values("series", kind="mergesort").reset_index(drop=True)


if SMOKE_TEST and 0 < len(_e) < MAX_SERIES:
    print(f"\nNOTE: only {len(_e)} series are eligible from this partial index, fewer than the "
          f"{MAX_SERIES} SMOKE_TEST asks for.")
    print(f"      That is fine for a plumbing check. To get more, raise MAX_MEMBERS_TO_INDEX "
          f"(currently {MAX_MEMBERS_TO_INDEX:,}) and re-run cell 9 -- it keeps what it has.")

MANIFEST_SAMPLER_VERSION = 2      # bump to force a fresh draw


def manifest_fingerprint(df):
    """A hash of the cohort ITSELF, not of the settings that produced it.

    This is what makes the cache safe under any interleaving of writes: the key states which
    series it vouches for, so a key paired with a different parquet -- an older cohort left
    behind by a run that died between the two writes -- is detected instead of trusted.
    """
    import hashlib
    ids = sorted(str(s) for s in df["series"])
    h = hashlib.sha256(("\n".join(ids)).encode()).hexdigest()[:32]
    return {"n": int(len(ids)), "sha256_16": h}


def _manifest_key(eligible):
    """What the cached manifest must still agree with to be reusable."""
    return {"sampler": MANIFEST_SAMPLER_VERSION, "max_series": MAX_SERIES,
            "seed": int(SAMPLE_SEED), "n_eligible": int(len(eligible)),
            "subtypes": list(RSNA_SUBTYPES)}


# REUSED WHEN IT STILL APPLIES. Every session recomputing its own manifest is how parallel
# shards drift apart; and on a resume it is also how a cohort quietly changes underneath
# features that were already extracted for the old one.
_MKEY_PATH = MANIFEST_PATH + ".key.json"
_want = _manifest_key(_e)
MANIFEST = None
_manifest_from_cache = False
if os.path.exists(MANIFEST_PATH) and os.path.exists(_MKEY_PATH):
    try:
        with open(_MKEY_PATH) as _fh:
            _have = json.load(_fh)
        _cfg_ok = all(_have.get(k) == v for k, v in _want.items())
        if _cfg_ok:
            _cand = drive_retry(lambda: pd.read_parquet(MANIFEST_PATH),
                                what="reading the cached manifest")
            _fp = manifest_fingerprint(_cand)
            if _have.get("fingerprint") == _fp:
                MANIFEST = _cand
                _manifest_from_cache = True
                print(f"\nmanifest REUSED from {MANIFEST_PATH} ({len(MANIFEST):,} series, "
                      f"fingerprint {_fp['sha256_16'][:12]}) -- the cohort is identical to "
                      f"the earlier run and to every other session.")
            else:
                print(f"\nmanifest cache REJECTED: the key vouches for "
                      f"{(_have.get('fingerprint') or {}).get('n')} series "
                      f"({str((_have.get('fingerprint') or {}).get('sha256_16'))[:12]}) but "
                      f"the parquet holds {_fp['n']} ({_fp['sha256_16'][:12]}).")
                print(f"  A run died between writing the two. Drawing a new cohort -- which "
                      f"is the safe outcome, not a problem.")
        else:
            _changed = [k for k in _want if _have.get(k) != _want[k]]
            print(f"\nmanifest cache ignored: {_changed} changed since it was written. "
                  f"Drawing a new cohort.")
    except Exception as _e_:
        print(f"\nmanifest cache unreadable ({type(_e_).__name__}); drawing a new cohort.")
if MANIFEST is None:
    MANIFEST = stratified_series_sample(_e, MAX_SERIES, RSNA_SUBTYPES, seed=SAMPLE_SEED)
for _df, _path in ((SERIES, SERIES_TABLE_PATH), (MANIFEST, MANIFEST_PATH)):
    drive_retry(lambda d=_df, q=_path: d.to_parquet(q + ".tmp", index=False),
                what=f"writing {os.path.basename(_path)}", quiet=False)
    drive_retry(lambda q=_path: os.replace(q + ".tmp", q),
                what=f"committing {os.path.basename(_path)}")

# members of each selected series, ordered along the slice normal
_sel = set(MANIFEST["series"])
_slices = _ok[_ok["series"].isin(_sel)].sort_values(["series", "z"], kind="mergesort")
SERIES_MEMBERS = _slices.groupby("series")["member"].apply(list).to_dict()

if len(MANIFEST) == 0:
    _msg = ["no series is eligible, so the manifest is empty and Part 3 would have nothing to do."]
    if not globals().get("INDEX_IS_COMPLETE", True):
        _msg.append(f"The header index covers {len(INDEX)/max(len(MEMBERS),1):.1%} of the archive, "
                    f"which is almost certainly the reason: each series has only a fraction of its "
                    f"slices and so fails SERIES_MIN_SLICES={SERIES_MIN_SLICES}.")
        if SMOKE_TEST:
            _msg.append(f"You are in SMOKE_TEST mode, so the fix is to index more: raise "
                        f"MAX_MEMBERS_TO_INDEX (currently {MAX_MEMBERS_TO_INDEX:,}) and re-run "
                        f"cell 9 -- the chunks already written are kept, so it only does the new "
                        f"ones.")
        else:
            _msg.append("Either finish the index (MAX_MEMBERS_TO_INDEX = None and re-run cell 9 "
                        "until it reports 100%), or set SMOKE_TEST = True for a coherent "
                        "small run.")
    else:
        _msg.append(f"The index is complete, so this is a real property of the data. The exclusion "
                    f"counts printed above say which test each series failed; the usual causes are "
                    f"SERIES_MIN_SLICES={SERIES_MIN_SLICES} being too high or slices missing "
                    f"ImagePositionPatient.")
    raise RuntimeError(" ".join(_msg))

print()
print("=" * 96)
print(f"MANIFEST: {len(MANIFEST):,} series selected"
      + (f" (stratified sample of {len(_e):,}, seed {SAMPLE_SEED})"
         if MAX_SERIES is not None and MAX_SERIES < len(_e) else " (all eligible)"))
print("=" * 96)
print(f"  {'subtype':<18}{'sample':>10}{'prevalence':>13}{'full set':>13}{'drift':>9}")
for c in RSNA_SUBTYPES:
    a, b = MANIFEST[c].mean(), _e[c].mean()
    print(f"  {c:<18}{int(MANIFEST[c].sum()):>10,}{a:>13.4f}{b:>13.4f}{a-b:>+9.4f}")
print(f"  slices to read: {int(MANIFEST['n_slices'].sum()):,} "
      f"({MANIFEST['n_slices'].sum()*0.5/1e3:.1f} GB at ~500 KB each)")
print(f"  patients {MANIFEST['patient'].nunique():,} | studies {MANIFEST['study'].nunique():,}")
# The key is written LAST, and only once the parquet it describes is actually on Drive --
# see manifest_fingerprint. Writing it earlier is what lets a key outlive the cohort it was
# written for.
if not _manifest_from_cache:
    _want_full = {**_want, "fingerprint": manifest_fingerprint(MANIFEST)}
    with contextlib.suppress(Exception):
        with open(_MKEY_PATH + ".tmp", "w") as _fh:
            json.dump(_want_full, _fh, indent=2)
        os.replace(_MKEY_PATH + ".tmp", _MKEY_PATH)
    print(f"  manifest fingerprint {_want_full['fingerprint']['sha256_16'][:12]} "
          f"({_want_full['fingerprint']['n']:,} series) recorded for the other sessions")
print(f"  -> {MANIFEST_PATH}")

---

## Part 3 — Phase A: volumes, then frozen features

The expensive forward pass happens **once per backbone**; every adapter tested afterwards trains on
the cached vectors in seconds. That is what makes a grid of backbones × adapters tractable later —
the costly step runs once per backbone, not once per cell.

Three things happen per series, in this order:

1. **Assemble the HU volume.** Read that series' members — concurrently and in *archive* order,
   not slice order, because RSNA is a flat directory and one series' ~35 slices are scattered
   through 195 GB, so reading them in z order means seeking backwards across the whole file. The
   decode stays serial and in slice order, and cell 11 proves on your own data that the reordered
   read produces a bit-identical volume. Then apply the 12-bit correction per slice,
   drop duplicate SOPInstanceUIDs, keep the modal in-plane shape, order along the slice normal, and
   measure the through-plane spacing from the geometry rather than trusting `SliceThickness`.
2. **QC, before any GPU time is spent.** Air near −1000 *and* a top percentile that is not sitting
   where wrapped values live. Both, because the first alone does not catch partial wraparound.
3. **Resample to the fixed grid, window twice, and run the frozen backbone** once per window,
   concatenating to 512 x len(HU_WINDOWS) = 1536-d.

### Where the preprocessing constants come from

Not from a blog post about the competition — from the source of the solutions that placed. Each
of these was read out of the repository, and the ones that could not be verified are named as
unverified rather than smoothed over.

| choice | value | who |
|---|---|---|
| brain window | (40, 80) | unanimous across every top-5 repo |
| subdural window | (80, 200) | darraghdog (2nd), okotaku (3rd), appian42 (11th) |
| third window | soft tissue (40, 380) | darraghdog (2nd), okotaku (3rd), appian42 (11th) — **not** the bone window; okotaku had (600, 2800) behind a flag and shipped with it off |
| 12-bit correction gate | `BitsStored == 12 and PixelRepresentation == 0 and RescaleIntercept > -100` | appian42 (11th) and Anjum48 (5th), both citing Jeremy Howard's fastai kernel. 1st/2nd/3rd did **not** apply it |
| slice ordering | `ImagePositionPatient`, never `InstanceNumber` | all six repos read; not one references `InstanceNumber` |
| known-corrupt file | `ID_6431af929` | the only hard-coded ID in any of them |
| in-plane resolution | 480–512 px over ~250 mm | 1st, 2nd, 3rd, 5th |

Two deliberate **non**-adoptions, both because the evidence says they do not pay:

* **No brain/head cropping.** 1st and 3rd place did not crop at all, and 2nd place — who did, with
  a `threshold=0` bounding box — reported in his own README that "just with preprocessed crop
  achieves same score". The 5th-place `ndimage.label` second-largest-blob crop is the most
  principled version if it is ever wanted, but nothing here suggests it is worth the complexity.
* **No explicit −2000 padding rule.** No top solution has one; their window clamp absorbs it,
  because they clamp to the window's own lower bound rather than to zero. This notebook clamps to
  −1024 HU one step earlier instead, which produces the identical windowed output and additionally
  lets the air-percentile QC work — that check is ours, and it is what the padding was breaking.

Resumable at series granularity — only `"ok"` counts as done — so a dropped session costs minutes
and a bug fix costs only the series the bug actually broke.

In [ ]:
# 11. Volume assembly, including the two corrections this dataset genuinely needs
#
# OUT-OF-FOV PADDING  (present in essentially every series)
# --------------------------------------------------------
# CT reconstructs a CIRCLE and stores it in a SQUARE. The corners -- 1 - pi/4 = 21.5% of every
# slice -- are filled with a padding value, and on this dataset it is the classic one: raw -2000,
# which after RescaleIntercept -1024 lands at exactly -3024 HU. The giveaway that it is padding
# and not anatomy is that it is the SAME constant in every series.
#
# It is clamped to air below, before anything measures or resamples the volume, for one reason
# that matters: 21.5% is far more than 1%, so the 1st percentile lands inside the padding. Left
# alone it reads -3024, and the air check then rejects a perfectly good scan with "the rescale
# intercept is wrong for this series" when the intercept is exactly right. On the real archive
# that rejected the first two series it looked at -- and would have rejected nearly all 4,000.
#
# It was invisible in testing because the synthetic fixtures reconstructed into the full square
# and had no padding at all. There is now a fixture that pads like a real scanner.
#
# THE 12-BIT WRAPAROUND  (0.7% of slices, measured on the real archive)
# --------------------------------------------------------------------
# Some RSNA-ICH files store 12-bit unsigned data whose values have wrapped modulo 4096, tagged
# with a RescaleIntercept near 0 instead of ~-1000. Air then reads about +3096 HU instead of
# -1000, every window is computed on nonsense, and the backbone returns a perfectly healthy-looking
# feature vector of garbage. Nothing raises. (An earlier version of this note said "a large
# minority" -- that was 45%, the rate on a synthetic rig built to exercise the correction. Cell 9
# measured the real archive: 5,379 of 752,803 slices, 0.7%.)
#
# The correction below is Jeremy Howard's fastai cleaning kernel for this competition, reproduced
# in several medal-winning solutions:
#
#     if BitsStored == 12 and PixelRepresentation == 0 and int(RescaleIntercept) > -100:
#         x = pixel_array + 1000;  x[x >= 4096] -= 4096;  RescaleIntercept = -1000
#
# What is worth knowing, and what the QC below is built around: on a mixed series the usual
# air-percentile check does NOT catch this. Measured on a rig where 45% of slices were wrapped,
# the 1st-percentile HU still read -1011 because the clean slices dominate that tail; only the
# 99th percentile gave it away (4094 instead of 1946). So both ends are checked.
import concurrent.futures as _cf
import io
from collections import Counter

import numpy as np
import pydicom



# WHAT THE KERNEL CANNOT RECOVER, and why that is left as a QC rejection rather than guessed
# at. The map `stored = HU mod 4096` is only invertible by `x = stored + 1000; x[x >= 4096]
# -= 4096` for HU in [-1000, 3095]. Outside it:
#     HU below -1000  ->  comes back as +2096 .. +3095
#     HU at or above 3096  ->  comes back as -1000 .. -1
# A real reconstruction floors around -1024, so the first band is a thin sliver of air just
# under the floor, and it reads as roughly +3080. That is indistinguishable, voxel by voxel,
# from genuine dense metal at the same value -- so the correction does NOT try to guess. It
# leaves the residue, the 99th-percentile QC check rejects the series, and the count below
# tells you which of the two causes it was. On the real archive this band does not occur:
# 5,379 of 752,803 slices were corrected and every corrected series passed QC.


def fix_rsna_pixels(arr, bits_stored, px_repr, intercept):
    """Returns (array, intercept_override_or_None)."""
    try:
        needs = (int(bits_stored) == 12 and int(px_repr) == 0 and int(intercept) > -100)
    except Exception:
        return arr, None
    if not needs:
        return arr, None
    x = arr.astype(np.int32) + 1000
    x[x >= 4096] -= 4096
    return x, -1000.0


def _slice_normal(ds):
    iop = getattr(ds, "ImageOrientationPatient", None)
    if iop is None or len(iop) != 6:
        return None
    try:
        v = [float(t) for t in iop]
    except Exception:
        return None
    n = np.cross(v[:3], v[3:6])
    nn = float(np.linalg.norm(n))
    return (n / nn) if np.isfinite(nn) and nn > 1e-6 else None


# One executor for the whole extraction, not one per series. Its threads are what make the
# thread-local readers above worth having: 4,000 series x 8 short-lived pools would rebuild them
# 32,000 times.
_EXTRACT_EXEC = {"ex": None, "w": None}


def _extract_executor(w):
    if _EXTRACT_EXEC["ex"] is None or _EXTRACT_EXEC["w"] != w:
        if _EXTRACT_EXEC["ex"] is not None:
            _EXTRACT_EXEC["ex"].shutdown(wait=True)
        _EXTRACT_EXEC["ex"] = _cf.ThreadPoolExecutor(max_workers=max(1, int(w)))
        _EXTRACT_EXEC["w"] = int(w)
    return _EXTRACT_EXEC["ex"]


def _fetch_series_bytes(members, member_lookup, workers=None):
    """Fetch one series' member bytes, concurrently and in ARCHIVE-OFFSET order.

    Two separate reasons, both worth stating:

    ORDER. RSNA-ICH is a flat directory, so the slices of one series are scattered arbitrarily
    through the 195 GB archive. Reading them in slice order means seeking backwards and forwards
    across the whole file; reading them in offset order advances monotonically, which is what a
    FUSE-mounted Drive is good at. This is free to do because `assemble_volume` sorts the decoded
    slices by z itself -- the order they arrive in has never mattered to the result.

    CONCURRENCY. At ~35 slices per volume and a round trip that dominates the cost, a serial read
    spends most of its wall clock waiting. Each worker gets its own `MultiPartFile` because that
    object carries a file position and cannot be shared.

    Only the BYTES are fetched here. Decoding stays serial, in the caller's order, with the same
    CRC check inside `read_member` -- so nothing about how the volume is built changes, only when
    the bytes turn up. Failures are returned rather than raised, so one bad member still fails as
    itself and the series keeps its per-member accounting.

    Returns {member_name: bytes | Exception}.
    """
    # EXTRACT_READ_WORKERS may still be the string "auto" here: the self-tests below run
    # before the calibration block that resolves it. Resolve to a sane default rather than
    # letting a sentinel reach int().
    _w = workers if workers is not None else EXTRACT_READ_WORKERS
    w = max(1, int(_w)) if isinstance(_w, (int, float)) else 8
    todo = [(n, member_lookup[n]) for n in members if n in member_lookup]
    todo.sort(key=lambda t: t[1][0])
    # Bound the memory one series can take. compress_size x3 is a deliberately loose upper bound on
    # what a DICOM deflates from; a series projected past the cap is read the old way, serially,
    # rather than held whole.
    projected_mb = sum(int(r[1]) for _n, r in todo) * 3 / 1e6
    if w == 1 or len(todo) <= 2 or projected_mb > EXTRACT_PREFETCH_MAX_MB:
        out = {}
        for n, r in todo:
            try:
                out[n] = read_member(MPF, r[0], r[1], r[2], crc=r[3])
            except Exception as e:
                out[n] = e
        return out

    per = (len(todo) + w - 1) // w
    blocks = [todo[i:i + per] for i in range(0, len(todo), per)]

    def _one(block):
        # thread_reader(), not a fresh MultiPartFile: building one per block per series stat'd all
        # 91 parts each time -- 2.9 MILLION metadata calls over a 4,000-series extraction, 21 for
        # every slice actually read.
        m = thread_reader()
        res = []
        for n, r in block:
            try:
                res.append((n, read_member(m, r[0], r[1], r[2], crc=r[3])))
            except Exception as e:
                res.append((n, e))
        return res

    out = {}
    for res in _extract_executor(w).map(_one, blocks):
        out.update(res)
    return out


def assemble_volume(members, mpf, member_lookup, blobs=None):
    """Read one series' members and stack them into a (D, H, W) HU volume.

    Returns (volume, {"sx","sy","sz"} in mm, info). Raises ValueError with a specific message when
    the series cannot honestly become a volume -- it never returns a partial one silently.
    """
    decoded, seen_sop, notes = [], set(), []
    n_read_failed = n_undecodable = n_dup_sop = n_fixed = n_crc_bad = n_io = 0
    n_wrap_residue = 0
    n_padding = n_voxels = 0
    if blobs is None:
        blobs = _fetch_series_bytes(members, member_lookup)

    for name in members:
        blob = blobs.get(name)
        if blob is None:
            n_read_failed += 1                      # not in the member table at all
            continue
        if isinstance(blob, Exception):
            n_read_failed += 1
            if "CRC mismatch" in str(blob):
                n_crc_bad += 1
            elif is_transient_io(blob):
                n_io += 1
            continue
        try:
            ds = pydicom.dcmread(io.BytesIO(blob), force=True)
        except Exception:
            n_read_failed += 1
            continue
        sop = getattr(ds, "SOPInstanceUID", None)
        if sop is not None:
            if sop in seen_sop:
                n_dup_sop += 1          # re-exported studies really do contain these, and a
                continue                # duplicated slice both distorts the measured spacing and
            seen_sop.add(sop)           # doubles that slice's weight in the volume
        try:
            arr = ds.pixel_array
        except Exception:
            n_undecodable += 1
            continue
        icpt = safe_float_attr(ds, "RescaleIntercept", 0.0) or 0.0
        slope = safe_float_attr(ds, "RescaleSlope", 1.0) or 1.0
        if APPLY_RSNA_PIXEL_FIX:
            arr, override = fix_rsna_pixels(arr, getattr(ds, "BitsStored", 0),
                                            getattr(ds, "PixelRepresentation", 0), icpt)
            if override is not None:
                icpt = override
                n_fixed += 1
        hu = np.asarray(arr, dtype=np.float32) * slope + icpt
        if CLAMP_HU_FLOOR:
            # Everything outside the circular reconstruction FOV. On this dataset it is stored as
            # raw -2000 and lands at exactly -3024 HU after the intercept -- an identical constant
            # in every series, which is what gives it away as padding rather than anatomy. It is
            # ~21.5% of every square slice, so leaving it in puts the 1st percentile 2000 HU below
            # air and makes the air check reject a perfectly good scan.
            below = int(np.count_nonzero(hu < HU_AIR_FLOOR))
            if below:
                n_padding += below
                hu = np.maximum(hu, HU_AIR_FLOOR)
        # Voxels sitting in the band the 12-bit kernel cannot invert (see the note above it).
        # Counted whether or not this slice was corrected, because the two cases the QC
        # rejection has to distinguish are "a slice that should have been corrected and was
        # not" and "a slice that was corrected and has sub-floor residue left in it".
        n_wrap_residue += int(np.count_nonzero((hu >= 2900.0) & (hu <= 3200.0)))
        n_voxels += hu.size
        decoded.append((ds, hu))

    if n_io:
        # A transient Drive error must not quietly shorten a volume. The reads above already
        # retried; if some still failed, the honest outcome is a FAILED series that this cell
        # will pick up again next run -- not a stack that is missing slices, marked "ok", and
        # cached under a signature that says it is complete. Slices lost from the middle would
        # trip the z-uniformity check, but slices lost from either END would not: the volume
        # would simply be a little shorter than the patient's head, and nothing would say so.
        raise ValueError(
            f"{n_io} of {len(members)} slice(s) still failed with a transient Drive I/O error "
            f"after retrying. Failing this series rather than building it from the {len(decoded)} "
            f"that survived -- re-run this cell when the mount is healthy and it will be retried.")

    if not decoded:
        raise ValueError(f"no readable/decodable pixel data ({len(members)} member(s): "
                         f"{n_read_failed} unreadable of which {n_crc_bad} failed CRC, "
                         f"{n_undecodable} undecodable, {n_dup_sop} duplicate SOP)")

    info = {"n_members": len(members), "n_read_failed": n_read_failed, "n_io_transient": n_io,
            "n_undecodable": n_undecodable, "n_duplicate_sop": n_dup_sop,
            "n_pixel_corrected": n_fixed, "n_crc_mismatch": n_crc_bad,
            "n_wrap_residue": n_wrap_residue,
            "frac_wrap_residue": (n_wrap_residue / n_voxels) if n_voxels else 0.0,
            "frac_below_floor": (n_padding / n_voxels) if n_voxels else 0.0,
            "hu_floor": (float(HU_AIR_FLOOR) if CLAMP_HU_FLOOR else None)}
    if n_crc_bad:
        notes.append(f"{n_crc_bad} member(s) failed their CRC-32 check and were dropped -- those "
                     f"bytes are not what was archived")

    normal = _slice_normal(decoded[0][0])
    _iops = {tuple(round(float(t), 5)
                   for t in (getattr(d, "ImageOrientationPatient", ()) or ())) for d, _a in decoded}
    if len(_iops) > 1:
        notes.append(f"slices disagree on ImageOrientationPatient ({len(_iops)} distinct values) "
                     f"-- this is not one coherent stack; ordering uses the first slice's normal")
    info["slice_normal"] = ([round(float(v), 5) for v in normal] if normal is not None else None)

    zs, from_ipp = [], []
    for i, (ds, _a) in enumerate(decoded):
        got, z = False, float(i)
        ipp = getattr(ds, "ImagePositionPatient", None)
        if ipp is not None and len(ipp) == 3:
            try:
                p = np.array([float(ipp[0]), float(ipp[1]), float(ipp[2])])
                z = float(p @ normal) if normal is not None else float(p[2])
                got = True
            except Exception:
                pass
        if not got:
            n = safe_float_attr(ds, "InstanceNumber", None)
            z = float(n) if n is not None else float(i)
        zs.append(z)
        from_ipp.append(got)

    all_ipp = all(from_ipp)
    if not all_ipp and any(from_ipp):
        # Mixed scales. Re-derive EVERY key as an ordinal: sorting a millimetre value of 285.0
        # against an ordinal of 12 drops mid-brain slices to the bottom of the stack, and the
        # median-of-gaps spacing estimate survives that scrambling without complaining.
        notes.append(f"{len(from_ipp)-sum(from_ipp)}/{len(from_ipp)} slice(s) had no "
                     f"ImagePositionPatient; the whole series was ordered by InstanceNumber "
                     f"rather than mixing millimetres with ordinals")
        zs = [(safe_float_attr(d, "InstanceNumber", i) or float(i))
              for i, (d, _a) in enumerate(decoded)]

    order = np.argsort(np.asarray(zs, dtype=np.float64), kind="stable")
    decoded = [decoded[i] for i in order]
    zs = [zs[i] for i in order]
    info["z_from_ipp"] = all_ipp

    # Reference in-plane shape = the MOST COMMON one, not the first slice's. Slices are sorted by
    # z at this point, so "the first slice" is whichever sits at the bottom of the stack; if that
    # one is the odd one out, taking it as the reference throws away the real volume and keeps the
    # junk. kept in lockstep with the datasets, because spacing is read from decoded[0] below and
    # an odd-shaped frame is exactly the one whose PixelSpacing differs.
    shapes = Counter(a.shape for _d, a in decoded)
    ref_shape, _ = shapes.most_common(1)[0]
    keep = [i for i, (_d, a) in enumerate(decoded) if a.shape == ref_shape]
    n_shape_dropped = len(decoded) - len(keep)
    info["n_shape_mismatch"] = n_shape_dropped
    if n_shape_dropped:
        notes.append(f"{n_shape_dropped} slice(s) dropped for not matching the modal in-plane "
                     f"shape {ref_shape}; shapes seen: {dict(shapes)}")
    decoded = [decoded[i] for i in keep]
    zs = [zs[i] for i in keep]

    if len(decoded) < SERIES_MIN_SLICES:
        raise ValueError(f"only {len(decoded)} usable slice(s), need >= {SERIES_MIN_SLICES}")
    kept = len(decoded) / max(len(members), 1)
    info["kept_slice_fraction"] = float(kept)
    if kept < MIN_KEPT_SLICE_FRACTION:
        # An absolute floor is not enough: a 40-slice series where 22 fail to decode still clears
        # it and would be logged 'ok' -- an embedding of half a head, indistinguishable downstream
        # from a genuinely short scan.
        raise ValueError(
            f"only {len(decoded)} of {len(members)} member(s) survived reading "
            f"({kept:.0%} < {MIN_KEPT_SLICE_FRACTION:.0%}): {n_read_failed} unreadable "
            f"({n_crc_bad} of them CRC failures), "
            f"{n_undecodable} undecodable, {n_dup_sop} duplicate SOP, {n_shape_dropped} shape "
            f"mismatch. Refusing to build a partial volume.")

    vol = np.stack([a for _d, a in decoded], axis=0).astype(np.float32)
    info["n_slices"] = int(vol.shape[0])

    ref = decoded[0][0]
    px = getattr(ref, "PixelSpacing", None)
    try:
        sx, sy = (float(px[1]), float(px[0])) if px is not None and len(px) == 2 else (1.0, 1.0)
    except Exception:
        sx, sy = 1.0, 1.0

    sz = None
    if all_ipp and len(zs) >= 2:
        gaps = np.diff(np.asarray(zs, dtype=np.float64))
        gaps = gaps[np.abs(gaps) > 1e-6]
        if gaps.size:
            sz = float(np.median(np.abs(gaps)))
            info["z_spacing_source"] = ("measured along the slice normal" if normal is not None
                                        else "measured from ImagePositionPatient z")
            tag = (safe_float_attr(ref, "SpacingBetweenSlices")
                   or safe_float_attr(ref, "SliceThickness"))
            if tag and abs(tag - sz) > max(0.15 * tag, 0.2):
                notes.append(f"measured z spacing {sz:.3f} mm differs from the tag value "
                             f"{tag:.3f} mm (overlap or gap); the measured value is used")
            n_dup_z = int((np.abs(np.diff(np.asarray(zs))) <= 1e-6).sum())
            if n_dup_z:
                notes.append(f"{n_dup_z} pair(s) of slices share a z position")
                info["n_duplicate_z"] = n_dup_z
            # The largest gap against the median gap: ~1.0 on an intact stack, ~2.0 when one
            # slice is missing. This is the only signal that a hole exists, because the median
            # spacing used for resampling is by construction blind to it.
            info["z_gap_ratio"] = float(np.max(np.abs(gaps)) / max(sz, 1e-6))
    if sz is None or not np.isfinite(sz) or sz <= 0:
        sz = (safe_float_attr(ref, "SpacingBetweenSlices")
              or safe_float_attr(ref, "SliceThickness") or 1.0)
        info["z_spacing_source"] = "SpacingBetweenSlices/SliceThickness tag"

    p = np.percentile(vol, [1, 50, 99])
    info["hu_p01"], info["hu_p50"], info["hu_p99"] = float(p[0]), float(p[1]), float(p[2])
    info["note"] = " | ".join(notes)
    return vol, {"sx": float(sx), "sy": float(sy), "sz": float(sz)}, info


def qc_volume(info):
    """Returns None when the volume passes, or a reason string when it does not."""
    if info.get("n_crc_mismatch", 0) > MAX_CRC_FAILURES_PER_SERIES:
        return (f"QC: {info['n_crc_mismatch']} member(s) failed their CRC-32 check "
                f"(limit {MAX_CRC_FAILURES_PER_SERIES}) -- those bytes on Drive are not what was "
                f"archived, and a volume built from the survivors is indistinguishable from an "
                f"intact one")
    if REJECT_NONUNIFORM_Z and info.get("z_gap_ratio") is not None:
        if info["z_gap_ratio"] > MAX_Z_GAP_RATIO:
            return (f"QC: largest inter-slice gap is {info['z_gap_ratio']:.2f}x the median "
                    f"(limit {MAX_Z_GAP_RATIO}) -- the stack has a hole, and resampling on the "
                    f"median spacing would compress the anatomy across it without any downstream "
                    f"symptom")
    frac = float(info.get("frac_below_floor", 0.0) or 0.0)
    if frac > MAX_PADDING_FRACTION:
        return (f"QC: {frac:.1%} of voxels were below the HU floor ({HU_AIR_FLOOR:.0f}), more "
                f"than the {MAX_PADDING_FRACTION:.0%} an out-of-FOV reconstruction circle can "
                f"account for (that is 21.5% of a square slice). Something other than padding is "
                f"down there -- most likely a rescale intercept that is wrong by a large negative "
                f"amount")
    if not (QC_HU_P01_RANGE[0] <= info["hu_p01"] <= QC_HU_P01_RANGE[1]):
        return (f"QC: 1st HU percentile {info['hu_p01']:.0f} outside {QC_HU_P01_RANGE} -- air "
                f"should read near -1000, so the rescale intercept is wrong for this series")
    if info["hu_p99"] > QC_HU_P99_MAX:
        # TWO different causes land here and they need different fixes, so the message says
        # which one it is instead of always blaming uncorrected slices.
        n_fix = int(info.get("n_pixel_corrected", 0) or 0)
        n_sl = int(info.get("n_slices", 0) or 0)
        frac = float(info.get("frac_wrap_residue", 0.0) or 0.0)
        if n_fix and frac > 0.001:
            why = (f"{n_fix}/{n_sl} slice(s) WERE corrected and {frac:.2%} of voxels still sit "
                   f"in +2900..+3200 -- that is air below the reconstruction floor, which the "
                   f"12-bit kernel cannot invert (see the note in the assembly cell). Not "
                   f"recoverable by re-running; this series is genuinely unusable")
        elif n_fix:
            why = (f"{n_fix}/{n_sl} slice(s) were corrected but the top percentile is still "
                   f"high, so something other than the wraparound is producing these values")
        else:
            why = ("NO slice was corrected, so some are still wrapped even though the air "
                   "percentile looks fine -- check BitsStored/PixelRepresentation/"
                   "RescaleIntercept on this series")
        return (f"QC: 99th HU percentile {info['hu_p99']:.0f} exceeds {QC_HU_P99_MAX:.0f}, "
                f"where 12-bit wraparound parks air (~+3096). {why}")
    if info["hu_p99"] < QC_HU_P99_MIN:
        # Clamping the floor is what makes this check necessary. An intercept that is wrong by a
        # large negative amount would push every voxel onto the floor, and the air check above
        # would then PASS on a volume that is uniformly air. A head CT contains skull; if the top
        # percentile does not reach bone, this is not a head CT that was read correctly.
        return (f"QC: 99th HU percentile {info['hu_p99']:.0f} is below {QC_HU_P99_MIN:.0f} -- a "
                f"head CT contains skull, so a top percentile this low means the volume is "
                f"essentially all air. Suspect the rescale intercept, not the anatomy")
    return None


MEMBER_LOOKUP = {n: (o, c, t, k) for n, o, c, t, k in zip(
    MEMBERS["name"], MEMBERS["header_offset"], MEMBERS["compress_size"], MEMBERS["compress_type"],
    MEMBERS["crc"])}
print(f"volume assembly ready. RSNA 12-bit correction: "
      f"{'ON' if APPLY_RSNA_PIXEL_FIX else '*** OFF ***'}")
print(f"member lookup: {len(MEMBER_LOOKUP):,} entries")


# ---- self-test on real members of THIS archive --------------------------------------------------
def _assembly_self_test(n=2):
    picked = list(SERIES_MEMBERS.items())[:n]
    out = []
    for suid, mem in picked:
        vol, sp, info = assemble_volume(mem, MPF, MEMBER_LOOKUP)
        out.append((suid, vol.shape, sp, info, qc_volume(info)))
    return out


def _prefetch_equivalence_test():
    """The concurrent, offset-ordered fetch must produce the SAME volume as reading serially in
    slice order. It is an I/O reordering and nothing more, so anything other than bit-identical
    output is a bug -- and a bug of the worst kind, because a subtly different volume still
    trains, still scores, and never raises."""
    if not SERIES_MEMBERS:
        return None
    suid, mem = next(iter(SERIES_MEMBERS.items()))
    v_par, sp_par, i_par = assemble_volume(mem, MPF, MEMBER_LOOKUP)
    serial = {}
    for _n in mem:
        _r = MEMBER_LOOKUP.get(_n)
        if _r is None:
            continue
        try:
            serial[_n] = read_member(MPF, _r[0], _r[1], _r[2], crc=_r[3])
        except Exception as _e:
            serial[_n] = _e
    v_ser, sp_ser, i_ser = assemble_volume(mem, MPF, MEMBER_LOOKUP, blobs=serial)
    same = (v_par.shape == v_ser.shape and np.array_equal(v_par, v_ser) and sp_par == sp_ser
            and i_par.get("n_members") == i_ser.get("n_members"))
    if not same:
        raise RuntimeError(
            "the concurrent slice prefetch produced a DIFFERENT volume from the serial read "
            f"({v_par.shape} vs {v_ser.shape}). Set EXTRACT_READ_WORKERS = 1 and do not trust any "
            "feature vector built with it until this is understood.")
    return len(mem), v_par.shape


if SERIES_MEMBERS:
    _eq = _prefetch_equivalence_test()
    if _eq:
        print(f"prefetch equivalence: a {_eq[0]}-slice series read concurrently in archive order "
              f"is bit-identical\n                      to the same series read serially in slice "
              f"order -> {_eq[1]}")
    print()
    print("self-test -- building the first volume(s) from this archive:")
    for _s, _shape, _sp, _i, _q in _assembly_self_test():
        print(f"  {_s[-16:]}  {_shape}  spacing=({_sp['sx']:.2f},{_sp['sy']:.2f},{_sp['sz']:.2f}) mm")
        print(f"      HU p01={_i['hu_p01']:8.1f}  p50={_i['hu_p50']:7.1f}  p99={_i['hu_p99']:8.1f}"
              f"   12-bit corrected: {_i['n_pixel_corrected']}/{_i['n_slices']} slices")
        print(f"      out-of-FOV padding: {_i.get('frac_below_floor', 0.0):.1%} of voxels were "
              f"below {HU_AIR_FLOOR:.0f} HU and were clamped to air"
              + ("   (a square slice holding a circular reconstruction is 21.5% padding)"
                 if _i.get('frac_below_floor', 0.0) > 0.05 else ""))
        print(f"      z from geometry: {_i['z_from_ipp']} ({_i.get('z_spacing_source','')})"
              f"   max/median gap: {_i.get('z_gap_ratio', float('nan')):.2f}x")
        print(f"      dropped: {_i['n_duplicate_sop']} dup-SOP, {_i['n_shape_mismatch']} shape, "
              f"{_i.get('n_crc_mismatch', 0)} CRC")
        print(f"      QC: {'PASS' if _q is None else _q}")
        if _i.get("note"):
            print(f"      note: {_i['note']}")

# ---- how many readers? Measured, not assumed. --------------------------------------------
def calibrate_read_workers(candidates=None, n_series=3, verbose=True):
    """Time assembling a few real series at several concurrencies and return the best.

    Only the ARRIVAL of bytes changes; the decode order, the CRC checks and the assembled
    volume are identical at every setting (see _prefetch_equivalence_test above), so this is
    free to tune. The series used are read and thrown away -- nothing is written and no
    progress is recorded, so this cannot interfere with a resume.
    """
    import statistics
    import time as _t
    cands = [c for c in (candidates or EXTRACT_READ_WORKER_CANDIDATES) if c >= 1]
    pool = [s for s in MANIFEST["series"] if len(SERIES_MEMBERS.get(s, [])) >= 10]
    if not pool or len(cands) < 2:
        return int(EXTRACT_READ_WORKERS if isinstance(EXTRACT_READ_WORKERS, int) else 8), []
    # spread the probe across the archive: consecutive series sit next to each other on disk
    # and would flatter whichever setting happened to go second, once the cache is warm
    step = max(1, len(pool) // (n_series + 1))
    probe = [pool[min(len(pool) - 1, (i + 1) * step)] for i in range(n_series)]
    results = []
    for w in cands:
        secs = []
        for suid in probe:
            try:
                _t0 = _t.time()
                assemble_volume(SERIES_MEMBERS[suid], MPF, MEMBER_LOOKUP, workers=w)
                secs.append(_t.time() - _t0)
            except Exception:
                continue                    # a QC-rejectable series still measures the read
        if secs:
            results.append((statistics.median(secs), w, len(secs)))
    if not results:
        return int(EXTRACT_READ_WORKERS if isinstance(EXTRACT_READ_WORKERS, int) else 8), []
    results.sort()
    best_s, best_w, _n = results[0]
    if verbose:
        print(f"  read-concurrency probe on {len(probe)} series "
              f"({sum(len(SERIES_MEMBERS[s]) for s in probe)} slices):")
        for s, w, k in sorted(results, key=lambda r: r[1]):
            bar = "#" * max(1, int(round(20 * best_s / max(s, 1e-9))))
            print(f"    {w:>3} worker(s)  {s:>7.2f} s/series  {bar}"
                  + ("   <- best" if w == best_w else ""))
        _slow = max(results)[0]
        if _slow > 1.15 * best_s:
            print(f"    the worst setting is {_slow/best_s:.2f}x the best -- on a "
                  f"{len(MANIFEST):,}-series run that is "
                  f"{(_slow-best_s)*len(MANIFEST)/3600:.1f} h of difference")
        else:
            print(f"    concurrency barely matters on this mount "
                  f"({_slow/best_s:.2f}x spread); leaving it at {best_w}")
    return int(best_w), results


if EXTRACT_READ_WORKERS == "auto":
    EXTRACT_READ_WORKERS, READ_CAL = calibrate_read_workers()
    print(f"  EXTRACT_READ_WORKERS = {EXTRACT_READ_WORKERS} (measured on this runtime)")
else:
    READ_CAL = []
    print(f"  EXTRACT_READ_WORKERS = {EXTRACT_READ_WORKERS} (fixed in the config; set "
          f'"auto" to measure it)')


In [ ]:
# 12. Preprocessing for the 3D backbone -- resample -> whole-brain crop/pad -> HU windowing
#
# Every choice below is sourced or reasoned, not picked by hand.
#
#   RESAMPLE to TARGET_SPACING_XYZ using each scan's OWN measured spacing (assemble_volume
#     measures z from the ImagePositionPatient gaps rather than trusting SliceThickness, which
#     disagrees with reality whenever there is slice overlap or a gap). SimpleITK, sitkLinear by
#     default: in plane this is a DOWNsample on this dataset (~0.45 mm native to the configured
#     TARGET_SPACING_XYZ, so about 2.2x at 1.0 mm), where B-spline buys nothing and its overshoot
#     at the skull/air boundary is a real artefact. (This note used to quote a 4x factor and a
#     2.0 mm target, left over from an earlier default -- the grid-fit report in cell 10 prints
#     the real factor measured on your own cohort.) Set RESAMPLE_INTERPOLATOR = "bspline" if you disagree; the preprocessing
#     signature tracks the choice, so cached features are re-extracted rather than mixed.
#
#   CROP/PAD to TARGET_SHAPE_DHW on a fixed whole-head grid, NOT a lesion-centred crop: negatives
#     and deployment-time cases, where no lesion location is known in advance, must be framed
#     identically to training positives, or the framing itself leaks the label. True centre
#     crop/pad on each axis: start = (current - target) // 2. Padding uses the volume's own
#     minimum, which for a head CT is air.
#
#   DUAL HU WINDOWING, brain + subdural, each clipped to [C - W/2, C + W/2] and
#     rescaled to [0, 1]. Then each window is z-scored over the whole volume, which is MedicalNet's
#     own intensity normalisation (Med3D, Chen et al. 2019) -- the pretrained weights expect
#     zero-mean unit-variance input, and feeding them [0, 1] instead shifts every activation in
#     the frozen stem before a single learned parameter is involved.
#
#   ONE FORWARD PASS PER WINDOW. MONAI refuses `pretrained=True` for any n_input_channels != 1
#     (see monai/networks/nets/resnet.py::_resnet), so a 2-channel input would mean inflating
#     conv1's weights -- which changes the pretrained function. Running the frozen backbone twice
#     and concatenating keeps the published weights bit-exact. Cost: 2x the forward passes, which
#     on a T4 is ~0.1 s per volume against ~30 s of zip reading. Not the bottleneck.
#
# The functions are in src/qct3d/preprocessing.py, moved verbatim; notebooks 01 (RSNA) and 05
# (external cohort) import the same file, so the two cohorts cannot drift apart. The settings
# chosen in this notebook's configuration cell are handed to that module here.
import numpy as np
import SimpleITK as sitk
import torch

import qct3d.preprocessing as _prep

_prep.configure(TARGET_SPACING_XYZ=TARGET_SPACING_XYZ, TARGET_SHAPE_DHW=TARGET_SHAPE_DHW,
                HU_WINDOWS=HU_WINDOWS, RESAMPLE_INTERPOLATOR=RESAMPLE_INTERPOLATOR,
                SKULL_STRIP=SKULL_STRIP)
from qct3d.preprocessing import (_SITK_INTERP, _preprocessing_self_test,  # noqa: E402,F401
                                 center_crop_or_pad, hu_window, preprocess_for_backbone,
                                 resample_to_spacing)

_rs, _os = _preprocessing_self_test()
_fov = tuple(round(TARGET_SHAPE_DHW[i] * TARGET_SPACING_XYZ[::-1][i]) for i in range(3))
print(f"windows: {[(n, f'C{c:.0f}/W{w:.0f}') for n, c, w in HU_WINDOWS]}")
print(f"grid   : {tuple(TARGET_SHAPE_DHW)} (D,H,W) at {TARGET_SPACING_XYZ} mm "
      f"-> {_fov[0]} x {_fov[1]} x {_fov[2]} mm field of view (D,H,W)")
print(f"self-test: anisotropic 40x320x384 @ (0.7,0.7,3.0) mm -> resampled {_rs} -> model input {_os}")
print("preprocessing self-test passed (axis order, shape, standardisation, short-volume padding, "
      "windows actually differ)")
if not SKULL_STRIP:
    print()
    print("NOTE: SKULL_STRIP is False. A frozen backbone can read skull thickness, scalp and")
    print("      scanner fingerprints instead of parenchyma, and the AUROC that results looks")
    print("      entirely normal. See the closing notes before reporting any of these numbers.")

In [ ]:
# 13. The frozen backbone zoo -- and the loading that PROVES itself
#
# THE FAILURE THIS CELL EXISTS TO PREVENT
# ---------------------------------------
# If one backbone is pretrained and another is silently randomly initialised, then the
# grid is not comparing architectures -- it is comparing "has weights" with "does not",
# and the pretrained one wins for a reason that has nothing to do with inductive bias.
# Nothing downstream complains: a random 3D CNN produces perfectly healthy-looking
# features, trains, and reports a plausible AUROC of noise.
#
# A key count cannot catch this (a state_dict that matched 3 buffers and nothing else
# still "loads"). The only check that cannot be fooled is behavioural: build a second,
# freshly-random copy and confirm the two produce DIFFERENT features. That is what
# verify_pretrained_loaded does, and it raises rather than warns.
#
# THE ZOO
#   B1  SwinUNETR encoder  hybrid CNN-Transformer  SuPreM, 2,100 CT, supervised
#   B2  SegResNet encoder  pure CNN                SuPreM, 2,100 CT, supervised
#   B3  ResNet-18          pure CNN (classic)      MedicalNet, 23 datasets
#   B4  SwinUNETR encoder  hybrid                  self-supervised, 5,050 CT  (optional)
#
# B1 and B2 share a pretraining corpus and supervision signal, so B1-vs-B2 isolates
# ARCHITECTURE. B4-vs-B1 holds architecture fixed and isolates the PRETRAINING REGIME.
import contextlib
import gc
import os

import numpy as np
import torch
import torch.nn as nn

SUPREM_URLS = {
    "suprem_swinunetr": "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/supervised_suprem_swinunetr_2100.pth",
    "suprem_segresnet": "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/supervised_suprem_segresnet_2100.pth",
    "ssl_swinunetr_5050": "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/self_supervised_nv_swin_unetr_5050.pt",
}


def download_checkpoint(key, dest_dir):
    """Cached on Drive: the ~250 MB SwinUNETR download happens once, not once per session."""
    url = SUPREM_URLS[key]
    os.makedirs(dest_dir, exist_ok=True)
    path = os.path.join(dest_dir, os.path.basename(url))
    if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
        print(f"  checkpoint cached: {os.path.basename(path)} "
              f"({os.path.getsize(path)/1e6:.0f} MB)")
        return path
    print(f"  downloading {os.path.basename(url)} ...")
    try:
        from huggingface_hub import hf_hub_download
        got = hf_hub_download(repo_id="MrGiovanni/SuPreM",
                              filename=os.path.basename(url), local_dir=dest_dir)
        print(f"  saved {got}")
        return got
    except Exception as e:
        print(f"  huggingface_hub failed ({type(e).__name__}); falling back to a direct GET")
        import urllib.request
        tmp = path + ".part"
        urllib.request.urlretrieve(url, tmp)
        os.replace(tmp, path)
        print(f"  saved {path} ({os.path.getsize(path)/1e6:.0f} MB)")
        return path


def _unwrap_state_dict(obj):
    for key in ("state_dict", "net", "model", "module", "student"):
        if isinstance(obj, dict) and key in obj and isinstance(obj[key], dict):
            return _unwrap_state_dict(obj[key])
    return obj


_STRIP_PREFIXES = ("module.", "backbone.", "model.", "net.", "swinViT.", "swin_vit.",
                   "encoder.", "_orig_mod.")


def _candidate_names(k):
    """Every name a checkpoint key `k` could correspond to in MY module tree.

    THE CASE THAT USED TO BE MISSING. A released checkpoint is saved from the authors'
    module tree, not from mine. SuPreM's SwinUNETR keys look like
    `module.swinViT.patch_embed...`, and my wrapper stores that trunk as `self.swinViT`,
    so simply stripping `module.` lands on the right name -- which is why B1 loaded at
    100%. Their SegResNet keys look like `module.backbone.convInit...`, and my wrapper
    stores it as `self.net`, so the right name is `net.convInit...`: their prefix has to
    come OFF and mine has to go ON, in that order.

    The old version generated only the strip-chain, plus each prefix prepended to the
    ORIGINAL key. It never combined the two, so that rename was unreachable and the load
    matched nothing -- 0% coverage, and the behavioural check below would have stopped the
    run at cell 13 with a message about prefix handling. Correct behaviour, hours before
    it needed to be, but only because the check exists.

    Now the candidates are the CROSS PRODUCT of {every stripped form} x {nothing, or each
    prefix put back on}. `_match_by_unique_suffix` covers whatever naming this still fails
    to anticipate.
    """
    out, seen = [], []

    def add(x):
        if x and x not in seen:
            seen.append(x)
            out.append(x)

    stripped = [k]
    cur = k
    for _ in range(6):
        changed = False
        for p in _STRIP_PREFIXES:
            if cur.startswith(p) and len(cur) > len(p):
                cur = cur[len(p):]
                if cur not in stripped:
                    stripped.append(cur)
                changed = True
        if not changed:
            break
    for s in stripped:            # the stripped forms first: an exact hit is preferred
        add(s)
    for s in stripped:            # then each of them with one of my own names put back on
        for p in _STRIP_PREFIXES:
            add(p + s)
    return out


def _match_by_unique_suffix(sd, target, already):
    """Last resort for a checkpoint whose module tree we did not anticipate.

    Matches a checkpoint key to a target key when one target ends with the checkpoint
    key's trailing path AND that target is the ONLY one that does AND the shapes agree.
    Ambiguity is refused rather than guessed: a wrong-but-plausible mapping is the failure
    mode this whole cell exists to prevent, so a tie matches nothing.
    """
    free = [t for t in target if t not in already]
    extra = {}
    for k, v in sd.items():
        parts = k.split(".")
        for i in range(len(parts)):
            tail = "." + ".".join(parts[i:])
            hits = [t for t in free
                    if ("." + t).endswith(tail) and tuple(target[t].shape) == tuple(v.shape)]
            if len(hits) == 1:
                extra[hits[0]] = v
                free.remove(hits[0])
                break
            if len(hits) > 1:
                break             # ambiguous at this depth; a shorter tail is worse
    return extra


def load_state_dict_forgiving(module, ckpt_path, verbose=True):
    """Match by name-with-prefix-stripping AND shape. Reports; does not assert -- the
    behavioural check below is the one that actually decides."""
    raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd = _unwrap_state_dict(raw)
    if not isinstance(sd, dict):
        raise RuntimeError(f"{ckpt_path} holds no state_dict (got {type(sd).__name__})")
    sd = {k: v for k, v in sd.items() if isinstance(v, torch.Tensor)}
    target = module.state_dict()
    matched, mism = {}, []
    for k, v in sd.items():
        for cand in _candidate_names(k):
            if cand in target:
                if tuple(target[cand].shape) == tuple(v.shape):
                    matched[cand] = v
                else:
                    mism.append((k, tuple(v.shape), tuple(target[cand].shape)))
                break
    n_by_name = len(matched)
    if n_by_name < len(target):
        # Nothing above anticipated this checkpoint's module tree. Fall back to matching on
        # the trailing path, accepting only unambiguous, shape-compatible hits.
        matched.update(_match_by_unique_suffix(sd, target, set(matched)))
    module.load_state_dict({**target, **matched}, strict=False)
    tot = sum(int(np.prod(t.shape)) for t in target.values())
    got = sum(int(np.prod(t.shape)) for t in matched.values())
    cov = got / max(tot, 1)
    if verbose:
        print(f"  checkpoint tensors : {len(sd):,}")
        print(f"  matched            : {len(matched):,} / {len(target):,} tensors"
              + (f"  ({len(matched)-n_by_name} by unique suffix)"
                 if len(matched) > n_by_name else ""))
        print(f"  PARAMETER COVERAGE : {cov:.1%}")
        if mism:
            print(f"  shape mismatches   : {len(mism)} (e.g. {mism[:2]})")
    return {"coverage": cov, "matched": len(matched), "target": len(target),
            "by_suffix": len(matched) - n_by_name, "mismatches": len(mism)}


def verify_load_changed_model(model, load_fn, probe_shape=(1, 1, 64, 96, 96), atol=1e-6):
    """Compare the model against ITSELF, before and after loading. Unfoolable.

    WHY NOT "compare against a fresh random twin", which is the obvious design: two
    SEPARATELY constructed models have DIFFERENT random initialisations, so their features
    differ whether or not the checkpoint loaded anything. That check reports "weights
    loaded" for a model with 0% parameter coverage -- i.e. for a completely random network.
    It was written that way first, tested, and found to fail silently.

    Measuring the model against its own pre-load state has no such hole: if the checkpoint
    changed nothing, the features are bit-identical, and that is a fact about the
    checkpoint rather than about the RNG.
    """
    model.eval()
    torch.manual_seed(1234)
    x = torch.randn(*probe_shape)
    with torch.no_grad():
        before = model(x).float().clone()
    before_sd = {k: v.detach().clone() for k, v in model.state_dict().items()}

    rep = load_fn(model)

    after_sd = model.state_dict()
    n_changed = sum(1 for k in before_sd
                    if k in after_sd and not torch.equal(before_sd[k], after_sd[k]))
    model.eval()
    with torch.no_grad():
        after = model(x).float()
    rel = float((after - before).norm() / (before.norm() + 1e-9))
    identical = bool(torch.allclose(before, after, atol=atol))
    print(f"  tensors changed by the load : {n_changed} / {len(before_sd)}")
    print(f"  feature change vs pre-load  : {rel:.4f}")
    # FINITENESS FIRST. allclose() is False whenever either side holds NaN, so a checkpoint
    # that loads garbage -- negative BatchNorm variances are the usual way -- "differs from
    # the pre-load state" exactly as a good one does and passes the test below. The only
    # visible sign is `rel` printing as nan, one line above the word "verified".
    if not torch.isfinite(after).all():
        raise RuntimeError(
            "THE LOADED CHECKPOINT PRODUCES NON-FINITE FEATURES. The load did change the "
            "model, so the difference test below would have passed it -- but the model now "
            "outputs NaN or Inf, which means the tensors that arrived are not usable "
            "weights (negative BatchNorm running_var is the usual cause). This is a "
            "corrupt or mismatched checkpoint, not a loading-path problem.")
    if n_changed == 0 or identical:
        raise RuntimeError(
            "THE PRETRAINED WEIGHTS DID NOT REACH THIS BACKBONE. Loading the checkpoint "
            "changed nothing -- the model's features are bit-identical to what they were "
            "before the load, so it is still randomly initialised. Every number this "
            "notebook goes on to produce would be the output of a random network, and "
            "nothing else would complain. Fix the checkpoint path or the prefix handling "
            "above before continuing. Do NOT extract features in this state.")
    print("  -> weights verified: the load actually changed the model")
    return rep


def verify_constructor_loaded(build_pretrained, build_random, probe_shape=(1, 1, 64, 96, 96),
                              atol=1e-6):
    """For weights loaded INSIDE the constructor (MONAI's pretrained=True), where there is
    no pre-load state to snapshot. Both models are built under the SAME seed, so their
    random initialisations are identical by construction and any feature difference can
    only come from the pretrained weights."""
    torch.manual_seed(4321)
    m_pre = build_pretrained()
    torch.manual_seed(4321)
    m_rnd = build_random()
    m_pre.eval()
    m_rnd.eval()
    torch.manual_seed(1234)
    x = torch.randn(*probe_shape)
    with torch.no_grad():
        a, b = m_pre(x).float(), m_rnd(x).float()
    rel = float((a - b).norm() / (b.norm() + 1e-9))
    print(f"  pretrained vs same-seed random init: relative difference {rel:.4f}")
    # Same hole as in verify_load_changed_model: NaN makes the two "differ".
    if not torch.isfinite(a).all():
        raise RuntimeError(
            "THE PRETRAINED BACKBONE PRODUCES NON-FINITE FEATURES. It does differ from the "
            "same-seed random init, so the difference test below would have passed it -- "
            "but the weights that arrived are not usable (negative BatchNorm running_var is "
            "the usual cause). Do NOT extract features in this state.")
    if bool(torch.allclose(a, b, atol=atol)):
        raise RuntimeError(
            "THE PRETRAINED WEIGHTS DID NOT REACH THIS BACKBONE: built with "
            "pretrained=True it is bit-identical to the same architecture built with "
            "pretrained=False under the same seed. Every number downstream would come "
            "from a random network.")
    print("  -> weights verified: pretrained differs from same-seed random init")
    return m_pre


# ---- encoder wrappers -------------------------------------------------------------------------
class SwinUNETREncoder(nn.Module):
    """SwinUNETR's swinViT trunk; deepest stage has 16 * feature_size channels."""

    def __init__(self, feature_size=48, spatial_map_size=None):
        super().__init__()
        from monai.networks.nets import SwinUNETR
        # `img_size` was a REQUIRED positional argument in MONAI 1.3, optional-and-ignored
        # in 1.4, and gone in 1.5+. Calling the 1.5 signature on 1.3 raises TypeError; the
        # fallback keeps this cell working on every version in between rather than pinning
        # the notebook to whatever MONAI Colab happened to ship the week it was written.
        _kw = dict(in_channels=1, out_channels=1, feature_size=feature_size,
                   spatial_dims=3)
        try:
            net = SwinUNETR(**_kw)
        except TypeError:
            import warnings as _w
            with _w.catch_warnings():
                _w.simplefilter("ignore")
                net = SwinUNETR(img_size=(96, 96, 96), **_kw)   # MONAI <= 1.3
        del _kw
        self.swinViT = net.swinViT
        self.feat_dim = 16 * feature_size
        self.spatial_map_size = spatial_map_size

    def _deep(self, x):
        return self.swinViT(x, normalize=True)[-1]

    def forward(self, x):
        return self._deep(x).flatten(2).mean(-1)

    def forward_map(self, x):
        h = self._deep(x)
        if self.spatial_map_size:
            s = self.spatial_map_size
            h = nn.functional.adaptive_avg_pool3d(h, (s, s, s))
        return h


class SegResNetEncoder(nn.Module):
    def __init__(self, init_filters=16, blocks_down=(1, 2, 2, 4), spatial_map_size=None):
        super().__init__()
        from monai.networks.nets import SegResNet
        self.net = SegResNet(spatial_dims=3, in_channels=1, out_channels=1,
                             init_filters=init_filters, blocks_down=blocks_down,
                             blocks_up=(1,) * (len(blocks_down) - 1))
        self.feat_dim = init_filters * (2 ** (len(blocks_down) - 1))
        self.spatial_map_size = spatial_map_size

    def forward(self, x):
        h, _ = self.net.encode(x)
        return h.flatten(2).mean(-1)

    def forward_map(self, x):
        h, _ = self.net.encode(x)
        if self.spatial_map_size:
            s = self.spatial_map_size
            h = nn.functional.adaptive_avg_pool3d(h, (s, s, s))
        return h


class MedicalNetResNetEncoder(nn.Module):
    """MONAI refuses pretrained=True unless spatial_dims==3, n_input_channels==1,
    feed_forward is False and (shortcut_type, bias_downsample) match the depth. Passed
    explicitly, because getting them wrong on a different MONAI version yields a randomly
    initialised backbone that looks entirely healthy."""

    def __init__(self, depth=18, pretrained=True, spatial_map_size=None):
        super().__init__()
        from monai.networks.nets import resnet18, resnet34, resnet50
        builders = {18: resnet18, 34: resnet34, 50: resnet50}
        self.net = builders[depth](
            pretrained=pretrained, spatial_dims=3, n_input_channels=1, feed_forward=False,
            shortcut_type=("A" if depth in (18, 34) else "B"),
            bias_downsample=(depth in (18, 34)))
        self.feat_dim = 512 if depth in (18, 34) else 2048
        self.spatial_map_size = spatial_map_size

    def forward(self, x):
        return self.net(x).flatten(1)

    def forward_map(self, x):
        n = self.net
        h = n.act(n.bn1(n.conv1(x)))
        if not n.no_max_pool:
            h = n.maxpool(h)
        h = n.layer4(n.layer3(n.layer2(n.layer1(h))))
        if self.spatial_map_size:
            s = self.spatial_map_size
            h = nn.functional.adaptive_avg_pool3d(h, (s, s, s))
        return h


_SMS = SPATIAL_MAP_SIZE if SAVE_SPATIAL_MAP else None

BACKBONE_REGISTRY = {
    "B1_swinunetr_suprem": {
        "family": "hybrid CNN-Transformer",
        "arch": "SwinUNETR swinViT trunk (feature_size=48)",
        "weights": "SuPreM supervised, 2,100 CT volumes",
        "ckpt_key": "suprem_swinunetr",
        "feat_dim_per_window": 768,
        "builder": lambda: SwinUNETREncoder(48, spatial_map_size=_SMS),
        "citation": "Tang et al. CVPR 2022 (architecture); Li et al. ICLR 2024 (SuPreM weights)",
    },
    "B2_segresnet_suprem": {
        "family": "pure CNN",
        "arch": "SegResNet encoder (init_filters=16, blocks_down=(1,2,2,4))",
        "weights": "SuPreM supervised, 2,100 CT volumes -- SAME corpus as B1",
        "ckpt_key": "suprem_segresnet",
        "feat_dim_per_window": 128,
        "builder": lambda: SegResNetEncoder(16, (1, 2, 2, 4), spatial_map_size=_SMS),
        "citation": "Myronenko 2018 (architecture); Li et al. ICLR 2024 (SuPreM weights)",
    },
    "B3_resnet18_medicalnet": {
        "family": "pure CNN (classic)",
        "arch": "3D ResNet-18",
        "weights": "MedicalNet, 23 medical datasets (MONAI pretrained=True)",
        "ckpt_key": None,
        "feat_dim_per_window": 512,
        "builder": lambda: MedicalNetResNetEncoder(18, True, spatial_map_size=_SMS),
        "citation": "Chen et al. 2019, Med3D / MedicalNet",
    },
    "B4_swinunetr_ssl5050": {
        "family": "hybrid CNN-Transformer",
        "arch": "SwinUNETR swinViT trunk (feature_size=48)",
        "weights": "self-supervised, 5,050 CT volumes",
        "ckpt_key": "ssl_swinunetr_5050",
        "feat_dim_per_window": 768,
        "builder": lambda: SwinUNETREncoder(48, spatial_map_size=_SMS),
        "citation": "Tang et al. CVPR 2022",
    },
}


def build_backbone(name):
    spec = BACKBONE_REGISTRY[name]
    print(f"backbone {name}")
    print(f"  family   : {spec['family']}")
    print(f"  arch     : {spec['arch']}")
    print(f"  weights  : {spec['weights']}")
    print(f"  citation : {spec['citation']}")
    if spec["ckpt_key"]:
        model = spec["builder"]()
        path = download_checkpoint(spec["ckpt_key"], CKPT_DIR)
        rep = verify_load_changed_model(
            model, lambda m: load_state_dict_forgiving(m, path))
        if rep["coverage"] < 0.50:
            print(f"  WARNING: only {rep['coverage']:.1%} of parameters matched. The load "
                  f"did change the model, so this is not fatal -- but check the missing "
                  f"keys above before trusting the features.")
    else:
        # MONAI loads inside the constructor, so there is no pre-load state to snapshot.
        model = verify_constructor_loaded(
            lambda: MedicalNetResNetEncoder(18, True, spatial_map_size=_SMS),
            lambda: MedicalNetResNetEncoder(18, False, spatial_map_size=_SMS))

    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad_(False)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if n_train:
        raise RuntimeError(f"{name} still has {n_train:,} trainable parameters after "
                           f"freezing; the entire design assumes a frozen backbone.")
    if model.feat_dim != spec["feat_dim_per_window"]:
        raise RuntimeError(
            f"{name} produced {model.feat_dim}-d features but the registry (and therefore "
            f"FEATURE_DIM, and therefore every cached vector) says "
            f"{spec['feat_dim_per_window']}. Fix the registry before extracting.")
    print(f"  parameters: {sum(p.numel() for p in model.parameters()):,} (frozen, verified)")
    return model, spec


@torch.no_grad()
def extract_features(model, x, use_amp=None):
    """x: (n_windows, 1, D, H, W). Returns the concatenated pooled vector, and -- when
    SAVE_SPATIAL_MAP -- the stacked spatial maps for the Q3 arm."""
    use_amp = (USE_AMP if use_amp is None else use_amp) and DEVICE == "cuda"
    x = x.to(DEVICE, non_blocking=True)
    vecs, maps = [], []
    bs = max(1, int(EXTRACT_BATCH_SIZE))
    for i in range(0, x.shape[0], bs):
        chunk = x[i:i + bs]
        if use_amp:
            with torch.autocast("cuda", dtype=torch.float16):
                v = model(chunk)
                m = model.forward_map(chunk) if SAVE_SPATIAL_MAP else None
        else:
            v = model(chunk)
            m = model.forward_map(chunk) if SAVE_SPATIAL_MAP else None
        vecs.append(v.float().cpu())
        if m is not None:
            maps.append(m.float().cpu())
    feats = torch.cat(vecs, 0).reshape(-1).numpy().astype(np.float32)
    smap = None
    if SAVE_SPATIAL_MAP and maps:
        # (n_windows, C, s, s, s) -> concatenate along channels: (n_windows*C, s, s, s)
        smap = torch.cat(maps, 0).numpy()
        smap = smap.reshape(-1, *smap.shape[-3:]).astype(SPATIAL_MAP_DTYPE)
    return feats, smap


# Every selected backbone is held in memory at once. Combined that is ~100M frozen
# parameters (~400 MB fp32) for the default three -- comfortable on any GPU that can run
# the extraction at all, and the price of reading the archive only once.
import time as _time

LOADED, SPECS, DROPPED = {}, {}, {}
for _bid in list(BACKBONE_IDS):
    _t0 = _time.time()
    try:
        LOADED[_bid], SPECS[_bid] = build_backbone(_bid)
        print(f"  loaded in {_time.time()-_t0:.1f}s on {DEVICE}\n")
    except Exception as _e:
        if not DROP_BACKBONE_ON_FAILURE:
            raise
        DROPPED[_bid] = f"{type(_e).__name__}: {_e}"
        LOADED.pop(_bid, None)
        SPECS.pop(_bid, None)
        print(f"  *** {_bid} DID NOT LOAD -- dropping it and continuing ***")
        print(f"      {DROPPED[_bid]}\n")
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

BACKBONE_IDS = [b for b in BACKBONE_IDS if b in LOADED]
if not BACKBONE_IDS:
    raise RuntimeError(
        "not one backbone could be built, so there is nothing to extract. The reasons are "
        f"printed above: {DROPPED}")
BACKBONE_ID = BACKBONE_IDS[0]
_spec = SPECS[BACKBONE_ID]

# ---- one backbone on the GPU, or all of them? decided by measurement below --------------------
ONE_AT_A_TIME = (GPU_ONE_AT_A_TIME is True or str(GPU_ONE_AT_A_TIME).lower() == "true")
_GPU_RESIDENT = None


def _activate(bid):
    """LOADED[bid], guaranteed to be on the compute device. A no-op unless ONE_AT_A_TIME."""
    global _GPU_RESIDENT
    m = LOADED[bid]
    if not ONE_AT_A_TIME or DEVICE != "cuda":
        return m
    if _GPU_RESIDENT != bid:
        if _GPU_RESIDENT is not None and _GPU_RESIDENT in LOADED:
            LOADED[_GPU_RESIDENT].to("cpu")
        m.to(DEVICE)
        _GPU_RESIDENT = bid
        torch.cuda.empty_cache()
    return m


def _oom_escalate(why=""):
    """Fall back to one backbone on the GPU. Idempotent; safe to call from the hot loop."""
    global ONE_AT_A_TIME, _GPU_RESIDENT
    if ONE_AT_A_TIME or DEVICE != "cuda":
        return False
    ONE_AT_A_TIME = True
    for _o in list(LOADED):
        LOADED[_o].to("cpu")
    _GPU_RESIDENT = None
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  -> switched to ONE backbone on the GPU at a time{(' (' + why + ')') if why else ''}. "
          f"Costs ~0.1 s per series in host-to-device copies; nothing else changes.")
    return True


if ONE_AT_A_TIME:
    _oom_escalate.__wrapped__ = None          # marker: the fallback was requested, not measured
    for _o in list(LOADED):
        LOADED[_o].to("cpu")
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("GPU_ONE_AT_A_TIME is on: backbones are held on the CPU and moved to the GPU in turn.")
print(f"{len(LOADED)} backbone(s) resident: "
      f"{sum(sum(p.numel() for p in m.parameters()) for m in LOADED.values()):,} "
      f"frozen parameters total")


# ---- prove the whole model path works before spending hours on it -----------------------------
# Synthetic head phantoms, not random noise: a soft-tissue ellipsoid inside a dense shell
# inside air. Random noise passes every shape assertion while telling you nothing about
# whether the windowing produced anything with structure in it.
def _phantom(D=160, H=512, W=512, seed=1):
    # np.ogrid, NOT np.mgrid. mgrid materialises three int64 arrays of D*H*W elements --
    # 1.0 GB for the default 160x512x512 -- and this runs once per backbone, so the
    # self-test alone allocated ~3 GB it never needed. ogrid returns broadcastable views
    # and the arithmetic below broadcasts identically, for a few hundred bytes.
    rng = np.random.default_rng(seed)
    zz, yy, xx = np.ogrid[0:D, 0:H, 0:W]
    r = ((zz - D / 2) / 62) ** 2 + ((yy - H / 2) / 190) ** 2 + ((xx - W / 2) / 175) ** 2
    vol = np.full((D, H, W), -1000.0, np.float32)
    vol[r < 1.00] = 35.0
    vol[(r >= 1.00) & (r < 1.14)] = 900.0
    vol += rng.standard_normal(vol.shape).astype(np.float32) * 8.0
    return vol, (zz, yy, xx, D, H, W)


def _backbone_self_test(model, spec):
    base, (zz, yy, xx, D, H, W) = _phantom()
    lesion = base.copy()
    lesion[((zz - D / 2) / 34) ** 2 + ((yy - H / 2 - 40) / 60) ** 2
           + ((xx - W / 2 - 30) / 60) ** 2 < 1.0] = 68.0
    noise = (np.random.default_rng(3).standard_normal((D, H, W)).astype(np.float32)
             * 300.0) - 200.0
    sp = {"sx": 0.45, "sy": 0.45, "sz": 1.0}
    # Time ONE real extraction (all windows, at the configured grid) -- this is the number
    # the extraction cell quotes as its ETA. Timing the whole self-test instead would fold
    # in phantom generation and resampling and overstate the forward-pass cost.
    _xb = preprocess_for_backbone(base, sp)
    import time as _t
    # WARM UP FIRST. The first forward pass through a model on CUDA pays for lazy context
    # creation, cuDNN algorithm selection and kernel autotuning -- routinely several
    # seconds that are never paid again. Timing that pass and calling it "seconds per
    # volume" overstates the steady-state cost by an order of magnitude, and this number
    # is what the CPU gate and every ETA in the extraction cell are computed from.
    # CUDA is also asynchronous: without a synchronize, the clock stops while kernels are
    # still queued. (extract_features ends in .cpu(), which does synchronise, but relying
    # on that is exactly the kind of accident that stops being true after an edit.)
    f_base, m_base = extract_features(model, _xb)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    _t0 = _t.time()
    f_base, m_base = extract_features(model, _xb)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    _sec_per_vol = _t.time() - _t0
    f_les, _ = extract_features(model, preprocess_for_backbone(lesion, sp))
    f_noise, _ = extract_features(model, preprocess_for_backbone(noise, sp))

    exp = spec["feat_dim_per_window"] * len(HU_WINDOWS)
    assert f_base.shape == (exp,), f"feature dim {f_base.shape} != expected ({exp},)"
    assert np.isfinite(f_base).all(), "non-finite features"
    assert float(np.std(f_base)) > 1e-6, "the backbone returned a constant vector"

    def rel(a, b):
        return float(np.linalg.norm(a - b) / max(np.linalg.norm(a), 1e-12))

    r_les, r_noise = rel(f_base, f_les), rel(f_base, f_noise)
    assert r_les > 1e-9, (
        "adding a hyperdense collection changed the features by less than 1e-9 relative -- "
        "two different volumes produced the same features. Suspect preprocess_for_backbone "
        "handing the model the same array twice.")
    assert r_noise > 2 * r_les, (
        f"pure noise moved the features only {r_noise/max(r_les,1e-12):.1f}x as much as an "
        f"added lesion ({r_noise:.4f} vs {r_les:.4f}) -- not enough separation to believe "
        f"the backbone is responding to image content.")
    return exp, f_base, r_les, r_noise, m_base, _sec_per_vol


# Time a REAL forward pass at the configured grid, so the extraction cell can quote a
# measured ETA instead of a hardcoded constant. The inherited "~2 s per volume" was
# measured for ResNet-18 on a smaller grid; at (64,256,256) with three windows the true
# CPU cost is ~26 s for ResNet-18 and ~106 s for SwinUNETR -- a 50x understatement for the
# heaviest backbone, which is exactly the number someone would rely on when deciding
# whether to set ALLOW_CPU_EXTRACTION.
MEASURED_SEC_PER_VOLUME, PEAK_GPU_GB = {}, {}


def _self_test_once(bid):
    """Run the self-test with peak GPU memory recorded. Raises whatever it raises."""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    out = _backbone_self_test(_activate(bid), SPECS[bid])
    if DEVICE == "cuda":
        PEAK_GPU_GB[bid] = torch.cuda.max_memory_allocated() / 2 ** 30
    return out


for _bid in list(BACKBONE_IDS):
    _res = None
    try:
        _res = _self_test_once(_bid)
    except torch.cuda.OutOfMemoryError as _e:
        # The real deployment condition just ran out of memory. Fall back to one backbone on
        # the GPU and try again -- that is the whole point of measuring before extracting.
        gc.collect()
        torch.cuda.empty_cache()
        if _oom_escalate(f"{_bid} ran out of memory with {len(LOADED)} backbones resident"):
            try:
                _res = _self_test_once(_bid)
            except Exception as _e2:
                _e = _e2
        if _res is None:
            DROPPED[_bid] = f"{type(_e).__name__}: {_e}"
    except Exception as _e:
        DROPPED[_bid] = f"{type(_e).__name__}: {_e}"

    if _res is None:
        if not DROP_BACKBONE_ON_FAILURE:
            raise RuntimeError(f"{_bid} failed its self-test: {DROPPED.get(_bid)}")
        print(f"\n  *** {_bid} FAILED ITS SELF-TEST -- dropping it and continuing ***")
        print(f"      {DROPPED[_bid]}")
        LOADED.pop(_bid, None)
        SPECS.pop(_bid, None)
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        continue

    _exp, _f, _rl, _rn, _m, _sec = _res
    MEASURED_SEC_PER_VOLUME[_bid] = _sec
    if FEATURE_DIMS[_bid] != _exp:
        DROPPED[_bid] = (f"FEATURE_DIMS says {FEATURE_DIMS[_bid]} but the backbone produced "
                         f"{_exp}; the registry and the cached vectors would disagree")
        print(f"\n  *** {_bid}: {DROPPED[_bid]} -- dropping it ***")
        LOADED.pop(_bid, None)
        SPECS.pop(_bid, None)
        MEASURED_SEC_PER_VOLUME.pop(_bid, None)
        continue
    print(f"\n[{_bid}] self-test on synthetic head phantoms")
    print(f"  feature vector          : {_exp}-d ({len(HU_WINDOWS)} windows x "
          f"{SPECS[_bid]['feat_dim_per_window']}), all finite")
    print(f"  + hyperdense collection : relative change {_rl:.5f}")
    print(f"  vs pure noise           : relative change {_rn:.5f}")
    print(f"  -> responds {_rn/max(_rl,1e-12):.0f}x more to a different volume than to one "
          f"added lesion, which is the right ordering.")
    if _m is not None:
        print(f"  spatial map             : {_m.shape} -> {_m.nbytes/1024:.0f} KB per series")
    print(f"  MEASURED forward cost   : {_sec:.2f} s per volume on {DEVICE} "
          f"(warmed up; the first pass is not what you pay 4,000 times)")
    if _bid in PEAK_GPU_GB:
        _tot_gb = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
        print(f"  PEAK GPU memory         : {PEAK_GPU_GB[_bid]:.2f} GB of {_tot_gb:.1f} GB"
              f"   ({PEAK_GPU_GB[_bid]/_tot_gb:.0%} of the card)")
BACKBONE_IDS = [b for b in BACKBONE_IDS if b in LOADED]
if not BACKBONE_IDS:
    raise RuntimeError(f"every backbone was dropped, so there is nothing to extract: {DROPPED}")
BACKBONE_ID = BACKBONE_IDS[0]
_spec = SPECS[BACKBONE_ID]
if DROPPED:
    print("\n" + "=" * 94)
    print(f"DROPPED {len(DROPPED)} BACKBONE(S) -- extraction continues without them")
    for _b, _why in DROPPED.items():
        print(f"  {_b}\n    {str(_why)[:300]}")
    print("  Re-run this notebook later with only these listed in BACKBONE_IDS; everything")
    print("  already extracted tonight is skipped, so it costs only what these need.")
    print("=" * 94)

# Written to Drive so nb3's RQ4 cell can quote a MEASURED cost per volume for the 3D
# arm without rebuilding these backbones (and re-downloading their checkpoints) just to
# time them. RQ4's claim is accuracy PER UNIT COST, so the cost half has to be real.
_cost = {
    "device": str(DEVICE), "n_windows": len(HU_WINDOWS),
    "grid": list(TARGET_SHAPE_DHW), "spacing": list(TARGET_SPACING_XYZ),
    "backbones": {b: {"sec_per_volume": float(MEASURED_SEC_PER_VOLUME[b]),
                      "parameters": int(sum(p.numel() for p in LOADED[b].parameters())),
                      "feature_dim": int(FEATURE_DIMS[b]),
                      "arch": SPECS[b]["arch"], "family": SPECS[b]["family"]}
                  for b in BACKBONE_IDS},
}
with contextlib.suppress(Exception):
    import json as _json
    # MERGE, NEVER REPLACE, AND KEY BY DEVICE. This file is read by nb3 to price the 3D arm
    # in RQ4's accuracy-per-cost claim. It used to be overwritten wholesale by whichever
    # notebook ran last -- so a CPU-only session replaced the T4 measurements and RQ4 quoted
    # 144.89 s per volume against the 3.24 s that was actually measured on the GPU.
    _cost_path = os.path.join(OUTPUT_DIR, "backbone_cost.json")
    _all = {}
    if os.path.exists(_cost_path):
        with contextlib.suppress(Exception):
            _all = _json.load(open(_cost_path))
    if "by_device" not in _all:                      # migrate the old flat layout
        _all = {"by_device": ({_all.get("device", "unknown"): _all} if _all else {})}
    _all["by_device"][str(DEVICE)] = _cost
    _all.update({k: v for k, v in _cost.items() if k != "backbones"})
    _all["backbones"] = _cost["backbones"]           # top level = this run, for compatibility
    with open(_cost_path, "w") as _fh:
        _json.dump(_all, _fh, indent=2)
    print(f"\nbackbone cost written -> {os.path.join(OUTPUT_DIR, 'backbone_cost.json')}")

_TOTAL_SEC_PER_VOL = sum(MEASURED_SEC_PER_VOLUME.values())
print(f"\nALL BACKBONES: {_TOTAL_SEC_PER_VOL:.2f} s of GPU per volume combined "
      f"({len(BACKBONE_IDS)} backbones x {len(HU_WINDOWS)} windows)")
print(f"  -> {_TOTAL_SEC_PER_VOL*MAX_SERIES/3600:.1f} h of forward passes for "
      f"MAX_SERIES={MAX_SERIES:,}, before any Drive reads.")
print(f"  Reading the archive is the dominant cost on this dataset, and it is now paid")
print(f"  ONCE for all {len(BACKBONE_IDS)} backbones rather than once each.")
if DEVICE == "cpu":
    print(f"\n  This is a CPU runtime. Phase A wants a GPU: Runtime > Change runtime type")
    print(f"  > GPU, then Runtime > Restart session. Nothing cached is lost.")
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


# ==================================================================================== #
# GO / NO-GO -- the last thing before a run measured in days
# ==================================================================================== #
# One place, one verdict. Everything here is verified elsewhere too; what this adds is a
# CONFIG FINGERPRINT that the parallel sessions can be compared on. Two sessions that differ
# in MAX_SERIES or SAMPLE_SEED build different manifests, and their shards then overlap and
# leave gaps -- which no session can detect from the inside.
def config_fingerprint():
    import hashlib
    import json as _j
    payload = {
        "preproc": {b: preproc_signature(b) for b in BACKBONE_IDS}
        if "preproc_signature" in globals() else
        {"note": "cell 14 not run yet; grid + windows below are what matter"},
        "shape": list(TARGET_SHAPE_DHW), "spacing": list(TARGET_SPACING_XYZ),
        "windows": [[n, c, w] for n, c, w in HU_WINDOWS],
        "interp": RESAMPLE_INTERPOLATOR, "skullstrip": bool(SKULL_STRIP),
        "smap": [bool(SAVE_SPATIAL_MAP), int(SPATIAL_MAP_SIZE)],
        "pixfix": bool(APPLY_RSNA_PIXEL_FIX),
        "floor": (float(HU_AIR_FLOOR) if CLAMP_HU_FLOOR else None),
        "qc": [bool(QC_REJECT_ON_FAIL), list(QC_HU_P01_RANGE), float(QC_HU_P99_MAX),
               int(SERIES_MIN_SLICES), float(MIN_KEPT_SLICE_FRACTION)],
        "cohort": [int(MAX_SERIES) if MAX_SERIES else None, int(SAMPLE_SEED)],
        "backbones": list(BACKBONE_IDS),
    }
    blob = _j.dumps(payload, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:16], payload


_FP, _payload = config_fingerprint()
print()
print("=" * 94)
print("GO / NO-GO")
print("=" * 94)
print(f"  CONFIG FINGERPRINT   {_FP}")
print(f"    Shards of THIS notebook must all print this. It includes the backbone list, so a")
print(f"    notebook loading a different set of backbones prints a different string -- expected.")
try:
    _cfp = manifest_fingerprint(MANIFEST)
    print(f"  COHORT FINGERPRINT   {_cfp['sha256_16'][:12]}   ({_cfp['n']:,} series)")
    print(f"    THIS is the one that must match across every notebook and every shard: it is")
    print(f"    what says the sessions are partitioning the same cohort.")
except Exception as _e:
    print(f"  COHORT FINGERPRINT   unavailable ({_e})")
print()

_stop, _warn2 = [], []

# 1. the grid decision, which cannot be revisited later without discarding everything
if "GRID_FIT" not in globals():
    _stop.append("cell 10 has not run in this session, so the grid-fit report does not exist. "
                 "Run it: it is the check that the fixed crop is not throwing findings away, "
                 "and both grid settings are in the preprocessing signature.")
else:
    _gf = GRID_FIT
    _ok_grid = (_gf["frac_positive_slices_cropped"] <= MAX_POSITIVE_SLICE_CROP
                and _gf["n_series_emptied"] == 0)
    print(f"  grid {tuple(TARGET_SHAPE_DHW)} at {TARGET_SPACING_XYZ} mm "
          f"-> {_gf['fov_d_mm']:.0f} mm deep")
    print(f"    positive slices cropped away : "
          f"{_gf['frac_positive_slices_cropped']:.2%}"
          + ("  OK" if _ok_grid else
             ("  ACKNOWLEDGED" if ACKNOWLEDGE_GRID_CROP else "  OVER THE LIMIT")))
    if not _ok_grid and not ACKNOWLEDGE_GRID_CROP:
        _stop.append("the grid crops away more findings than MAX_POSITIVE_SLICE_CROP allows")

# 2. every selected backbone actually loaded, and proved it
_missing_bb = [b for b in BACKBONE_IDS if b not in globals().get("LOADED", {})]
if _missing_bb:
    _stop.append(f"these backbones were never built in this session: {_missing_bb}")
else:
    print(f"  backbones            : {len(BACKBONE_IDS)} loaded and behaviourally verified")
    for b in BACKBONE_IDS:
        print(f"    {b:<26} {FEATURE_DIMS[b]:>5}-d   "
              f"{MEASURED_SEC_PER_VOLUME.get(b, float('nan')):.2f} s/volume on {DEVICE}")

# 3. the cohort, and whether it came from the shared cache
_n_mine = len(MANIFEST) if SHARD_COUNT <= 1 else \
    len([s for i, s in enumerate(MANIFEST["series"]) if i % SHARD_COUNT == SHARD_INDEX])
print(f"  cohort               : {len(MANIFEST):,} series"
      + (f", this shard {SHARD_INDEX + 1}/{SHARD_COUNT} = {_n_mine:,}"
         if SHARD_COUNT > 1 else "")
      + (" (reused from the cache)" if globals().get("_manifest_from_cache")
         else " (drawn fresh and cached)"))

# 4. the device
if DEVICE != "cuda":
    (_stop if not ALLOW_CPU_EXTRACTION else _warn2).append(
        "no GPU: Runtime > Change runtime type > GPU, then Restart session")
else:
    print(f"  device               : {torch.cuda.get_device_name(0)}")

# 5. what this will cost, from the two things that were measured rather than guessed
_model_s = float(sum(MEASURED_SEC_PER_VOLUME.values())) if MEASURED_SEC_PER_VOLUME else 0.0
_read_s = globals().get("MEASURED_READ_SEC_PER_SERIES")
if _read_s is None and globals().get("READ_CAL"):
    _read_s = min(r[0] for r in READ_CAL)
if _read_s:
    _per = _read_s + _model_s
    print(f"  measured cost        : read {_read_s:.1f} s + model {_model_s:.1f} s "
          f"= {_per:.1f} s per series")
    print(f"  PROJECTED            : {_n_mine * _per / 3600:.1f} h for this session's "
          f"{_n_mine:,} series")
    if MAX_SECONDS_PER_RUN is None and _n_mine * _per > 11 * 3600:
        _warn2.append(f"this is longer than a Colab session usually lasts. Nothing is lost "
                      f"when the kernel is recycled -- the loop is resumable -- but setting "
                      f"MAX_SECONDS_PER_RUN gives a clean stop instead of a kill.")
else:
    print(f"  measured cost        : model {_model_s:.1f} s per series; the read cost is "
          f"measured when extraction starts")

print()
if _warn2:
    print(f"  {len(_warn2)} note(s):")
    for w in _warn2:
        print(f"    - {w}")
if _stop:
    print("=" * 94)
    raise RuntimeError(
        "NO-GO. " + " | ".join(_stop) +
        "\n  This stops here rather than at hour thirty. Fix the above and re-run this cell.")
print("  GO. Before starting the other sessions, confirm each prints the same fingerprint")
print(f"      {_FP}   and a DIFFERENT SHARD_INDEX.")
print("      Then run the next cell. It is resumable: interrupt it whenever you like.")
print("=" * 94)


In [ ]:
# 14. Phase A -- ONE pass over the archive, ALL backbones  [RESUMABLE, budgeted]
#
# WHY THIS IS ONE LOOP AND NOT THREE RUNS
# ---------------------------------------
# Phase A is I/O bound, not compute bound. A series costs roughly 10-30 s to pull out of
# the split zip and reassemble, against ~0.2-2 s of GPU for even the heaviest backbone.
# Running the notebook once per backbone pays the EXPENSIVE part three times over to gain
# nothing: the reassembled volume is bit-identical every time, and the only thing that
# differs downstream is which frozen network it is fed to.
#
# So the volume is assembled, QC'd and preprocessed ONCE, then pushed through every
# backbone that still needs it. Wall clock goes from ~3x(I/O + model) to ~(I/O + 3x model).
#
# RESUMABILITY IS STILL PER BACKBONE. Each keeps its own progress log, so:
#   * a backbone you already finished in an earlier single-backbone run is skipped here;
#   * adding a fourth backbone later re-reads only the series that one is missing;
#   * a failure in one backbone does not mark the series done for the others.
import contextlib
import gc
import json
import os
import time
import traceback

import numpy as np
import torch
from tqdm.auto import tqdm

PREPROC_CODE_VERSION = 1

# Deliberately NOT ".npy"-terminated -- see _save_npy_with_retry.
PARTIAL_SUFFIX = ".partial"


def sweep_partials(dirs):
    """Delete temp files left by a kernel that died mid-write. They are always incomplete
    (the rename is what publishes a finished file), so there is nothing to recover."""
    n = 0
    for d in dirs:
        if not os.path.isdir(d):
            continue
        for f in os.listdir(d):
            if f.endswith(PARTIAL_SUFFIX):
                with contextlib.suppress(Exception):
                    os.remove(os.path.join(d, f))
                    n += 1
    return n


def preproc_signature(backbone_id):
    return "|".join([
        f"v{PREPROC_CODE_VERSION}",
        f"backbone={backbone_id}",
        f"shape={tuple(TARGET_SHAPE_DHW)}",
        f"spacing={tuple(TARGET_SPACING_XYZ)}",
        f"windows={[(n, c, w) for n, c, w in HU_WINDOWS]}",
        f"interp={RESAMPLE_INTERPOLATOR}",
        f"skullstrip={bool(SKULL_STRIP)}",
        f"smap={bool(SAVE_SPATIAL_MAP)}x{SPATIAL_MAP_SIZE if SAVE_SPATIAL_MAP else 0}",
        f"pixfix={bool(APPLY_RSNA_PIXEL_FIX)}",
        f"floor={HU_AIR_FLOOR if CLAMP_HU_FLOOR else None}",
        f"qc={bool(QC_REJECT_ON_FAIL)}{tuple(QC_HU_P01_RANGE)}{QC_HU_P99_MAX}",
        f"minslices={int(SERIES_MIN_SLICES)}",
        f"minkept={float(MIN_KEPT_SLICE_FRACTION)}",
    ])


PREPROC_SIGS = {b: preproc_signature(b) for b in BACKBONE_IDS}
PREPROC_SIG = PREPROC_SIGS[BACKBONE_ID]
for _b, _s in PREPROC_SIGS.items():
    print(f"preprocessing signature [{_b}]:\n  {_s}")


def load_progress(path, sig):
    """Only 'ok' under the CURRENT signature counts as done, so fixing a bug and re-running
    redoes exactly the series that bug broke and nothing else."""
    done, last = set(), {}
    if not os.path.exists(path):
        return done, last
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue        # a torn last line from a killed kernel: skip, not fatal
            k = rec.get("series")
            last[k] = rec
            if rec.get("status") == "ok" and rec.get("preproc_sig") == sig:
                done.add(k)
            else:
                done.discard(k)
    return done, last


def _save_npy_with_retry(path, arr, attempts=4, base_delay=2.0):
    """Drive's FUSE mount throws intermittently under sustained small-file writes, and this
    loop writes several per series for hours. Without a retry a rate-limit blip is recorded
    as a failed series; if it lasts minutes it is recorded as hundreds of them, and the
    night's work looks like a bug in the model rather than a hiccup in the filesystem.
    Written to a temp file and renamed, so a crash mid-write cannot leave a truncated .npy
    that later looks valid."""
    last = None
    tmp = path + PARTIAL_SUFFIX
    for k in range(attempts):
        try:
            # np.save appends ".npy" unless the name already ends in it, which is why the
            # temp file must be handed an OPEN FILE OBJECT: naming it "<series>.npy.tmp"
            # and letting np.save fix it up gives "<series>.npy.tmp.npy", a leftover that
            # every `endswith(".npy")` scan downstream then counts as a real feature file.
            with open(tmp, "wb") as _fh:
                np.save(_fh, arr)
            os.replace(tmp, path)
            return
        except Exception as e:
            last = e
            with contextlib.suppress(Exception):
                if os.path.exists(tmp):
                    os.remove(tmp)
            if k < attempts - 1:
                time.sleep(base_delay * (2 ** k))
    raise RuntimeError(f"could not write {path} after {attempts} attempts: "
                       f"{type(last).__name__}: {last}")


DONE, LAST, OUT_DIRS, SMAP_DIRS, PROG = {}, {}, {}, {}, {}
for b in BACKBONE_IDS:
    DONE[b], LAST[b] = load_progress(progress_log_path(b), PREPROC_SIGS[b])
    OUT_DIRS[b] = os.path.join(FEATURE_DIR, b)
    os.makedirs(OUT_DIRS[b], exist_ok=True)
    if SAVE_SPATIAL_MAP:
        SMAP_DIRS[b] = os.path.join(SPATIAL_DIR, b)
        os.makedirs(SMAP_DIRS[b], exist_ok=True)

_swept = sweep_partials(list(OUT_DIRS.values()) + list(SMAP_DIRS.values()))
if _swept:
    print(f"cleaned up {_swept} partial file(s) from an interrupted run")

if not (isinstance(SHARD_COUNT, int) and SHARD_COUNT >= 1
        and isinstance(SHARD_INDEX, int) and 0 <= SHARD_INDEX < SHARD_COUNT):
    raise SystemExit(f"SHARD_INDEX must be an integer in [0, SHARD_COUNT); got "
                     f"SHARD_INDEX={SHARD_INDEX!r}, SHARD_COUNT={SHARD_COUNT!r}")

# The shard is taken by POSITION in the manifest, and the manifest is a deterministic
# function of MAX_SERIES and SAMPLE_SEED -- so parallel sessions partition the cohort with
# no coordination and no overlap, provided those two config values match.
_all_series = list(MANIFEST["series"])
_mine = ([s for i, s in enumerate(_all_series) if i % SHARD_COUNT == SHARD_INDEX]
         if SHARD_COUNT > 1 else _all_series)
if SHARD_COUNT > 1:
    print(f"SHARD {SHARD_INDEX+1} of {SHARD_COUNT}: {len(_mine):,} of "
          f"{len(_all_series):,} series belong to this session.")
    print(f"  The other {SHARD_COUNT-1} session(s) must use the SAME MAX_SERIES "
          f"({MAX_SERIES}) and SAMPLE_SEED ({SAMPLE_SEED}),")
    print(f"  or the shards will overlap and each will redo the other's work.")

# A series is pending if ANY selected backbone still needs it.
_pending = [s for s in _mine if any(s not in DONE[b] for b in BACKBONE_IDS)]
print()
for b in BACKBONE_IDS:
    print(f"  {b:<26} {len(DONE[b] & set(_mine)):>6,} done, "
          f"{sum(1 for s in _mine if s not in DONE[b]):>6,} to do (this shard)")
print(f"  series needing at least one backbone: {len(_pending):,} of {len(_mine):,}")

_SEC_PER_VOL = float(sum(MEASURED_SEC_PER_VOLUME.values()))
if DEVICE == "cpu" and len(_pending) > CPU_EXTRACTION_WARN_ABOVE and not ALLOW_CPU_EXTRACTION:
    raise RuntimeError(
        f"{len(_pending):,} series still to extract and no GPU.\n"
        f"  MEASURED on this runtime: {_SEC_PER_VOL:.1f} s per volume across "
        f"{len(BACKBONE_IDS)} backbone(s), so about "
        f"{len(_pending)*_SEC_PER_VOL/3600:.1f} h of forward passes alone, on top of the "
        f"Drive reads.\n"
        f"  * To use a GPU: Runtime > Change runtime type > GPU, then Runtime > Restart "
        f"session and re-run. Nothing already done is lost -- this cell is resumable.\n"
        f"  * To proceed on CPU anyway: set ALLOW_CPU_EXTRACTION = True in the config.\n"
        f"  * To extract a small batch now and the rest later: set MAX_SERIES_PER_RUN.")
if DEVICE == "cpu" and _pending:
    print(f"NOTE: backbones on CPU at a MEASURED {_SEC_PER_VOL:.1f} s per volume combined, "
          f"so ~{len(_pending)*_SEC_PER_VOL/3600:.1f} h of forward passes.")

_limit = TRIAL_RUN_MAX_SERIES if TRIAL_RUN_MAX_SERIES is not None else MAX_SERIES_PER_RUN
if TRIAL_RUN_MAX_SERIES is not None and MAX_SERIES_PER_RUN is not None:
    _limit = min(TRIAL_RUN_MAX_SERIES, MAX_SERIES_PER_RUN)
if _limit is not None:
    print(f"this run will stop after {_limit} series")

for b in BACKBONE_IDS:
    PROG[b] = open(progress_log_path(b), "a")


_UNFLUSHED = {b: 0 for b in BACKBONE_IDS}


def log_progress(b, rec):
    """Append one record. Flushing is batched: see PROGRESS_FLUSH_EVERY in the config for
    why that is safe here (a lost record costs a re-extraction, never a wrong result)."""
    _t = time.time()
    PROG[b].write(json.dumps(rec) + "\n")
    _UNFLUSHED[b] += 1
    if _UNFLUSHED[b] >= max(1, int(PROGRESS_FLUSH_EVERY)):
        PROG[b].flush()
        if PROGRESS_FSYNC:
            with contextlib.suppress(Exception):
                os.fsync(PROG[b].fileno())
        _UNFLUSHED[b] = 0
    try:
        timing["write"] += time.time() - _t
    except NameError:
        pass                            # first call happens before `timing` exists
    if rec.get("status") == "ok":
        DONE[b].add(rec["series"])


def flush_progress():
    """Every exit path from the loop goes through this, including KeyboardInterrupt and an
    unexpected exception -- otherwise a batched buffer would throw away up to
    PROGRESS_FLUSH_EVERY finished series precisely when the run is being cut short."""
    for b in BACKBONE_IDS:
        with contextlib.suppress(Exception):
            PROG[b].flush()
            if PROGRESS_FSYNC:
                os.fsync(PROG[b].fileno())
            _UNFLUSHED[b] = 0


_meta = MANIFEST.set_index("series")[["patient", "study", "n_slices"]].to_dict("index")
run_t0 = time.time()
n_series_ok = n_series_fail = 0
n_pairs_ok = n_pairs_fail = 0
failures = []
stop_reason = None
# "write" is measured because it is NOT free: every series writes one feature .npy and one
# spatial-map .npy per backbone to a FUSE-mounted Drive, plus an fsync'd progress record.
# An earlier version timed only read/preprocess/model and then printed those three as
# percentages "of wall clock" -- on a real run they summed to 64% of it, so the split was
# quietly renormalising over itself and the missing third looked like it did not exist.
timing = {"read": 0.0, "preprocess": 0.0, "model": 0.0, "write": 0.0}
amp_active = bool(USE_AMP and DEVICE == "cuda")

try:
    for _i_series, suid in enumerate(tqdm(_pending, desc="series")):
        # The limit counts SERIES SUCCESSFULLY EXTRACTED, not iterations attempted -- which
        # is what the original did and what someone setting TRIAL_RUN_MAX_SERIES=20 for a
        # smoke test actually wants. Counting attempts would hand back 5 usable volumes out
        # of 20 tries on a bad patch of the archive and call it done.
        if _limit is not None and n_series_ok >= _limit:
            stop_reason = f"series limit ({_limit}) reached"
            break
        if MAX_SECONDS_PER_RUN is not None and (time.time() - run_t0) > MAX_SECONDS_PER_RUN:
            stop_reason = f"reached MAX_SECONDS_PER_RUN = {MAX_SECONDS_PER_RUN}s"
            break

        needed = [b for b in BACKBONE_IDS if suid not in DONE[b]]
        if not needed:
            continue
        members = SERIES_MEMBERS.get(suid, [])
        m = _meta.get(suid, {})
        base = {"series": suid, "patient": m.get("patient", ""), "study": m.get("study", ""),
                "amp": amp_active, "ts": int(time.time())}
        if not members:
            for b in needed:
                log_progress(b, {**base, "preproc_sig": PREPROC_SIGS[b], "status": "failed",
                                 "detail": "no members in the index"})
            failures.append((suid, "no members in the index"))
            n_series_fail += 1
            n_pairs_fail += len(needed)
            continue

        # ---- read + QC + preprocess: ONCE, shared by every backbone -------------------
        try:
            t = time.time()
            vol, spacing, vinfo = assemble_volume(members, MPF, MEMBER_LOOKUP)
            timing["read"] += time.time() - t

            reason = qc_volume(vinfo)          # QC before any GPU time is spent
            if reason and QC_REJECT_ON_FAIL:
                raise ValueError(reason)

            t = time.time()
            x = preprocess_for_backbone(vol, spacing)
            # Moved to the device ONCE and reused by every backbone. extract_features would
            # otherwise copy the same ~50 MB tensor across PCIe once per backbone.
            x_dev = x.to(DEVICE, non_blocking=True) if DEVICE == "cuda" else x
            timing["preprocess"] += time.time() - t
            del vol
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            for b in needed:
                log_progress(b, {**base, "preproc_sig": PREPROC_SIGS[b], "status": "failed",
                                 "detail": msg})
            failures.append((suid, msg))
            n_series_fail += 1
            n_pairs_fail += len(needed)
            continue

        # ---- one shared volume -> every backbone that still needs it -----------------
        _series_all_ok = True
        try:
            for b in needed:
                try:
                    t = time.time()
                    try:
                        feat, smap = extract_features(_activate(b), x_dev)
                    except torch.cuda.OutOfMemoryError:
                        # An OOM leaves the allocator fragmented; without clearing it the
                        # NEXT series usually OOMs too and one unlucky volume empties the
                        # rest of the run.
                        print(f"    CUDA OOM on {suid[-12:]} [{b}] -- clearing and retrying")
                        torch.cuda.empty_cache()
                        gc.collect()
                        time.sleep(2.0)
                        _oom_escalate('an OOM during extraction')
                        feat, smap = extract_features(_activate(b), x_dev)
                    timing["model"] += time.time() - t
                    if not np.isfinite(feat).all():
                        raise ValueError("the backbone returned non-finite values")
                    if feat.shape[0] != FEATURE_DIMS[b]:
                        raise ValueError(f"got {feat.shape[0]}-d features, expected "
                                         f"{FEATURE_DIMS[b]} for {b}")
                    t = time.time()
                    _save_npy_with_retry(os.path.join(OUT_DIRS[b], f"{suid}.npy"),
                                         feat.astype(np.float32))
                    # The Q3 arm's input, written BEFORE the progress record so a series
                    # logged "ok" always has both files. A crash between the two writes
                    # would otherwise leave a series the feature table admits and the Q3
                    # arm cannot load.
                    if SAVE_SPATIAL_MAP and smap is not None:
                        _save_npy_with_retry(os.path.join(SMAP_DIRS[b], f"{suid}.npy"), smap)
                    timing["write"] += time.time() - t
                    log_progress(b, {
                        **base, "preproc_sig": PREPROC_SIGS[b], "status": "ok", "detail": "",
                        "dim": int(feat.shape[0]),
                        "n_slices": int(vinfo.get("n_slices", 0)),
                        "spacing": [spacing["sx"], spacing["sy"], spacing["sz"]],
                        "z_spacing_source": vinfo.get("z_spacing_source", ""),
                        "n_pixel_corrected": int(vinfo.get("n_pixel_corrected", 0)),
                        "n_duplicate_sop": int(vinfo.get("n_duplicate_sop", 0)),
                        "n_crc_mismatch": int(vinfo.get("n_crc_mismatch", 0)),
                        "n_shape_mismatch": int(vinfo.get("n_shape_mismatch", 0)),
                        "kept_slice_fraction": vinfo.get("kept_slice_fraction"),
                        "z_gap_ratio": vinfo.get("z_gap_ratio"),
                        "hu_p01": vinfo.get("hu_p01"), "hu_p50": vinfo.get("hu_p50"),
                        "hu_p99": vinfo.get("hu_p99"),
                        "qc": reason or "pass", "note": vinfo.get("note", "")})
                    n_pairs_ok += 1
                except Exception as e:
                    msg = f"{type(e).__name__}: {e}"
                    log_progress(b, {**base, "preproc_sig": PREPROC_SIGS[b],
                                     "status": "failed", "detail": msg})
                    failures.append((f"{suid} [{b}]", msg))
                    n_pairs_fail += 1
                    _series_all_ok = False
        finally:
            del x, x_dev
        n_series_ok += int(_series_all_ok)
        n_series_fail += int(not _series_all_ok)
        # NOT empty_cache() every series. It hands the caching allocator's blocks back to
        # the driver, so the very next volume has to go through cudaMalloc again -- 4,000
        # times, for no benefit, since the loop's peak allocation is identical each pass.
        # It belongs where it helps: after an OOM (below), and occasionally alongside the
        # Python-side collection.
        if (_i_series + 1) % CONSOLIDATE_EVERY_N_SERIES == 0:
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
except KeyboardInterrupt:
    stop_reason = "interrupted from the keyboard"
except Exception:
    stop_reason = "an unexpected error ended the loop"
    traceback.print_exc(limit=4)
finally:
    flush_progress()
    for b in BACKBONE_IDS:
        with contextlib.suppress(Exception):
            PROG[b].close()

print()
print(f"this run: {n_series_ok:,} series fully extracted, {n_series_fail:,} with a failure")
print(f"          {n_pairs_ok:,} (series, backbone) pairs written, {n_pairs_fail:,} failed")
print(f"          {(time.time()-run_t0)/60:.1f} min")
# Percentages are taken against MEASURED WALL CLOCK, with the unattributed remainder shown
# explicitly. A split that renormalises over its own buckets always sums to 100% and can
# therefore never tell you that something big is missing from it.
_wall = max(time.time() - run_t0, 1e-9)
_tsum = sum(timing.values())
print("time split: " + " | ".join(f"{k} {v/60:.1f} min ({100*v/_wall:.0f}%)"
                                  for k, v in timing.items())
      + f" | unattributed {(_wall-_tsum)/60:.1f} min ({100*(_wall-_tsum)/_wall:.0f}%)")
if n_series_ok + n_series_fail:
    _per = _wall / (n_series_ok + n_series_fail)
    print(f"            {_per:.1f} s per series end to end -> "
          f"{_per*len(_mine)/3600:.0f} h projected for this shard's {len(_mine):,} series")
print()
print(f"  READ is the number that decides whether the single-pass design paid off. At "
      f"{100*timing['read']/_wall:.0f}% of wall clock,")
print(f"  running {len(BACKBONE_IDS)} separate single-backbone passes would have cost "
      f"roughly {len(BACKBONE_IDS)*timing['read']/60:.0f} min of reading")
print(f"  instead of {timing['read']/60:.0f} min.")
if timing["write"] > 0.15 * _wall:
    print()
    print(f"  WRITE is {100*timing['write']/_wall:.0f}% of wall clock -- small files on a "
          f"FUSE-mounted Drive.")
    print(f"  Raise PROGRESS_FLUSH_EVERY (currently {PROGRESS_FLUSH_EVERY}) to batch the "
          f"progress log harder;")
    print(f"  the only cost is re-extracting up to that many series if the kernel dies.")
if stop_reason:
    print(f"\nstopped early: {stop_reason}. Re-run this cell to continue -- nothing is lost.")
# The full list goes to a CSV, as in the original: a grouped summary tells you WHAT went
# wrong, but chasing down WHICH series needs the actual list. This cell is resumed many
# times over a run, so the file MERGES with what is already there -- opening it "w" each
# time meant the CSV only ever described the last session, and a session that happened to
# fail nothing left the previous session's failures sitting there looking current.
import csv as _csv

# One file per backbone (and per shard), because a single shared path means concurrent
# sessions truncate it underneath each other. Failures that happened before any backbone
# was reached -- a bad read, a QC rejection -- are recorded against every backbone that
# was waiting on that volume, which is what actually happened.
_by_backbone = {b: [] for b in BACKBONE_IDS}
for _s, _e in failures:
    if _s.endswith("]") and " [" in _s:
        _sid, _b = _s.rsplit(" [", 1)
        _b = _b[:-1]
        if _b in _by_backbone:
            _by_backbone[_b].append((_sid, _e))
            continue
    for _b in BACKBONE_IDS:
        _by_backbone[_b].append((_s, _e))

_written = 0
for _b, _rows in _by_backbone.items():
    _path = failures_csv_path(_b)
    _prev = []
    if os.path.exists(_path):
        with contextlib.suppress(Exception):
            with open(_path, newline="") as f:
                _prev = [tuple(r[:2]) for r in _csv.reader(f)][1:]
    _merged = list(dict.fromkeys([(str(a), str(c)) for a, c in _prev] +
                                 [(str(a), str(c)) for a, c in _rows]))
    if not _merged:
        continue
    with open(_path, "w", newline="") as f:
        w = _csv.writer(f)
        w.writerow(["series", "error"])
        w.writerows(_merged)
    _written += 1
    print(f"\n{_b}: {len(_merged):,} failure record(s) "
          f"({len(_rows):,} from this run) -> {_path}")
if failures:
    groups = {}
    for _s, _e in failures:
        groups.setdefault(_e[:160], []).append(_s)
    print(f"\n{len(failures)} failure(s), {len(groups)} distinct message(s) "
          f"-- full lists in {os.path.basename(failures_csv_path('<backbone>'))}:")
    for _e, _ss in sorted(groups.items(), key=lambda kv: -len(kv[1]))[:8]:
        print(f"  {len(_ss):>5}x  {_e[:110]}")
        print(f"         e.g. {_ss[0]}")

In [ ]:
# 15. The feature tables, their quality control, and the extraction fingerprint
#
# PROVENANCE FIRST. A .npy sitting in a feature directory does not by itself prove it came
# from the current version of this notebook -- that directory lives on Drive and survives
# every code change. So only series logged "ok" under the CURRENT preprocessing signature
# are admitted, and anything else on disk is reported separately rather than swept in.
import json
import os

import numpy as np
import pandas as pd


def load_ok_records(path, require_sig):
    """Last record wins per series -- a retry supersedes a stale failure."""
    latest = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                latest[rec.get("series")] = rec
    ok = {k: v for k, v in latest.items() if v.get("status") == "ok"}
    current = {k: v for k, v in ok.items() if v.get("preproc_sig") == require_sig}
    return current, len(ok) - len(current)


def _npy_ids(d):
    """Series ids of FINISHED .npy files in d. `.partial` files from an interrupted write
    are excluded explicitly rather than by hoping their name does not end in .npy."""
    if not os.path.isdir(d):
        return set()
    return {f[:-4] for f in os.listdir(d)
            if f.endswith(".npy") and not f.endswith(PARTIAL_SUFFIX)}


FEATURE_TABLES, TABLE_ERRORS = {}, {}
for b in BACKBONE_IDS:
    print("=" * 78)
    print(b)
    print("=" * 78)
    ok_records, n_stale = load_ok_records(progress_log_path(b), PREPROC_SIGS[b])
    if n_stale:
        print(f"EXCLUDING {n_stale:,} series extracted under an older preprocessing "
              f"signature. They are still on disk and still logged ok; re-running the "
              f"extraction cell will redo them. Until it does, this table is a SUBSET.")
    d = os.path.join(FEATURE_DIR, b)
    on_disk = _npy_ids(d)
    wanted = set(ok_records)
    usable = sorted(wanted & on_disk)
    missing = sorted(wanted - on_disk)
    untracked = sorted(on_disk - wanted)

    vecs, uids, bad = [], [], []
    dim = None
    for s in usable:
        try:
            v = np.load(os.path.join(d, f"{s}.npy")).astype(np.float32).reshape(-1)
        except Exception:
            missing.append(s)
            continue
        if dim is None:
            dim = int(v.shape[0])
        if v.shape[0] != dim or not np.isfinite(v).all():
            bad.append(s)
            continue
        vecs.append(v)
        uids.append(s)

    print(f"  logged ok (current sig) : {len(wanted):,}")
    print(f"  .npy present on disk    : {len(on_disk):,}")
    print(f"  usable                  : {len(uids):,}" + (f" (dim {dim})" if dim else ""))
    if missing:
        print(f"  logged ok, file missing : {len(missing):,} (e.g. {missing[:2]})")
    if bad:
        print(f"  wrong dim / non-finite  : {len(bad):,} (excluded)")
    if untracked:
        print(f"  on disk, no ok record   : {len(untracked):,} (excluded)")
    if not uids:
        print("  NO USABLE FEATURES YET -- run the extraction cell first (or longer).\n")
        continue
    if dim != FEATURE_DIMS[b]:
        # RECORDED, NOT RAISED. Raising here aborted the whole loop, so a problem with the
        # first backbone meant the other two never got their tables written either -- and
        # the run that produced them was hours long.
        TABLE_ERRORS[b] = (f"features are {dim}-d but the config expects "
                           f"{FEATURE_DIMS[b]} ({len(HU_WINDOWS)} windows x "
                           f"{_BACKBONE_DIMS[b]}). Something changed mid-run.")
        print(f"  SKIPPED: {TABLE_ERRORS[b]}\n")
        continue

    X = np.vstack(vecs)
    cols = [f"f_{i:04d}" for i in range(X.shape[1])]
    T = pd.DataFrame(X, columns=cols)
    T.insert(0, "series", uids)
    meta_cols = ["series", "patient", "study", "n_slices", "z_extent_mm", "sx",
                 "sz_measured", "n_wrapped"] + RSNA_SUBTYPES
    if "any" in MANIFEST.columns:
        meta_cols.append("any")
    T = T.merge(MANIFEST[[c for c in dict.fromkeys(meta_cols) if c in MANIFEST.columns]],
                on="series", how="left")
    p = features_parquet_path(b)
    drive_retry(lambda: T.to_parquet(p + ".tmp", index=False),
                what=f"writing the {b} feature table", quiet=False)
    drive_retry(lambda: os.replace(p + ".tmp", p), what="committing the feature table")
    FEATURE_TABLES[b] = T
    print(f"  -> {p}")
    print(f"     {T.shape[0]:,} rows x {len(cols)} features")

    # ---- feature health -----------------------------------------------------------------
    zero_frac = float((np.abs(X) < 1e-8).mean())
    dead = int((X.std(axis=0) < 1e-8).sum()) if X.shape[0] > 1 else 0
    print(f"     mean {X.mean():.4f}, std {X.std():.4f}, {zero_frac:.1%} exactly zero, "
          f"{dead} constant dimension(s)")
    if dead > 0.5 * X.shape[1]:
        print(f"     WARNING: over half the dimensions are constant across every volume. "
              f"That is a backbone producing almost no signal -- check the windowing.")
    if SAVE_SPATIAL_MAP:
        sd = os.path.join(SPATIAL_DIR, b)
        n_sm = len(_npy_ids(sd))
        print(f"     spatial maps on disk    : {n_sm:,} "
              f"({'Q3 can run' if n_sm >= len(uids) else 'INCOMPLETE -- Q3 will be skipped'})")
    print()

print("=" * 78)
if TABLE_ERRORS:
    print("BACKBONES WITH NO TABLE:")
    for b, e in TABLE_ERRORS.items():
        print(f"  {b}: {e}")
    print()
if FEATURE_TABLES:
    _common = None
    for b, T in FEATURE_TABLES.items():
        s = set(T["series"])
        _common = s if _common is None else (_common & s)
    print(f"series present in ALL {len(FEATURE_TABLES)} table(s): {len(_common):,}")
    print("That intersection is what nb2's grid runs on -- comparing backbones on different")
    print("cohorts would confound 'which backbone' with 'which series'.")
    for b, T in FEATURE_TABLES.items():
        if len(T) != len(_common):
            print(f"  {b}: {len(T)-len(_common):,} series not shared by every backbone")
else:
    print("no feature tables written yet.")
FEATURES = FEATURE_TABLES.get(BACKBONE_ID, pd.DataFrame())

---

## Phase A is done

Every backbone in `BACKBONE_IDS` now has a feature table and a spatial-map directory.

Open **`nb2_phaseB_grid.ipynb`** next. It runs the full backbone x quantum-family grid on
these cached features and needs **no GPU** -- a CPU runtime is the right choice there.
